UPscaledEV Project Main Code (rolling MPC version)
Under the funding from TotalEnergies
Author: Yizhan Gu
Email: yig031@ucsd.edu
Affiliation: UCSD CER


Readme:

This is the canonical Rolling 24-h MPC notebook used for the SEGAN study.

1. The optimization equations remain in this notebook; the site-study runner below reuses these exact cells rather than maintaining a second model.
2. Edit the **Canonical paper-run configuration** at the beginning of the Parameters cell to select the site, resource case, forecast updater, and run scope.
3. `EV_BESS` and `FULL_BTM` are optimized cases. `BUILDING_ONLY` and `BUILDING_PV` are passive-meter counterfactuals and are billed directly without an artificial optimization problem.
4. Site inputs are produced by `upscaledev_site_data_postprocessing.ipynb` and stored under `2025Data/Site_Data_2025`.
5. Formal site results are written under `Results_Rolling/Site_Cases_2025`; temporary executed notebook copies are never written to the repository.
6. The legacy aggregate EV+BESS interactive defaults remain available when `RUN_CANONICAL_STUDY=False`.


Set working path and import packages

In [ ]:
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / '2025Data').exists():
    raise FileNotFoundError(
        'Start Jupyter from the UPSCALeDEV_2024 repository root so relative data paths are reproducible.'
    )
os.chdir(PROJECT_ROOT)
print("Path is:", os.getcwd(), "\n")
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
from datetime import timedelta
from datetime import datetime
import time
from time import process_time
import calendar
import holidays
from pathlib import Path
import cvxpy as cp
import sys
from forecast_ED_PD import KnownUser, UnKnownUser
import glob
import warnings
from tqdm import tqdm
import gurobipy as gp
import seaborn as sns
import plotly.express as px
warnings.filterwarnings("ignore", message="The problem includes expressions that don't support CPP backend")
warnings.filterwarnings("ignore", message="Solution may be inaccurate.*")


Parameters

In [2]:
# MPC parameters
import warnings
warnings.filterwarnings(
    "ignore",
    message="Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.",
    category=UserWarning,
)
# ===== Canonical paper-run configuration ================================
# The safe default is review-only. Set RUN_CANONICAL_STUDY=True (or the
# matching environment variable to 1) only when a formal run is intended.
RUN_CANONICAL_STUDY = os.environ.get('RUN_CANONICAL_STUDY', '0') == '1'
CANONICAL_SITE_CASE = os.environ.get('CANONICAL_SITE_CASE', 'Both')
CANONICAL_RESOURCE_CASE = os.environ.get('CANONICAL_RESOURCE_CASE', 'FULL_BTM')
CANONICAL_FORECAST_CASE = os.environ.get('CANONICAL_FORECAST_CASE', 'Both')
CANONICAL_RUN_SCOPE = os.environ.get('CANONICAL_RUN_SCOPE', 'smoke').lower()  # smoke | month | june
CANONICAL_RUN_MONTH = int(os.environ.get('CANONICAL_RUN_MONTH', '6'))
# Zero preserves legacy standalone runs. Three starts March-onward executed history.
CANONICAL_CHAIN_START_MONTH = int(os.environ.get('CANONICAL_CHAIN_START_MONTH', '0'))
REBUILD_CANONICAL_BASELINES = os.environ.get('REBUILD_CANONICAL_BASELINES', '0') == '1'
CANONICAL_INPUT_ROOT = Path('2025Data') / 'Site_Data_2025'
CANONICAL_RESULTS_ROOT = Path('Results_Rolling') / 'Site_Cases_2025'

CANONICAL_SITE_CONFIG = {
    'NTPLL': {
        'ev_file': CANONICAL_INPUT_ROOT / 'NTPLL_EV_2025_QC.csv',
        'btm_file': CANONICAL_INPUT_ROOT / 'NTPLL_BTM_2025_June_July_15min_QC.csv',
    },
    'Center_Hall': {
        'ev_file': CANONICAL_INPUT_ROOT / 'Center_Hall_EV_2025_QC.csv',
        'btm_file': CANONICAL_INPUT_ROOT / 'Center_Hall_BTM_2025_June_July_15min_QC.csv',
    },
}
if CANONICAL_CHAIN_START_MONTH:
    for _site, _config in CANONICAL_SITE_CONFIG.items():
        _config['btm_file'] = CANONICAL_INPUT_ROOT / f'{_site}_BTM_2025_15min_QC.csv'
CANONICAL_OPTIMIZED_RESOURCE_CASES = {'EV_BESS', 'FULL_BTM'}
CANONICAL_DIRECT_RESOURCE_CASES = {'BUILDING_ONLY', 'BUILDING_PV'}

if CANONICAL_SITE_CASE not in {*CANONICAL_SITE_CONFIG, 'Both'}:
    raise ValueError(f'Unknown CANONICAL_SITE_CASE={CANONICAL_SITE_CASE!r}')
if CANONICAL_RESOURCE_CASE not in {
    *CANONICAL_OPTIMIZED_RESOURCE_CASES, *CANONICAL_DIRECT_RESOURCE_CASES, 'Both'
}:
    raise ValueError(f'Unknown CANONICAL_RESOURCE_CASE={CANONICAL_RESOURCE_CASE!r}')
if CANONICAL_FORECAST_CASE not in {'Persistence', 'Perfect', 'Both'}:
    raise ValueError(f'Unknown CANONICAL_FORECAST_CASE={CANONICAL_FORECAST_CASE!r}')
if CANONICAL_RUN_SCOPE not in {'smoke', 'month', 'june'}:
    raise ValueError("CANONICAL_RUN_SCOPE must be 'smoke', 'month', or legacy alias 'june'")
if not 1 <= CANONICAL_RUN_MONTH <= 12:
    raise ValueError('CANONICAL_RUN_MONTH must be an integer from 1 to 12')

# Legacy interactive/sensitivity defaults. These reproduce the established
# aggregate EV+BESS case unless the canonical runner explicitly overrides them.
SENSITIVITY_ENABLED = False
SENSITIVITY_CASE_NAME = 'reference'
BASE_VERSION_DIR_EV_BESS = Path('Results_Rolling')
BASE_VERSION_DIR_PV_BUILDING = Path('Results_Rolling_PV_Building')
SENSITIVITY_OUTPUT_ROOT = Path('Sensitivity_Results') / 'Rolling_24h'
EV_DATA_FILE = Path('2025Data/EV_data/UCSD_AllSites_Merge_PostProcessedSession_QC.csv')
BASELINE_DISPATCH_FILE = None
PASSIVE_DER_ENABLED = False
BASE_VERSION_DIR = BASE_VERSION_DIR_PV_BUILDING if PASSIVE_DER_ENABLED else BASE_VERSION_DIR_EV_BESS
BTM_BUILDING_FILE = Path('2025Data/Bldg_data/Bldg_load_2025_15min_QC.csv')
BTM_PV_FILE = Path('2025Data/PV_data/PV_generation_2025_15min_QC.csv')
BTM_LOAD_SCALE = 1.0
BTM_PV_SCALE = 1.0
AS_ACTIVATION_MODE = 'constant'  # 'constant' or 'caiso_regulation'
AS_ACTIVATION_DEFAULTS = {'RU': 0.7, 'RD': 0.7, 'SP': 0.2, 'NSP': 0.2}
AS_ACTIVATION_SCALE_DEFAULTS = {'RU': 1.0, 'RD': 1.0, 'SP': 1.0, 'NSP': 1.0}
AS_ACTIVATION_CONSTANTS = dict(AS_ACTIVATION_DEFAULTS)
AS_ACTIVATION_SCALE = dict(AS_ACTIVATION_SCALE_DEFAULTS)
AS_REGULATION_ATTENUATION_FILE = Path('2025Data/AS_Call_Rates/caiso_regulation_attenuation_hourly_2025.csv')
AS_CONTINGENCY_ACTIVATION_FILE = None
_AS_ACTIVATION_CACHE = {}

def _safe_sensitivity_slug(value):
    slug = ''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in str(value)).strip('_')
    if not slug:
        raise ValueError('SENSITIVITY_CASE_NAME must contain at least one letter or digit')
    return slug

def _read_csv_cached(path):
    path = Path(path)
    key = (str(path.resolve()), path.stat().st_mtime_ns)
    if key not in _AS_ACTIVATION_CACHE:
        _AS_ACTIVATION_CACHE.clear()
        _AS_ACTIVATION_CACHE[key] = pd.read_csv(path)
    return _AS_ACTIVATION_CACHE[key].copy()

def _get_as_activation_profile(time_index):
    """Return horizon-aligned RU/RD/SP/NSP activation fractions.

    CAISO publishes hourly SOC attenuation factors for regulation up/down.
    Spin/non-spin deployment is a different data product; when an explicit
    event file is not supplied, their documented reference assumptions remain.
    """
    idx = pd.DatetimeIndex(time_index)
    if idx.empty:
        raise ValueError('AS activation profile requested for an empty horizon')
    mode = str(AS_ACTIVATION_MODE).lower()
    values = {k: np.full(len(idx), float(AS_ACTIVATION_CONSTANTS[k])) for k in ('RU', 'RD', 'SP', 'NSP')}
    if mode == 'caiso_regulation':
        lookup = _read_csv_cached(AS_REGULATION_ATTENUATION_FILE)
        required = {'quarter', 'trade_hour', 'alpha_RU', 'alpha_RD'}
        if not required.issubset(lookup.columns):
            raise ValueError(f'Missing CAISO attenuation columns: {sorted(required - set(lookup.columns))}')
        table = {(str(r.quarter), int(r.trade_hour)): (float(r.alpha_RU), float(r.alpha_RD)) for r in lookup.itertuples()}
        pairs = [(f'{ts.year}Q{ts.quarter}', int(ts.hour) + 1) for ts in idx]
        missing = [pair for pair in pairs if pair not in table]
        if missing:
            raise ValueError(f'CAISO attenuation coverage missing for {missing[:5]}')
        values['RU'] = np.array([table[pair][0] for pair in pairs], dtype=float)
        values['RD'] = np.array([table[pair][1] for pair in pairs], dtype=float)
    elif mode != 'constant':
        raise ValueError("AS_ACTIVATION_MODE must be 'constant' or 'caiso_regulation'")
    if AS_CONTINGENCY_ACTIVATION_FILE is not None:
        contingency = _read_csv_cached(AS_CONTINGENCY_ACTIVATION_FILE)
        required = {'Interval start', 'alpha_SP', 'alpha_NSP'}
        if not required.issubset(contingency.columns):
            raise ValueError(f'Missing contingency activation columns: {sorted(required - set(contingency.columns))}')
        contingency['Interval start'] = pd.to_datetime(contingency['Interval start'], errors='raise')
        contingency = contingency.set_index('Interval start').sort_index().reindex(idx)
        if contingency[['alpha_SP', 'alpha_NSP']].isna().any().any():
            raise ValueError('Contingency activation file does not cover the complete model horizon')
        values['SP'] = contingency['alpha_SP'].to_numpy(float)
        values['NSP'] = contingency['alpha_NSP'].to_numpy(float)
    for product in values:
        values[product] = values[product] * float(AS_ACTIVATION_SCALE.get(product, 1.0))
        if (not np.isfinite(values[product]).all()) or (values[product] < 0).any() or (values[product] > 1).any():
            raise ValueError(f'Activation fraction for {product} must remain within [0, 1]')
    return values['RU'], values['RD'], values['SP'], values['NSP']

SENSITIVITY_RESOURCE_TAG = 'ev_bess_pv_building_retail' if PASSIVE_DER_ENABLED else 'ev_bess'
VERSION_DIR = str(BASE_VERSION_DIR if not SENSITIVITY_ENABLED else SENSITIVITY_OUTPUT_ROOT / _safe_sensitivity_slug(SENSITIVITY_CASE_NAME) / SENSITIVITY_RESOURCE_TAG)
dt_m_EV = 15
dt_h = dt_m_EV/60
interval = timedelta(minutes=dt_m_EV)
c_BESS_penalty = 1e-3  # penalty coefficient for BESS ramping
M = 1e5  # large number for big-M method
# BESS parameters
SOC_BESS_min = 0.05
SOC_BESS_max = 0.95
P_BESS_max = 250                     # kW
C_BESS = 332                         # kWh
gamma = 0.9                          # efficiency
# Plot styles
palette = plt.get_cmap('tab10').colors
label_fontsize = 12
title_fontsize = 14
tick_fontsize = 10
legend_fontsize = 8
linewidth = 1.5
# SDG&E DG-R Primary tariff representation used by the study.
# Demand-rate provenance: SDG&E 2024 Medium & Large Commercial rate table.
# TOU period definitions: https://www.sdge.com/regulatory-filing/16021/commercial-time-use-periods-all-non-residential-except-pat1
# The numerical energy rates below preserve the study's frozen tariff inputs;
# the calendar classifier now applies summer/winter, weekday, weekend, and US-holiday rules automatically.
SDGE_DGR_PRIMARY_2024 = {
    'Summer': {
        'c_PD': 3.05,
        'on_peak': 0.335 + 0.490,
        'off_peak': 0.03921 + 0.175,
        'super_off_peak': 0.03921 + 0.0815,
    },
    'Winter': {
        'c_PD': 0.63,
        'on_peak': 0.384 + 0.169,
        'off_peak': 0.03978 + 0.095,
        'super_off_peak': 0.03978 + 0.073,
    },
}
c_NCD = 15.38
_SDGE_HOLIDAY_CACHE = {}

def _sdge_season(timestamp):
    ts = pd.Timestamp(timestamp)
    return 'Summer' if 6 <= ts.month <= 10 else 'Winter'

def _sdge_holiday_dates(year_value):
    year_value = int(year_value)
    if year_value not in _SDGE_HOLIDAY_CACHE:
        _SDGE_HOLIDAY_CACHE[year_value] = set(holidays.US(years=year_value).keys())
    return _SDGE_HOLIDAY_CACHE[year_value]

def _sdge_tou_period(timestamp):
    """Return SDG&E commercial TOU period for one local interval start."""
    ts = pd.Timestamp(timestamp)
    hour = ts.hour + ts.minute / 60.0
    if 16.0 <= hour < 21.0:
        return 'on_peak'
    weekend_or_holiday = ts.dayofweek >= 5 or ts.date() in _sdge_holiday_dates(ts.year)
    if weekend_or_holiday:
        return 'off_peak' if (14.0 <= hour < 16.0 or 21.0 <= hour < 24.0) else 'super_off_peak'
    if hour < 6.0 or 10.0 <= hour < 14.0:
        return 'super_off_peak'
    return 'off_peak'

def get_sdge_dgr_tariff_profile(time_index):
    """Return TOU $/kWh, period labels, and season labels aligned to timestamps."""
    idx = pd.DatetimeIndex(pd.to_datetime(time_index))
    periods = np.array([_sdge_tou_period(ts) for ts in idx], dtype=object)
    seasons = np.array([_sdge_season(ts) for ts in idx], dtype=object)
    rates = np.array([
        SDGE_DGR_PRIMARY_2024[season][period]
        for season, period in zip(seasons, periods)
    ], dtype=float)
    return rates, periods, seasons

def get_sdge_on_peak_mask(time_index):
    return np.array([
        _sdge_tou_period(ts) == 'on_peak'
        for ts in pd.to_datetime(time_index)
    ], dtype=float)

def get_sdge_pd_rate(timestamp):
    return float(SDGE_DGR_PRIMARY_2024[_sdge_season(timestamp)]['c_PD'])

# Backward-compatible reference values are only defaults before a dated run starts.
_TARIFF_REFERENCE_DAY = pd.Timestamp('2025-06-02 00:00:00')
_tariff_reference_index = pd.date_range(_TARIFF_REFERENCE_DAY, periods=96, freq='15min')
c_e_TOU_AL, _, _ = get_sdge_dgr_tariff_profile(_tariff_reference_index)
season = _sdge_season(_TARIFF_REFERENCE_DAY)
c_PD = get_sdge_pd_rate(_TARIFF_REFERENCE_DAY)
# Fail fast on the distinctions that the former static vector missed.
assert _sdge_tou_period('2025-06-02 10:00') == 'super_off_peak'  # weekday
assert _sdge_tou_period('2025-06-01 10:00') == 'super_off_peak'  # weekend
assert _sdge_season('2025-01-15') == 'Winter'

# Tax
c_tax_DWR = 0.00580                               # x Total kWh
c_tax_oEESbo_Franchise = 0.0688 * c_tax_DWR       # x Total kWh
c_tax_CA_Surcharge = 0.00030                      # x Total kWh
c_tax_CA_Regulatory = 0.00058                     # x Total kWh
# x Total Bill (UDC+Commodity)
c_tax_SD_Franchise = 0.0578
c_tax_all = c_tax_DWR + c_tax_oEESbo_Franchise + c_tax_CA_Surcharge + c_tax_CA_Regulatory
# Forecast methods
Numb_EVs = 0
Numb_AbsDiffEVs = 0
Fc_SessionkWh = 'PersistenceSessionkWh'  # 'PerfectSessionkWh' #'PersistenceSessionkWh'
Fc_NumbEV = 'PersistenceNumbEV'          # 'PerfectNumbEV'     #'PersistenceNumbEV'
Fc_AtArrival = 'PerfectatArrival'    # 'PerfectatArrival'  #'MLatArrival' from Avik
Fc_building = 'Perfect'              # 'Perfectat'  #'Persistence'
Fc_PV = 'Perfect'                    # 'Perfectat'  #'Persistence'
# NOTE: to get the Dispatch file before baseline calculation, run the baseline.py file before running this file
DAM = 1 # 1/0: w/o day-ahead market participation (Demand Response)
# Case configuration. Set RUN_CASE1=True to also run the reduced-service Case1.
# Default False forces 100% energy service for all EV drivers and speeds up tests.
RUN_CASE1 = False
# Opt-in minimum delivered fraction; 1.0 preserves the formal full-service model.
# Full request remains the upper bound: reduced service is allowed, not forced.
SERVICE_LEVEL_MIN = float(globals().get('SERVICE_LEVEL_MIN', 1.0))
if not 0.0 <= SERVICE_LEVEL_MIN <= 1.0:
    raise ValueError('SERVICE_LEVEL_MIN must lie in [0, 1].')
if SERVICE_LEVEL_MIN < 1.0 and RUN_CASE1:
    raise ValueError('Minimum-service sensitivity uses Base only, not legacy event-based Case1.')
Cases = ['Base', 'Case1'] if RUN_CASE1 else ['Base']
DA_CASE_IDX = len(Cases)  # index for the DA/V0G reference case in len(Cases)+1 arrays
RT_DA_COMMITMENT_MODE = 'settlement'  # CAISO two-settlement: fixed DA parameter plus RT deviation settlement
MPC_VERSION_LABEL = 'Rolling MPC'
SHOW_6PANEL_SUMMARY_BOX = False  # The old RT summary box used local solver-output values, not daily totals.
PLOT_DAILY_6PANEL_DATES = 'all'  # []: skip daily 6-panel; None/'all': all days; or use [1, 2] / ['20250701']
ANALYSIS_DAYS_CONFIG = None  # None uses RUN_DAYS_CONFIG for final financial plots
SAVE_FINAL_FINANCIAL_FIGURES = False  # main notebooks save CSVs; paper figures are made in paper_financial_comparison_plots.ipynb
Direction_Aligned_AddOn = True # True/False: whether to add constraints when direction aligned in WM
#sym:WM_Mode switch for market participation
WM_Mode = 'full'  # 'full' or 'wm_only' or 'retail_only'
Enable_WM = (WM_Mode in ['full'])
year = 2025
_formal_days_env = os.environ.get('RUN_FORMAL_DAYS', '').strip()
RUN_DAYS_CONFIG = ([int(d.strip()) for d in _formal_days_env.split(',') if d.strip()]
                   if _formal_days_env else list(range(1, 31)))
GUROBI_MIPGAP = 1e-2
SOLVER_THREADS = 6
RT_SOLVER_TIME_LIMIT = None  # no normal RT cutoff: every accepted step must reach the target MIPGap
RT_SOLVER_EMERGENCY_TIME_LIMIT = 300.0  # watchdog only; hitting it aborts the scenario instead of accepting an incumbent
REQUIRE_RT_OPTIMAL = True
RT_SOLVER_STATUS_COUNTS = {}
RT_SOLVER_LIMIT_EVENTS = []
def _record_rt_solver_status(problem, date_label, interval_idx, case_label):
    """Record whether Gurobi actually reached its RT TimeLimit."""
    status = str(problem.status)
    RT_SOLVER_STATUS_COUNTS[status] = RT_SOLVER_STATUS_COUNTS.get(status, 0) + 1
    extra = getattr(getattr(problem, 'solver_stats', None), 'extra_stats', None)
    gurobi_status = getattr(extra, 'Status', None)
    hit_limit = (status == 'user_limit') or (gurobi_status == 9)
    if hit_limit:
        def _safe_attr(name):
            try:
                return float(getattr(extra, name))
            except Exception:
                return float('nan')
        RT_SOLVER_LIMIT_EVENTS.append({
            'date': pd.Timestamp(date_label).strftime('%Y-%m-%d'),
            'interval': int(interval_idx),
            'case': str(case_label),
            'forecast': str(Fc_SessionkWh),
            'mode': str(WM_Mode),
            'cvxpy_status': status,
            'gurobi_status': gurobi_status,
            'runtime_s': _safe_attr('Runtime'),
            'mip_gap': _safe_attr('MIPGap'),
        })
# VERSION_DIR is selected by the isolated sensitivity controls above.
TARGET_SAVE = 'RT' # 'RT'/'DA'
def _date_key_allowed(date_like, selected_dates):
    """Return whether a daily expensive plot should be saved for date_like."""
    if selected_dates is None or selected_dates == 'all':
        return True
    if selected_dates == [] or selected_dates == () or selected_dates == set():
        return False
    target = pd.Timestamp(date_like).strftime('%Y%m%d')
    allowed = set()
    for item in selected_dates:
        s = str(item)
        if s.isdigit() and len(s) <= 2:
            allowed.add(f"{int(globals().get('year', 2025)):04d}{int(globals().get('month', 7)):02d}{int(s):02d}")
        else:
            allowed.add(pd.Timestamp(item).strftime('%Y%m%d'))
    return target in allowed
RUN_MONTHS_CONFIG = [6, 7]


Data processing

In [3]:
if Fc_AtArrival == 'MLatArrival':
    User_Data = pd.read_csv("Driver_Table.csv")
    User_known = User_Data['driver_id'][User_Data['TotSession']>10].unique()
    UserNoBess = []
# create weekdays (excluding holidays) and weekends (including holidays) lists
Holidays = holidays.US(years=year)
Holidays_dates = list(Holidays.keys())
Holidays_dates = [date for date in Holidays_dates if date.year == year]
start_ind_Y = datetime(year,1,1)
end_ind_Y = datetime(year+1,1,1)
DateSeries_ThisY = []
while start_ind_Y < end_ind_Y:
    DateSeries_ThisY.append(start_ind_Y)
    start_ind_Y += timedelta(hours=24)
DateSeries_ThisY = pd.Series(DateSeries_ThisY)
Y_Weekends = DateSeries_ThisY[DateSeries_ThisY.dt.dayofweek>=5]
Y_Holidays = DateSeries_ThisY[DateSeries_ThisY.dt.date.isin(Holidays_dates)]
Y_WeekendsWH = pd.concat([Y_Weekends,Y_Holidays],axis=0).drop_duplicates(keep='first', inplace=False).sort_values(axis=0, ascending=True).reset_index(drop=True)
Y_WeekdaysWOH = DateSeries_ThisY[~(DateSeries_ThisY.isin(Y_WeekendsWH))].sort_values(axis=0, ascending=True).reset_index(drop=True)
if len(Y_WeekdaysWOH) + len(Y_WeekendsWH) != len(DateSeries_ThisY):
    print("Error in holiday identification!\n")
    sys.exit()
# VERSION: 2025 EV data
Data = pd.read_csv(EV_DATA_FILE, low_memory=False)
Data['Interval start'] = pd.to_datetime(Data['Interval start'])
Data['Interval end'] = pd.to_datetime(Data['Interval end'])
Data['Session start'] = pd.to_datetime(Data['Session start'])
Data['Session end'] = pd.to_datetime(Data['Session end'])
Data['Interval max demand kW'] = pd.to_numeric(Data['Interval max demand kW'], errors='coerce')
Data['Interval average demand kW'] = pd.to_numeric(Data['Interval average demand kW'], errors='coerce')
print('EV sensitivity input:', EV_DATA_FILE)
print("Number of Intervals:", len(Data))
print("Test year:", year)
print("Date range:", Data['Interval start'].min(), "to", Data['Interval start'].max())
print("Unique sites:", len(Data['Site'].unique()))
print("Unique chargers:", Data['Station Name'].nunique(dropna=True))
Data['Car_'] = Data['10-digit UID'].astype(str)
# Read forecast-matched pre-period dispatch history. Site studies must supply
# their own history so the EV baseline is never inherited from the campus aggregate.
if BASELINE_DISPATCH_FILE is None:
    dir_Input = os.path.join('Results_Shrinking/Dispatch')
    filename_Input = dir_Input + '/2025_'+Fc_SessionkWh+'_'+Fc_NumbEV+'_'+Fc_AtArrival +'_baseline.csv'
else:
    filename_Input = str(Path(BASELINE_DISPATCH_FILE))
if not Path(filename_Input).exists():
    raise FileNotFoundError(f'Baseline dispatch history not found: {filename_Input}')
Dispatch_2025_baseline = pd.read_csv(filename_Input, low_memory=False, header=0)
Dispatch_2025_baseline['Interval start'] = pd.to_datetime(Dispatch_2025_baseline['Interval start'])

# Optional passive building/PV meter. The processed site file stores
# p_load_kW = reconstructed gross building demand,
# p_PV_kW = measured PV generation, and
# p_native_net_kW = the original building net-meter reading.
# Hence p_native_net_kW = p_load_kW - p_PV_kW. Building and PV enter only
# the PCC/retail balance; neither creates a wholesale energy or AS position.
BTM_Building_Data = None
BTM_PV_Data = None
if PASSIVE_DER_ENABLED:
    required_building = {'Interval start', 'p_load_kW'}
    required_pv = {'Interval start', 'p_PV_kW'}
    BTM_Building_Data = pd.read_csv(BTM_BUILDING_FILE)
    BTM_PV_Data = pd.read_csv(BTM_PV_FILE)
    if not required_building.issubset(BTM_Building_Data.columns):
        raise ValueError(
            'Run upscaledev_site_data_postprocessing.ipynb; '
            f'missing building columns {sorted(required_building - set(BTM_Building_Data.columns))}'
        )
    if not required_pv.issubset(BTM_PV_Data.columns):
        raise ValueError(
            'Run upscaledev_site_data_postprocessing.ipynb; '
            f'missing PV columns {sorted(required_pv - set(BTM_PV_Data.columns))}'
        )
    for frame, label in ((BTM_Building_Data, 'building'), (BTM_PV_Data, 'PV')):
        frame['Interval start'] = pd.to_datetime(frame['Interval start'], errors='raise')
        if frame['Interval start'].duplicated().any():
            raise ValueError(f'Duplicate processed {label} timestamps')
        frame.set_index('Interval start', inplace=True)
        frame.sort_index(inplace=True)

    # For site QC files, fail fast if gross load and PV no longer reconstruct
    # the original net meter. A 2e-5 kW tolerance covers CSV rounding only.
    same_file = Path(BTM_BUILDING_FILE).resolve() == Path(BTM_PV_FILE).resolve()
    if same_file and 'p_native_net_kW' in BTM_Building_Data.columns:
        identity_residual = (
            BTM_Building_Data['p_native_net_kW']
            - (BTM_Building_Data['p_load_kW'] - BTM_PV_Data['p_PV_kW'])
        )
        BTM_NET_IDENTITY_MAX_ERROR_KW = float(identity_residual.abs().max())
        if BTM_NET_IDENTITY_MAX_ERROR_KW > 2e-5:
            raise ValueError(
                'Processed site meter violates p_native_net = p_load_gross - p_PV: '
                f'{BTM_NET_IDENTITY_MAX_ERROR_KW:.3e} kW'
            )
    else:
        BTM_NET_IDENTITY_MAX_ERROR_KW = np.nan

def _get_passive_meter_inputs(time_index):
    idx = pd.DatetimeIndex(time_index)
    zeros = np.zeros(len(idx), dtype=float)
    if not PASSIVE_DER_ENABLED:
        return zeros, zeros
    building = BTM_Building_Data.reindex(idx)
    pv = BTM_PV_Data.reindex(idx)
    if building[['p_load_kW']].isna().any().any():
        missing = building.index[building[['p_load_kW']].isna().any(axis=1)]
        raise ValueError(f'Missing building meter inputs for: {list(missing[:5])}')
    if pv[['p_PV_kW']].isna().any().any():
        missing = pv.index[pv[['p_PV_kW']].isna().any(axis=1)]
        raise ValueError(f'Missing PV meter inputs for: {list(missing[:5])}')
    p_load = BTM_LOAD_SCALE * building['p_load_kW'].to_numpy(float)
    p_pv = BTM_PV_SCALE * pv['p_PV_kW'].to_numpy(float)
    if (p_load < -1e-6).any() or (p_pv < -1e-6).any():
        raise ValueError('Processed gross building load and PV generation must be nonnegative')
    return p_load, p_pv

print(
    f'Passive building/PV retail meter enabled: {PASSIVE_DER_ENABLED}; '
    f'net-meter identity max error: {BTM_NET_IDENTITY_MAX_ERROR_KW if PASSIVE_DER_ENABLED else "n/a"}; '
    f'output: {VERSION_DIR}'
)


EV sensitivity input: 2025Data/EV_data/UCSD_AllSites_Merge_PostProcessedSession_QC.csv
Number of Intervals: 374890
Test year: 2025
Date range: 2025-01-01 03:15:00 to 2025-07-31 21:00:00
Unique sites: 23
Unique chargers: 440
Passive building/PV retail meter enabled: False; output: Results_Rolling


Define function for plotting and testing

In [4]:
# Helper: generate daily fig01 inside the day loop
def plot_daily_solver_choice_figures(
    TheDate_Day0,
    dt_h,
    Solver_Outputs,
    c_e_TOU_AL,
    Bid_Pr_DA,
    Bid_Pr_RT,
    AS_Pr_RU_DA,
    AS_Pr_RU_RT,
    AS_Pr_RD_DA,
    AS_Pr_RD_RT,
    AS_Pr_SP_DA,
    AS_Pr_SP_RT,
    AS_Pr_NSP_DA,
    AS_Pr_NSP_RT,
    alpha_RU,
    alpha_RD,
    alpha_SP,
    alpha_NSP,
    run_tag,
    M_Th_NCD,
    M_Th_PD,
    P_BESS_max,
    P_EV_max,
    mpc_version_label=None,
    forecast_label=None,
    threshold_case_idx=0,
):
    from matplotlib.ticker import MaxNLocator
    res = Solver_Outputs
    def _fit(vec, n):
        arr = np.asarray(vec, dtype=float).flatten()
        if arr.size == n:
            return arr
        if arr.size == 1:
            return np.full(n, float(arr[0]))
        if arr.size < n:
            return np.pad(arr, (0, n - arr.size), mode="edge")
        return arr[:n]
    n = len(np.asarray(res["p_GI"], dtype=float).flatten())
    x = np.arange(n)
    time_index = pd.date_range(start=TheDate_Day0, periods=n, freq=f"{int(round(dt_h * 60))}min")
    hour_step = max(1, int(round(2.0 / dt_h)))
    ticks = np.arange(0, n, hour_step)
    tick_labels = [time_index[i].strftime("%H:%M") for i in ticks]
    p_da = _fit(res["p_DA"], n)
    p_rt = _fit(res["p_RT"], n)
    c_ru_da_raw = np.abs(_fit(res["c_RU_DA"], n))
    c_ru_rt_raw = _fit(res["c_RU_RT"], n)
    c_rd_da_raw = np.abs(_fit(res["c_RD_DA"], n))
    c_rd_rt_raw = _fit(res["c_RD_RT"], n)
    c_sp_da_raw = np.abs(_fit(res["c_SP_DA"], n))
    c_sp_rt_raw = _fit(res["c_SP_RT"], n)
    c_nsp_da_raw = np.abs(_fit(res["c_NSP_DA"], n))
    c_nsp_rt_raw = _fit(res["c_NSP_RT"], n)
    p_ch = np.abs(_fit(res["p_ch_BESS"], n))
    p_dch = np.abs(_fit(res["p_dch_BESS"], n))
    p_ch_wm = np.abs(_fit(res["p_ch_BESS_WM"], n))
    p_dch_wm = np.abs(_fit(res["p_dch_BESS_WM"], n))
    p_ch_nwm = np.abs(_fit(res["p_ch_BESS_NWM"], n))
    p_dch_nwm = np.abs(_fit(res["p_dch_BESS_NWM"], n))
    p_bess = _fit(res["p_BESS"], n)
    p_ev = _fit(res["p_EV"], n)
    p_gi = _fit(res["p_GI"], n)
    soc = _fit(res["soc_BESS"], n)
    p_dch_wm_neg = -p_dch_wm
    p_dch_nwm_neg = -p_dch_nwm
    soc_pct = soc * 100.0
    bid_pr_da = _fit(Bid_Pr_DA, n)
    bid_pr_rt = _fit(Bid_Pr_RT, n)
    as_ru_da = _fit(AS_Pr_RU_DA, n)
    as_ru_rt = _fit(AS_Pr_RU_RT, n)
    as_rd_da = _fit(AS_Pr_RD_DA, n)
    as_rd_rt = _fit(AS_Pr_RD_RT, n)
    as_sp_da = _fit(AS_Pr_SP_DA, n)
    as_sp_rt = _fit(AS_Pr_SP_RT, n)
    as_nsp_da = _fit(AS_Pr_NSP_DA, n)
    as_nsp_rt = _fit(AS_Pr_NSP_RT, n)
    tou_price = _fit(c_e_TOU_AL, n)
    tp_lmp_da = bid_pr_da
    tp_lmp_rt = bid_pr_rt
    tp_ru_da = bid_pr_rt * alpha_RU + as_ru_da
    tp_ru_rt = bid_pr_rt * alpha_RU + as_ru_rt
    # RD activation reduces exported energy, so its energy component is negative.
    tp_rd_da = as_rd_da - bid_pr_rt * alpha_RD
    tp_rd_rt = as_rd_rt - bid_pr_rt * alpha_RD
    tp_sp_da = bid_pr_rt * alpha_SP + as_sp_da
    tp_sp_rt = bid_pr_rt * alpha_SP + as_sp_rt
    tp_nsp_da = bid_pr_rt * alpha_NSP + as_nsp_da
    tp_nsp_rt = bid_pr_rt * alpha_NSP + as_nsp_rt
    # --- Save daily price table (once per day — same prices for all 3 WM_Mode cases) ---
    price_table_dir = Path(VERSION_DIR) / "Plots/Daily_Price_Tables" / time_index[0].strftime("%Y%m")
    price_table_dir.mkdir(parents=True, exist_ok=True)
    price_table_path = price_table_dir / f"daily_prices_{time_index[0].strftime('%Y%m%d')}.csv"
    df_prices = pd.DataFrame({
        "time": time_index,
        "TOU_$/kWh": np.round(tou_price, 4),
        "LMP_DA_$/kWh": np.round(tp_lmp_da, 4),
        "LMP_RT_$/kWh": np.round(tp_lmp_rt, 4),
        "AS_RU_DA_$/kWh": np.round(as_ru_da, 4),
        "AS_RU_RT_$/kWh": np.round(as_ru_rt, 4),
        "AS_RD_DA_$/kWh": np.round(as_rd_da, 4),
        "AS_RD_RT_$/kWh": np.round(as_rd_rt, 4),
        "AS_SP_DA_$/kWh": np.round(as_sp_da, 4),
        "AS_SP_RT_$/kWh": np.round(as_sp_rt, 4),
        "AS_NSP_DA_$/kWh": np.round(as_nsp_da, 4),
        "AS_NSP_RT_$/kWh": np.round(as_nsp_rt, 4),
        "TP_RU_DA_$/kWh": np.round(tp_ru_da, 4),
        "TP_RU_RT_$/kWh": np.round(tp_ru_rt, 4),
        "TP_RD_DA_$/kWh": np.round(tp_rd_da, 4),
        "TP_RD_RT_$/kWh": np.round(tp_rd_rt, 4),
        "TP_SP_DA_$/kWh": np.round(tp_sp_da, 4),
        "TP_SP_RT_$/kWh": np.round(tp_sp_rt, 4),
        "TP_NSP_DA_$/kWh": np.round(tp_nsp_da, 4),
        "TP_NSP_RT_$/kWh": np.round(tp_nsp_rt, 4),
    })
    df_prices.to_csv(price_table_path, index=False, float_format="%.4f")
    vol = np.abs(np.diff(p_bess, prepend=p_bess[0]))
    turn_candidates = [
        i
        for i in range(1, n - 1)
        if (p_bess[i] - p_bess[i - 1]) * (p_bess[i + 1] - p_bess[i]) <= 0
    ]
    ranked = sorted(set(turn_candidates), key=lambda i: -float(vol[i]))
    if not ranked:
        ranked = [int(i) for i in np.argsort(-vol)]
    key_idx = []
    min_gap = max(2, int(round(0.75 / dt_h)))
    for idx in ranked:
        idx = int(idx)
        if all(abs(idx - j) >= min_gap for j in key_idx):
            key_idx.append(idx)
        if len(key_idx) >= 6:
            break
    key_idx = sorted(key_idx)
    c_e_tou = _fit(c_e_TOU_AL, n)
    wm_signal = tp_lmp_da + tp_lmp_rt + tp_ru_da + tp_ru_rt + tp_rd_da + tp_rd_rt + tp_sp_da + tp_sp_rt + tp_nsp_da + tp_nsp_rt
    reason_lines = []
    for num, idx in enumerate(key_idx, 1):
        mode = "dch" if p_bess[idx] < 0 else "ch"
        total_flow = p_dch[idx] if mode == "dch" else p_ch[idx]
        wm_comp = p_dch_wm[idx] if mode == "dch" else p_ch_wm[idx]
        nwm_comp = p_dch_nwm[idx] if mode == "dch" else p_ch_nwm[idx]
        if total_flow <= 1e-9:
            wm_nwm = "NWM"
        elif wm_comp <= 1e-6 and nwm_comp > 1e-6:
            wm_nwm = "NWM"
        elif nwm_comp <= 1e-6 and wm_comp > 1e-6:
            wm_nwm = "WM"
        else:
            wm_nwm = "WM" if wm_comp >= nwm_comp else "NWM"
        tou_pct = 100 * np.mean(c_e_tou <= c_e_tou[idx])
        wm_pct = 100 * np.mean(wm_signal <= wm_signal[idx])
        if wm_nwm == "WM":
            if mode == "dch":
                reason = f"WM signal high ({wm_pct:.0f}pctl) -> WM discharge"
            else:
                reason = f"WM signal low ({wm_pct:.0f}pctl) -> WM charge"
        else:
            if mode == "dch":
                reason = f"TOU high ({tou_pct:.0f}pctl) -> NWM discharge"
            else:
                reason = f"TOU low ({tou_pct:.0f}pctl) -> NWM charge"
        reason_lines.append(f"{num}) {time_index[idx].strftime('%H:%M')} {wm_nwm}-{mode}: {reason}")
    note_text = "Reasons for battery actions:\n" + "\n".join(reason_lines) if reason_lines else "No actions for this day."
    # BESS cycle count under throughput cap: dt_h * sum(ch + dch) <= 2*C*(SOCmax-SOCmin)
    throughput_kwh = dt_h * float(np.sum(p_ch + p_dch))
    cycle_den = np.nan
    try:
        cycle_den = 2.0 * float(C_BESS) * float(SOC_BESS_max - SOC_BESS_min)
    except Exception:
        cycle_den = np.nan
    cycle_count = throughput_kwh / cycle_den if np.isfinite(cycle_den) and cycle_den > 0 else np.nan
    def _panel_tag(ax, txt):
        ax.text(-0.05, 1.05, txt, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom")
    def _smart_legend(ax, handles=None, labels=None, max_cols=4):
        if handles is None or labels is None:
            handles, labels = ax.get_legend_handles_labels()
        n_items = len(labels)
        if n_items == 0:
            return None
        longest = max(len(str(label)) for label in labels)
        width_cap = 3 if longest > 18 else max_cols
        ncol = max(1, min(width_cap, (n_items + 1) // 2))
        fontsize = 9.5 if n_items > 8 else 10.5
        return ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.32), ncol=ncol, fontsize=fontsize)
    plot_dir = Path(VERSION_DIR) / f"Plots/Solver_{TARGET_SAVE}_Choices" / run_tag / time_index[0].strftime("%Y%m%d")
    plot_dir.mkdir(parents=True, exist_ok=True)
    for old in plot_dir.glob("fig*.png"):
        old.unlink(missing_ok=True)
    # Build copy-friendly table for every timestep with BESS activity
    activity_mask = (p_ch + p_dch) > 1e-6
    wm_pct_all = np.array([100 * np.mean(wm_signal <= v) for v in wm_signal], dtype=float)
    tou_pct_all = np.array([100 * np.mean(c_e_tou <= v) for v in c_e_tou], dtype=float)
    def _classify_channel(k):
        if p_bess[k] >= 1e-6:
            if p_ch_wm[k] > 1e-6 and p_ch_nwm[k] > 1e-6:
                return "CH_MIX"
            if p_ch_wm[k] > 1e-6:
                return "CH_WM"
            if p_ch_nwm[k] > 1e-6:
                return "CH_NWM"
            return "CH"
        if p_bess[k] <= -1e-6:
            if p_dch_wm[k] > 1e-6 and p_dch_nwm[k] > 1e-6:
                return "DCH_MIX"
            if p_dch_wm[k] > 1e-6:
                return "DCH_WM"
            if p_dch_nwm[k] > 1e-6:
                return "DCH_NWM"
            return "DCH"
        return "IDLE"
    def _driver(channel, k):
        if channel == "DCH_WM":
            return "WM high"
        if channel == "CH_WM":
            return "WM low"
        if channel == "DCH_NWM":
            return "TOU high"
        if channel == "CH_NWM":
            return "TOU low"
        if channel.endswith("MIX"):
            return "Mixed"
        return "-"
    activity_rows = []
    for k in np.where(activity_mask)[0]:
        ch = _classify_channel(int(k))
        activity_rows.append({
            "time": time_index[k].strftime("%H:%M"),
            "channel_action": ch,
            "p_BESS_net(kW)": float(p_bess[k]),
            "p_ch_WM(kW)": float(p_ch_wm[k]),
            "p_dch_WM(kW)": float(p_dch_wm[k]),
            "p_ch_NWM(kW)": float(p_ch_nwm[k]),
            "p_dch_NWM(kW)": float(p_dch_nwm[k]),
            "SOC(%)": float(soc_pct[k]),
            "TOU($/kWh)": float(c_e_tou[k]),
            "LMP_DA($/kWh)": float(tp_lmp_da[k]),
            "LMP_RT($/kWh)": float(tp_lmp_rt[k]),
            "WM_signal($/kWh)": float(wm_signal[k]),
            "driver": _driver(ch, int(k)),
        })
    df_activity = pd.DataFrame(activity_rows)
    if not df_activity.empty:
        num_cols = [c for c in df_activity.columns if c not in ["time", "channel_action", "driver"]]
        df_activity[num_cols] = df_activity[num_cols].round(2)
    activity_csv = plot_dir / f"Bess_activity_table_{run_tag}.csv"
    if df_activity.empty:
        activity_txt_text = "No BESS activity timesteps for this day."
    else:
        activity_txt_text = df_activity.to_string(index=False)
    df_activity.to_csv(activity_csv, index=False, float_format='%.2f')
    fig = plt.figure(figsize=(28, 15), constrained_layout=False)
    date_str = time_index[0].strftime("%Y-%m-%d")
    mpc_txt = mpc_version_label or globals().get("MPC_VERSION_LABEL", "MPC")
    fc_txt = forecast_label or ("Perfect" if globals().get("Fc_SessionkWh", "").startswith("Perfect") else "Persistence")
    fig.suptitle(f"{mpc_txt} {TARGET_SAVE} 6-Panel | Date: {date_str} | Forecast: {fc_txt} | Mode: {WM_Mode}", fontsize=22, fontweight="bold", y=0.98)
    gs = fig.add_gridspec(3, 2, hspace=0.60, wspace=0.28, top=0.90, bottom=0.08)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1], sharex=ax1)
    ax3 = fig.add_subplot(gs[1, 0], sharex=ax1)
    ax4 = fig.add_subplot(gs[1, 1], sharex=ax1)
    ax5 = fig.add_subplot(gs[2, 0], sharex=ax1)
    ax6 = fig.add_subplot(gs[2, 1], sharex=ax1)
    for ax in [ax1, ax2, ax3, ax4, ax5, ax6]:
        ax.set_xticks(ticks)
    ax1.plot(x, tp_ru_da, color="#2ca02c", linewidth=1.4, linestyle="-", label="TP RU DA")
    ax1.plot(x, tp_ru_rt, color="#2ca02c", linewidth=1.0, linestyle="--", alpha=0.85, label="TP RU RT")
    ax1.plot(x, tp_rd_da, color="#d62728", linewidth=1.4, linestyle="-", label="TP RD DA")
    ax1.plot(x, tp_rd_rt, color="#d62728", linewidth=1.0, linestyle="--", alpha=0.85, label="TP RD RT")
    ax1.plot(x, tp_sp_da, color="#9467bd", linewidth=1.4, linestyle="-", label="TP SP DA")
    ax1.plot(x, tp_sp_rt, color="#9467bd", linewidth=1.0, linestyle="--", alpha=0.85, label="TP SP RT")
    ax1.plot(x, tp_nsp_da, color="#8c564b", linewidth=1.4, linestyle="-", label="TP NSP DA")
    ax1.plot(x, tp_nsp_rt, color="#8c564b", linewidth=1.0, linestyle="--", alpha=0.85, label="TP NSP RT")
    ax1.plot(x, tp_lmp_da, color="#1f77b4", linewidth=1.9, linestyle="-", alpha=0.95, label="LMP DA")
    ax1.plot(x, tp_lmp_rt, color="#1f77b4", linewidth=1.2, linestyle="--", alpha=0.95, label="LMP RT")
    ax1.axhline(0, color="black", linewidth=0.9)
    ax1_tou = ax1.twinx()
    ax1_tou.plot(x, tou_price, color="#00a5a5", linewidth=2.2, linestyle="-.", label="TOU")
    ax1.set_ylabel("WM price ($/kWh)")
    ax1_tou.set_ylabel("TOU ($/kWh)", color="#008b8b")
    ax1_tou.tick_params(axis="y", labelcolor="#008b8b")
    h1, l1 = ax1.get_legend_handles_labels()
    h1_tou, l1_tou = ax1_tou.get_legend_handles_labels()
    _smart_legend(ax1, h1 + h1_tou, l1 + l1_tou, max_cols=5)
    ax1.tick_params(labelbottom=False)
    _panel_tag(ax1, "(a) Total Expected Price")
    ax2.plot(x, as_ru_da, color="#2ca02c", linewidth=1.4, linestyle="-", label="Raw RU DA")
    ax2.plot(x, as_ru_rt, color="#2ca02c", linewidth=1.0, linestyle="--", alpha=0.85, label="Raw RU RT")
    ax2.plot(x, as_rd_da, color="#d62728", linewidth=1.4, linestyle="-", label="Raw RD DA")
    ax2.plot(x, as_rd_rt, color="#d62728", linewidth=1.0, linestyle="--", alpha=0.85, label="Raw RD RT")
    ax2.plot(x, as_sp_da, color="#9467bd", linewidth=1.4, linestyle="-", label="Raw SP DA")
    ax2.plot(x, as_sp_rt, color="#9467bd", linewidth=1.0, linestyle="--", alpha=0.85, label="Raw SP RT")
    ax2.plot(x, as_nsp_da, color="#8c564b", linewidth=1.4, linestyle="-", label="Raw NSP DA")
    ax2.plot(x, as_nsp_rt, color="#8c564b", linewidth=1.0, linestyle="--", alpha=0.85, label="Raw NSP RT")
    ax2.plot(x, tp_lmp_da, color="#1f77b4", linewidth=1.9, linestyle="-", alpha=0.95, label="LMP DA")
    ax2.plot(x, tp_lmp_rt, color="#1f77b4", linewidth=1.2, linestyle="--", alpha=0.95, label="LMP RT")
    ax2.axhline(0, color="black", linewidth=0.9)
    ax2_tou = ax2.twinx()
    ax2_tou.plot(x, tou_price, color="#00a5a5", linewidth=2.2, linestyle="-.", label="TOU")
    ax2.set_ylabel("WM price ($/kWh)")
    ax2_tou.set_ylabel("TOU ($/kWh)", color="#008b8b")
    ax2_tou.tick_params(axis="y", labelcolor="#008b8b")
    h2, l2 = ax2.get_legend_handles_labels()
    h2_tou, l2_tou = ax2_tou.get_legend_handles_labels()
    _smart_legend(ax2, h2 + h2_tou, l2 + l2_tou, max_cols=5)
    ax2.tick_params(labelbottom=False)
    _panel_tag(ax2, "(b) Raw Prices")
    def _draw_stacked_bars(ax, components, ylabel):
        bar_w = 0.88
        pos_base = np.zeros(n)
        neg_base = np.zeros(n)
        for label, vec, color in components:
            vec = np.asarray(vec, dtype=float)
            pos = np.where(vec > 0, vec, 0.0)
            neg = np.where(vec < 0, vec, 0.0)
            added_label = False
            if np.any(pos):
                ax.bar(x, pos, width=bar_w, bottom=pos_base, color=color, alpha=0.82, label=label)
                pos_base += pos
                added_label = True
            if np.any(neg):
                this_label = "_nolegend_" if added_label else label
                ax.bar(x, neg, width=bar_w, bottom=neg_base, color=color, alpha=0.82, label=this_label)
                neg_base += neg
                added_label = True
            if not added_label:
                ax.bar(x, np.zeros(n), width=bar_w, color=color, alpha=0.82, label=label)
        ax.axhline(0, color="black", linewidth=0.9)
        ax.set_ylabel(ylabel)
        # 在调用 _draw_stacked_bars 之后添加：
        all_pos_sums = pos_base  # 这是函数内部累加后的最高点
        all_neg_sums = neg_base  # 这是函数内部累加后的最低点
        y_max = np.max(all_pos_sums) if np.any(all_pos_sums > 0) else 0
        y_min = np.min(all_neg_sums) if np.any(all_neg_sums < 0) else 0
        # 设置 5% 的缓冲，防止贴边，但又不至于留白太多
        if y_max == 0 and y_min == 0:
            ax.set_ylim(-1, 1) # 防错
        else:
            # 向上取一点点空间，向下取一点点空间
            ax.set_ylim(y_min * (1.05 if y_min < 0 else 0.95), y_max * 1.05)
    # Show the fixed DA commitment and the RT increment/decrement separately.
    # Their physical sum is enforced in the model, but is intentionally not plotted here.
    _baseline_kw = _fit(res["Baseline(kW)"], n)
    _P_BESS_max_local = float(P_BESS_max)
    _P_EV_max_local = P_EV_max
    _p_up_line = _fit(res.get("p_up_bound(kW)", _P_BESS_max_local + _baseline_kw), n)
    _p_down_line = _fit(res.get("p_down_bound(kW)", np.maximum(_P_BESS_max_local - _baseline_kw + _P_EV_max_local, 0.0)), n)
    def _visible_components(components, tol=1e-5):
        return [(lbl, np.asarray(data, dtype=float), color) for lbl, data, color in components if np.nanmax(np.abs(np.asarray(data, dtype=float))) > tol]
    def _draw_signed_stacked_bars(ax, components, ylabel):
        # Match the original penalty-version panels: every named bid remains its own bar.
        # Positive and negative contributions are stacked independently from zero.
        positive_bottom = np.zeros(n)
        negative_bottom = np.zeros(n)
        for label, values, color in _visible_components(components):
            values = np.asarray(values, dtype=float)
            positive = np.where(values > 0.0, values, 0.0)
            negative = np.where(values < 0.0, values, 0.0)
            ax.bar(x, positive, width=0.88, bottom=positive_bottom, color=color, alpha=0.82, label=label)
            ax.bar(x, negative, width=0.88, bottom=negative_bottom, color=color, alpha=0.82)
            positive_bottom += positive
            negative_bottom += negative
        ax.axhline(0, color="black", linewidth=0.9)
        ax.set_ylabel(ylabel)
        return positive_bottom, negative_bottom
    up_components_c = [
        ("c_RU_DA", c_ru_da_raw, "#2ca02c"),
        ("c_RU_RT", c_ru_rt_raw, "#98df8a"),
        ("c_SP_DA", c_sp_da_raw, "#9467bd"),
        ("c_SP_RT", c_sp_rt_raw, "#c5b0d5"),
        ("c_NSP_DA", c_nsp_da_raw, "#8c564b"),
        ("c_NSP_RT", c_nsp_rt_raw, "#c49c94"),
    ]
    down_components_c = [
        ("c_RD_DA", c_rd_da_raw, "#d62728"),
        ("c_RD_RT", c_rd_rt_raw, "#ff9896"),
    ]
    components_c = [
        ("p_DA", p_da, "#1f77b4"),
        ("p_RT", p_rt, "#17becf"),
        *up_components_c,
        *((label, -vec, color) for label, vec, color in down_components_c),
    ]
    upper_bid_stack, lower_bid_stack = _draw_signed_stacked_bars(ax3, components_c, "Capacity / Bid (kW)")
    # Plot the model's physical capability bounds exactly. Do not expand these
    # lines to contain the separately displayed DA/RT accounting bars.
    upper_plot_bound = np.asarray(_p_up_line, dtype=float)
    lower_plot_bound = -np.asarray(_p_down_line, dtype=float)
    ax3.plot(x, upper_plot_bound, color="#e6550d", linewidth=2.0, linestyle="-", label="upper bound", zorder=6)
    ax3.plot(x, lower_plot_bound, color="#3182bd", linewidth=2.0, linestyle="-", label="lower bound", zorder=6)
    _ymin, _ymax = ax3.get_ylim()
    _ymin = min(_ymin, float(np.nanmin(lower_plot_bound)))
    _ymax = max(_ymax, float(np.nanmax(upper_plot_bound)))
    _ypad = 0.05 * max(_ymax - _ymin, 1.0)
    ax3.set_ylim(_ymin - _ypad, _ymax + _ypad)
    # Validate the actual settlement position against the real model bounds separately.
    p_actual = p_da + p_rt
    physical_upper = p_actual + sum((vec for _, vec, _ in _visible_components(up_components_c)), np.zeros(n))
    physical_lower = p_actual - sum((vec for _, vec, _ in _visible_components(down_components_c)), np.zeros(n))
    _upper_violation = float(np.nanmax(physical_upper - _p_up_line))
    _lower_violation = float(np.nanmax((-_p_down_line) - physical_lower))
    if max(_upper_violation, _lower_violation) > 1e-3:
        raise ValueError(f"Settlement position exceeds physical bounds: upper={_upper_violation:.6g} kW, lower={_lower_violation:.6g} kW")
    _smart_legend(ax3, max_cols=6)
    ax3.yaxis.set_major_locator(MaxNLocator(nbins=8))
    ax3.tick_params(labelbottom=False)
    _panel_tag(ax3, "(c) Bids")
    up_components_d = [
        ("Obl RU_DA", c_ru_da_raw * alpha_RU, "#2ca02c"),
        ("Obl RU_RT", c_ru_rt_raw * alpha_RU, "#98df8a"),
        ("Obl SP_DA", c_sp_da_raw * alpha_SP, "#9467bd"),
        ("Obl SP_RT", c_sp_rt_raw * alpha_SP, "#c5b0d5"),
        ("Obl NSP_DA", c_nsp_da_raw * alpha_NSP, "#8c564b"),
        ("Obl NSP_RT", c_nsp_rt_raw * alpha_NSP, "#c49c94"),
    ]
    down_components_d = [
        ("Obl RD_DA", c_rd_da_raw * alpha_RD, "#d62728"),
        ("Obl RD_RT", c_rd_rt_raw * alpha_RD, "#ff9896"),
    ]
    components_d = [
        ("Obl p_DA", p_da, "#1f77b4"),
        ("Obl p_RT", p_rt, "#17becf"),
        *up_components_d,
        *((label, -vec, color) for label, vec, color in down_components_d),
    ]
    _draw_signed_stacked_bars(ax4, components_d, "Energy Obligation")
    net_obligation = sum((np.asarray(vec, dtype=float) for _, vec, _ in _visible_components(components_d)), np.zeros(n))
    ax4.plot(x, net_obligation, color="black", linewidth=1.5, label="Net Obligation")
    _smart_legend(ax4, max_cols=6)
    ax4.yaxis.set_major_locator(MaxNLocator(nbins=8))
    ax4.tick_params(labelbottom=False)
    _panel_tag(ax4, "(d) Energy Obligation (α * Capacity)")
    wm_flow = float(np.max(np.abs(p_ch_wm)) + np.max(np.abs(p_dch_wm)))
    if wm_flow > 1e-6:
        ax5.bar(x, p_ch_wm, width=0.92, color="#1f77b4", alpha=0.72, label="BESS ch WM")
        ax5.bar(x, p_ch_nwm, width=0.92, bottom=p_ch_wm, color="#17becf", alpha=0.72, label="BESS ch NWM")
        ax5.bar(x, p_dch_wm_neg, width=0.92, color="#d62728", alpha=0.72, label="BESS dch WM")
        ax5.bar(x, p_dch_nwm_neg, width=0.92, bottom=p_dch_wm_neg, color="#ff9896", alpha=0.72, label="BESS dch NWM")
    else:
        ax5.bar(x, p_ch_nwm, width=0.92, color="#17becf", alpha=0.72, label="BESS ch NWM")
        ax5.bar(x, p_dch_nwm_neg, width=0.92, color="#ff9896", alpha=0.72, label="BESS dch NWM")
    ax5.axhline(0, color="black", linewidth=0.9)
    ax5.set_ylabel("BESS Power (kW)")
    ax5.text(
        0.02,
        0.03,
        f"Cycle count: {cycle_count:.3f} / 1.000",
        transform=ax5.transAxes,
        ha="left",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="lightgrey", boxstyle="round,pad=0.25"),
    )
    ax5_r = ax5.twinx()
    ax5_r.plot(x, soc_pct, color="#2ca02c", linewidth=1.8, alpha=0.95, label="SOC")
    ax5_r.set_ylabel("SOC (%)", color="#2ca02c")
    ax5_r.tick_params(axis="y", labelcolor="#2ca02c")
    for num, idx in enumerate(key_idx, 1):
        yv = soc_pct[idx]
        ax5_r.scatter(idx, yv, color="black", s=25, zorder=6)
        ax5_r.text(idx, yv + 1.8, str(num), fontsize=12, fontweight="bold", color="black", ha="center", va="bottom")
    h5, l5 = ax5.get_legend_handles_labels()
    h5_r, l5_r = ax5_r.get_legend_handles_labels()
    _smart_legend(ax5, h5 + h5_r, l5 + l5_r)
    ax5.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax5.set_xlabel("Time of day")
    _panel_tag(ax5, "(e) BESS Action & SOC")
    b_ev = _fit(res["Baseline(kW)"], n)
    ax6.step(x, p_ev, where="post", color="#1f77b4", linewidth=2.2, alpha=0.9, label="p_EV")
    ax6.step(x, b_ev, where="post", color="#6c757d", linewidth=1.8, linestyle="--", alpha=0.95, label="Baseline")
    ax6.fill_between(x, p_ev, b_ev, where=(p_ev >= b_ev), step="post", color="#f4a261", alpha=0.20, label="pEV > Baseline")
    ax6.fill_between(x, p_ev, b_ev, where=(p_ev < b_ev), step="post", color="#90caf9", alpha=0.20, label="pEV < Baseline")
    ax6.step(x, p_bess, where="post", color="#d62728", linewidth=2.0, alpha=0.9, label="p_BESS_net")
    ax6.step(x, p_gi, where="post", color="#2ca02c", linewidth=2.0, alpha=0.9, label="p_GI")
    ax6.axhline(0, color="black", linewidth=0.9)
    ncd_val = M_Th_NCD[threshold_case_idx]
    pd_val = M_Th_PD[threshold_case_idx]
    ax6.axhline(ncd_val, color="#ff9f1c", linestyle="--", alpha=0.8, linewidth=1.5, label=f"NCD Threshold")
    ax6.axhline(pd_val, color="#e76f51", linestyle="--", alpha=0.8, linewidth=1.5, label=f"PD Threshold")
    ax6.set_ylabel("Net Power (kW)")
    ax6.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax6.set_xlabel("Time of day")
    _smart_legend(ax6)
    _panel_tag(ax6, "(f) Overall System Power")
    def _to_scalar(v):
        if np.isscalar(v):
            return float(v)
        arr = np.asarray(v, dtype=float).flatten()
        return float(arr[0]) if arr.size > 0 else np.nan
    if globals().get("SHOW_6PANEL_SUMMARY_BOX", False):
        da_summary = (
            f"{TARGET_SAVE} Summary:\n"
            f"CostOpt=${_to_scalar(res['Cost_Opt']):.2f}, CostPD=${_to_scalar(res['Cost_PD']):.2f}, "
            f"CostNCD=${_to_scalar(res['Cost_NCD']):.2f}, CostTOU=${_to_scalar(res['Cost_TOU']):.2f}\n"
            f"RevenueEV=${_to_scalar(res['Revenue_EV']):.2f}, RevenueWM=${_to_scalar(res['Revenue_WM']):.2f}"
        )
        ax6.text(
            0.5,
            -0.45,
            da_summary,
            transform=ax6.transAxes,
            ha="center",
            va="top",
            fontsize=12,
            weight="normal",
            linespacing=1.5,
            bbox=dict(facecolor="#f8f9fa", alpha=0.9, edgecolor="lightgrey", boxstyle="round,pad=0.8"),
        )
    def align_yaxis(ax_left, ax_right, margin_factor=1.45):
        y1_min, y1_max = ax_left.get_ylim()
        y2_min, y2_max = ax_right.get_ylim()
        new_min = min(y1_min, y2_min)
        new_max = max(y1_max, y2_max)
        ax_left.set_ylim(new_min * margin_factor if new_min < 0 else new_min / margin_factor, new_max * margin_factor)
        ax_right.set_ylim(new_min * margin_factor if new_min < 0 else new_min / margin_factor, new_max * margin_factor)
    align_yaxis(ax3, ax4, margin_factor=1.45)
    fig01 = plot_dir / f"{TARGET_SAVE}_6panel_{run_tag}.png"
    fig.savefig(fig01, bbox_inches="tight", facecolor="white", dpi=300)
    plt.close(fig)
    return [str(fig01), str(activity_csv)]
def save_old_stairplot_beautified(
    H_Start_RT,
    TimeSeries_real,
    dt_m_EV,
    dt_h,
    EnergyDemand_Step_V0G_TheDates,
    EnergyDemand_Step_V1G_TheDates,
    p_GI_DA,
    dispatch_t0,
    Dispatch,
    p_BESS_value_RT,
    Baseline_96,
    EventHour_96,
    D_Th_NCD_96,
    D_Th_PD_96,
    Fc_SessionkWh,
    Fc_NumbEV,
    Fc_AtArrival,
    WM_Mode,
    cycle_label=None,
):
    from pathlib import Path
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    t_core = pd.to_datetime(TimeSeries_real[0])
    t_edges = TimeSeries_real[0].copy()
    t_edges.append(TimeSeries_real[0][-1] + timedelta(minutes=dt_m_EV))
    pw_v0g_max = np.asarray(EnergyDemand_Step_V0G_TheDates[0], dtype=float) / dt_h
    pw_v1g = np.asarray(EnergyDemand_Step_V1G_TheDates[0], dtype=float) / dt_h
    pw_opt_da = np.asarray(p_GI_DA, dtype=float)
    pw_opt_t0 = np.asarray(dispatch_t0[0], dtype=float)
    pw_opt_imp_base = np.asarray((Dispatch[0][0].sum(axis=1)) / dt_h + p_BESS_value_RT[0, :], dtype=float)
    pw_opt_imp_case1 = np.asarray((Dispatch[0][1].sum(axis=1)) / dt_h + p_BESS_value_RT[1, :], dtype=float)
    pw_baseline_base = np.asarray(Baseline_96[0].reshape(96) / dt_h, dtype=float)
    pw_baseline_case1 = np.asarray(Baseline_96[1].reshape(96) / dt_h, dtype=float)
    def _stairs(y):
        y = np.asarray(y, dtype=float)
        return np.concatenate(([y[0]], y))
    event_base = np.asarray(EventHour_96[0]).reshape(-1)
    event_case1 = np.asarray(EventHour_96[1]).reshape(-1)
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.step(t_edges, _stairs(pw_opt_da), where="post", color="#0b5ed7", linewidth=2.8, label="V1G opt DA (100%)")
    ax.step(t_edges, _stairs(pw_opt_t0), where="post", color="#5bc0eb", linewidth=2.4, label="V1G opt RT t0 (100%)")
    ax.step(t_edges, _stairs(pw_opt_imp_base), where="post", color="#c1121f", linestyle="--", linewidth=2.2, label="V1G opt imp (100%)")
    ax.step(t_edges, _stairs(pw_opt_imp_case1), where="post", color="#f28482", linestyle="--", linewidth=2.2, label="V1G opt imp (eta%)")
    ax.step(t_edges, _stairs(pw_baseline_base), where="post", color="#2a9d8f", linestyle=":", linewidth=2.0, label="Baseline (100%)")
    ax.step(t_edges, _stairs(pw_baseline_case1), where="post", color="#8ecae6", linestyle=":", linewidth=2.0, label="Baseline (eta%)")
    ax.step(t_edges, _stairs(pw_v0g_max), where="post", color="#588157", linewidth=1.8, alpha=0.9, label="V0G")
    ax.step(t_edges, _stairs(pw_v1g), where="post", color="#a7c957", linewidth=1.8, alpha=0.9, label="V1G real")
    ymax = max(
        np.max(pw_opt_da),
        np.max(pw_opt_t0),
        np.max(pw_opt_imp_base),
        np.max(pw_opt_imp_case1),
        np.max(pw_baseline_base),
        np.max(pw_baseline_case1),
        np.max(pw_v0g_max),
        np.max(pw_v1g),
    )
    ymin = min(
        np.min(pw_opt_da),
        np.min(pw_opt_t0),
        np.min(pw_opt_imp_base),
        np.min(pw_opt_imp_case1),
        np.min(pw_baseline_base),
        np.min(pw_baseline_case1),
        np.min(pw_v0g_max),
        np.min(pw_v1g),
    )
    marker_level_base = ymax * 0.96
    marker_level_case1 = ymax * 0.92
    ax.scatter(t_core[event_base > 0.5], np.full(np.sum(event_base > 0.5), marker_level_base), marker="x", color="#d00000", s=28, label="Event hour (100%)")
    ax.scatter(t_core[event_case1 > 0.5], np.full(np.sum(event_case1 > 0.5), marker_level_case1), marker="+", color="#2b9348", s=28, label="Event hour (eta%)")
    ax.plot(t_core[:5], np.full(5, D_Th_NCD_96[0][0]), color="#ff9f1c", marker="8", linestyle="", markersize=6, label="NCD start (100%)")
    ax.plot(t_core[:5], np.full(5, D_Th_NCD_96[1][0]), color="#2a9d8f", marker=">", linestyle="", markersize=5, label="NCD start (eta%)")
    ax.plot(t_core[:5], np.full(5, D_Th_PD_96[0][0]), color="#ff9f1c", marker="s", linestyle="", markersize=6, label="PD start (100%)")
    ax.plot(t_core[:5], np.full(5, D_Th_PD_96[1][0]), color="#2a9d8f", marker="<", linestyle="", markersize=5, label="PD start (eta%)")
    ax.set_title(f"Implemented EV + BESS Dispatch on {H_Start_RT.strftime('%Y/%m/%d')}", fontsize=18, fontweight="bold")
    if cycle_label is not None:
        ax.text(
            0.01,
            0.98,
            f"Cycle count: {cycle_label}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=11,
            fontweight="bold",
            bbox=dict(facecolor="white", alpha=0.80, edgecolor="lightgrey", boxstyle="round,pad=0.25"),
        )
    ax.set_xlabel("Time", fontsize=14)
    ax.set_ylabel("Power Demand (kW)", fontsize=14)
    ax.set_xlim(pd.to_datetime(t_edges[0]), pd.to_datetime(t_edges[-1]))
    ax.set_ylim(ymin - 0.08 * (ymax - ymin + 1e-9), ymax + 0.12 * (ymax - ymin + 1e-9))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.grid(axis="x", linestyle=":", alpha=0.20)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=11, frameon=False)
    fig.tight_layout()
    out_dir = Path(VERSION_DIR) / "Plots/Imp_Stair" / f"{H_Start_RT.strftime('%Y')}_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{WM_Mode}"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{H_Start_RT.strftime('%Y%m%d')}_Implemented_Pw_stair.png"
    fig.savefig(out_path, bbox_inches="tight", facecolor="white", dpi=300)
    plt.close(fig)
    return str(out_path)

Main Loop


In [5]:
# clean all csv files presaved 
base_dir = Path(VERSION_DIR) / 'Dispatch'
RUN_MAIN_LOOP_DIRECT = False  # False: define/load only; True: run this block directly
for f in (base_dir.glob('[01]*') if RUN_MAIN_LOOP_DIRECT else []):
    f.unlink(missing_ok=True)
prefix = f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{WM_Mode}'
_clear_impl_at_start = bool(globals().get('CLEAR_IMPLEMENTATION_AT_RUN_START', True))
if RUN_MAIN_LOOP_DIRECT and _clear_impl_at_start:
    for suffix in ['implementation.csv', 'daily_summary.csv', 'daily_cost.csv']:
        (base_dir / f'{prefix}_{suffix}').unlink(missing_ok=True)
if RUN_MAIN_LOOP_DIRECT:
    if _clear_impl_at_start:
        print("All dispatch files from last implementation have been deleted.\n")
    else:
        print("Existing implementation files preserved for continuous baseline history.\n")
else:
    print("RUN_MAIN_LOOP_DIRECT=False: main loop block loaded without execution.")
# When preserving implementation history, keep only rows before the first requested
# simulation day. This allows a standalone July rerun to learn from June dispatch
# without accidentally reading stale July rows as historical baseline data.
def _trim_existing_csv_before_run_start(out_path, first_timestamp):
    out_path = Path(out_path)
    if (not RUN_MAIN_LOOP_DIRECT) or _clear_impl_at_start or (not out_path.exists()):
        return
    try:
        _df_existing = pd.read_csv(out_path, low_memory=False)
        if 'Interval start' in _df_existing.columns:
            _row_time = pd.to_datetime(_df_existing['Interval start'], errors='coerce')
        elif 'Date' in _df_existing.columns:
            _row_time = pd.to_datetime(_df_existing['Date'].astype(str), format='%Y%m%d', errors='coerce')
        else:
            return
        _rows_before = len(_df_existing)
        _df_existing = _df_existing[_row_time < pd.Timestamp(first_timestamp)]
        _rows_after = len(_df_existing)
        _df_existing.to_csv(out_path, index=False)
    except Exception as _trim_err:
        print(f"Warning: could not trim existing implementation history in {out_path}: {_trim_err}")
# Runtime switches for data-structure / I-O overhead control
SAVE_STEP_DEBUG = False        # skip per-step debug CSV writes under Results_Rolling/Dispatch/0_* and 1_*
ENABLE_RT_TRACE_TABLE = False  # skip expensive per-step trace merges for IntervalkWh_table_Opt_list
# Original helper for incremental CSV writes. It is re-defined later in the notebook
# to enforce two-decimal rounding and a consistent float format.
def append_df_to_csv(df, out_path, index=False):
    out_path = Path(out_path)
    if (not SAVE_STEP_DEBUG) and (('Results_Rolling/Dispatch/0_' in str(out_path)) or ('Results_Rolling/Dispatch/1_' in str(out_path))):
        return
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        df.to_csv(out_path, mode='a', index=index, header=False)
    else:
        df.to_csv(out_path, mode='w', index=index, header=True)
# Read and parse AS files once to avoid repeated daily I/O.
dir_AS_DA = os.path.join("2025Data/AS_DAM/")
dir_AS_RT = os.path.join("2025Data/AS_RTM/")
Data_AS_DA_all = pd.read_csv(dir_AS_DA + 'AS_price_2025_clear.csv')
Data_AS_RT_all = pd.read_csv(dir_AS_RT + 'AS_price_2025_clear.csv')
# Include the single next-year day required by the Dec 31 Rolling horizon, when available.
_as_da_boundary = Path(dir_AS_DA) / 'AS_price_2026_boundary.csv'
_as_rt_boundary = Path(dir_AS_RT) / 'AS_price_2026_boundary.csv'
if _as_da_boundary.exists():
    Data_AS_DA_all = pd.concat([Data_AS_DA_all, pd.read_csv(_as_da_boundary)], ignore_index=True)
if _as_rt_boundary.exists():
    Data_AS_RT_all = pd.concat([Data_AS_RT_all, pd.read_csv(_as_rt_boundary)], ignore_index=True)
# New full-year AS clean files use a fixed, timezone-naive California model clock.
# Legacy timezone-aware files remain supported so older formal results stay readable.
def parse_as_model_clock(series):
    text = series.astype(str).str.strip()
    has_timezone = text.str.contains(r'(?:Z|[+-]\d{2}:\d{2})$', regex=True)
    if has_timezone.all():
        return pd.to_datetime(text, utc=True, errors='raise').dt.tz_convert('America/Los_Angeles').dt.tz_localize(None)
    if (~has_timezone).all():
        return pd.to_datetime(text, errors='raise')
    raise ValueError('AS datetime column mixes timezone-aware and timezone-naive values.')
Data_AS_DA_all["datetime"] = parse_as_model_clock(Data_AS_DA_all["datetime"])
Data_AS_RT_all["datetime"] = parse_as_model_clock(Data_AS_RT_all["datetime"])
if Fc_AtArrival == 'MLatArrival':
    # Cache this lookup table once to avoid repeated per-car file reads.
    UserBess = pd.read_csv("/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Data/PowerFlex input for EV statistics to be plugged into master sheet.csv")
# Main loop for monthly optimization with implementation
Cost_All = pd.DataFrame()
start_time = time.time()  # record start time
RUN_MONTHS_CONFIG = list(globals().get('RUN_MONTHS_CONFIG', [7]))
test_month = int(RUN_MONTHS_CONFIG[0])
SOC_BESS_last_step = np.ones(len(Cases)) * 0.5  # initial SOC at 50%
SOC_BESS_daily_end = np.ones(len(Cases)) * 0.5
# 🌟 新增：动态预测核心模块 (Dynamic Forecasting)
def get_dynamic_ev_forecast(i_t, H, current_real_data, forecast_method='Persistence'):
    # 【装甲1：输入参数合法性校验】防止生成奇葩维度的矩阵
    if H <= 0:
        raise ValueError(f"预测时间跨度 H 必须大于 0，当前值为 {H}")
    i_t = max(0, i_t) # 防止负数索引引发预期外的切片行为
    # 【装甲2：强健的字典键值提取】明确抛出易懂的异常
    try:
        ev_cols_full = list(current_real_data['ev_cols_full'])
        raw_ub = current_real_data['interval_ub_real']
        raw_min = current_real_data['session_min_real']
        raw_max = current_real_data['session_max_real']
    except KeyError as e:
        raise KeyError(f"current_real_data 缺失必要的基础数据键: {e}")
    # 【装甲3：彻底的 NaN 清洗】
    real_ub = raw_ub.reindex(columns=ev_cols_full, fill_value=0.0).fillna(0.0)
    real_min_df = raw_min.reindex(columns=ev_cols_full, fill_value=0.0).fillna(0.0)
    real_max_df = raw_max.reindex(columns=ev_cols_full, fill_value=0.0).fillna(0.0)
    # 【装甲4：防空表提取】
    real_min = real_min_df.iloc[0].values if not real_min_df.empty else np.zeros(len(ev_cols_full))
    real_max = real_max_df.iloc[0].values if not real_max_df.empty else np.zeros(len(ev_cols_full))
    service_min = float(globals().get('SERVICE_LEVEL_MIN', 1.0))
    if service_min < 1.0:
        original_df = current_real_data['session_original_real'].reindex(columns=ev_cols_full, fill_value=0.0).fillna(0.0)
        original_request = original_df.iloc[0].to_numpy(dtype=float) if not original_df.empty else np.zeros(len(ev_cols_full))
        # Subtract a fixed allowable shortfall, NOT a fraction of the remaining demand.
        real_min = np.maximum(real_max - (1.0 - service_min) * original_request, 0.0)
    # Align the next calendar day to the same stable EV column set.
    next_ub_raw = current_real_data.get('next_interval_ub_real', pd.DataFrame())
    next_session_raw = current_real_data.get('next_session_real', pd.DataFrame())
    next_ub = next_ub_raw.reindex(columns=ev_cols_full, fill_value=0.0).fillna(0.0)
    next_session_df = next_session_raw.reindex(columns=ev_cols_full, fill_value=0.0).fillna(0.0)
    next_session = next_session_df.iloc[0].to_numpy(dtype=float) if not next_session_df.empty else np.zeros(len(ev_cols_full))
    # 预测数据同理，加上 fillna 和防空表判断
    fc_ub = current_real_data.get('interval_ub_fc', raw_ub).reindex(columns=ev_cols_full, fill_value=0.0).fillna(0.0)
    fc_min_df = current_real_data.get('session_min_fc', raw_min).reindex(columns=ev_cols_full, fill_value=0.0).fillna(0.0)
    fc_min = fc_min_df.iloc[0].values if not fc_min_df.empty else np.zeros(len(ev_cols_full))
    fc_max_df = current_real_data.get('session_max_fc', raw_max).reindex(columns=ev_cols_full, fill_value=0.0).fillna(0.0)
    fc_max = fc_max_df.iloc[0].values if not fc_max_df.empty else np.zeros(len(ev_cols_full))
    if service_min < 1.0:
        fc_min = service_min * fc_max
    tail_ub_realized_raw = current_real_data.get('tail_ub_realized', None)
    if tail_ub_realized_raw is None:
        tail_ub_realized = pd.DataFrame(columns=ev_cols_full)
    else:
        tail_ub_realized = tail_ub_realized_raw.reindex(columns=ev_cols_full, fill_value=0.0).fillna(0.0)
    # 初始化预测结果矩阵
    ev_ub_pred = np.zeros((H, len(ev_cols_full)), dtype=float)
    ev_min_pred = np.zeros(len(ev_cols_full), dtype=float)
    ev_max_pred = np.zeros(len(ev_cols_full), dtype=float)
    ev_today_min_pred = np.zeros(len(ev_cols_full), dtype=float)
    # 【装甲5：方法路由校验】拦截所有非法方法名称
    if forecast_method not in ['Perfect', 'Persistence']:
        raise ValueError(f"未知的预测方法 '{forecast_method}'，仅支持 'Perfect' 或 'Persistence'")
    if forecast_method == 'Perfect':
        # A rolling 96-step horizon contains the remainder of day 0 followed
        # by the corresponding prefix of the next calendar day.
        today_part = real_ub.iloc[i_t:min(real_ub.shape[0], i_t + H), :].to_numpy(dtype=float)
        next_steps = max(0, H - today_part.shape[0])
        next_part = next_ub.iloc[:next_steps, :].to_numpy(dtype=float)
        if next_part.shape[0] < next_steps:
            next_part = np.vstack([next_part, np.zeros((next_steps - next_part.shape[0], len(ev_cols_full)))])
        ev_ub_pred[:, :] = np.vstack([today_part, next_part])[:H, :]
        next_deliverable = next_part.sum(axis=0)
        if next_ub.shape[0] > 0 and next_steps > 0:
            next_values = next_ub.to_numpy(dtype=float)
            row_index = np.arange(next_values.shape[0], dtype=int)[:, None]
            last_available = np.where(next_values > 1e-12, row_index, -1).max(axis=0)
            has_next_session = last_available >= 0
            departs_within_horizon = has_next_session & (last_available < next_steps)
        else:
            departs_within_horizon = np.zeros(len(ev_cols_full), dtype=bool)
        next_max = np.minimum(next_session, next_deliverable)
        next_required = np.where(departs_within_horizon, next_max, 0.0)
        if service_min < 1.0:
            next_required = np.where(departs_within_horizon, np.minimum(service_min * next_session, next_deliverable), 0.0)
        ev_today_min_pred[:] = real_min
        ev_min_pred[:] = real_min + next_required
        ev_max_pred[:] = real_max + next_max
    elif forecast_method == 'Persistence':
        for j in range(len(ev_cols_full)):
            # Include the current row because arrivals for i_t have already been
            # incorporated before this forecast is built.
            has_arrived_today = (real_ub.iloc[0:min(real_ub.shape[0], i_t + 1), j].sum() > 0)
            if has_arrived_today:
                ub_slice = real_ub.iloc[i_t:i_t + H, j].values
                ev_min_pred[j] = real_min[j]
                ev_max_pred[j] = real_max[j]
            else:
                ub_slice = fc_ub.iloc[i_t:i_t + H, j].values
                ev_min_pred[j] = fc_min[j]
                ev_max_pred[j] = fc_max[j]
            length = len(ub_slice)
            if length < H:
                # Rolling tail after 24:00 uses day0's already executed EV load
                # [1:k] instead of zeros for persistence forecasting.
                pad_len = H - length
                pad_values = tail_ub_realized.iloc[:pad_len, j].to_numpy(dtype=float)
                if len(pad_values) < pad_len:
                    pad_values = np.pad(pad_values, (0, pad_len - len(pad_values)), mode='constant', constant_values=0.0)
                ub_slice = np.concatenate([ub_slice, pad_values[:pad_len]])
            ev_ub_pred[:, j] = ub_slice
        ev_today_min_pred[:] = ev_min_pred
    return ev_ub_pred, ev_min_pred, ev_max_pred, ev_today_min_pred
for month in (RUN_MONTHS_CONFIG if RUN_MAIN_LOOP_DIRECT else []):
    print("Test month:", month)
    num_days = calendar.monthrange(year,month)[1] 
    run_days = list(RUN_DAYS_CONFIG)  # manually configured at block start
    run_days = [d for d in run_days if 1 <= d <= num_days]
    if len(run_days) == 0:
        raise ValueError('run_days is empty after month/day bounds check.')
    # Each study month is evaluated as an independent billing/control period.
    # Reset BESS carry-over state so running [6, 7] gives the same July initial
    # condition as running [7] alone.
    SOC_BESS_daily_end = np.ones(len(Cases)) * 0.5
    Data_AS_DA_month = Data_AS_DA_all[(Data_AS_DA_all["datetime"].dt.year == year) & (Data_AS_DA_all["datetime"].dt.month == month)]
    Data_AS_RT_month = Data_AS_RT_all[(Data_AS_RT_all["datetime"].dt.year == year) & (Data_AS_RT_all["datetime"].dt.month == month)]
    # NOTE: Initialize M_Th to 0 at the start of each month.
    # SDGE demand charge resets each month; start from 0 and let rolling max accumulate.
    _init_ncd = 0.0
    _init_pd = 0.0
    M_Th_NCD = [_init_ncd for _ in range(len(Cases)+1)]
    M_Th_PD = [_init_pd for _ in range(len(Cases)+1)]
    print(f'Month {month}: M_Th_NCD init={_init_ncd:.1f} kW, M_Th_PD init={_init_pd:.1f} kW (initialized to 0)')
    M_Th_NCD_list = [[] for _ in range(len(Cases)+1)]
    M_Th_PD_list = [[] for _ in range(len(Cases)+1)]
    M_Th_NCD_list_96 = [[] for _ in range(len(Cases)+1)]
    M_Th_PD_list_96 = [[] for _ in range(len(Cases)+1)]
    M_V0G = pd.DataFrame()
    M_V1G = pd.DataFrame()
    M_Opt_Offline = pd.DataFrame()
    M_Opt_Imp = [pd.DataFrame() for _ in range(len(Cases))] #pd.DataFrame()
    M_t = pd.DataFrame()
    M_ED_V0G = []
    M_ED_V1G = []
    M_ED_Opt_DA = []
    M_ED_Opt_t0 = []
    M_ED_Opt_base = []
    M_ED_Opt_case1 = []
    Status_list_DA = []
    for Day in tqdm(run_days, desc="Processing days"):
        revenue_WM_daily = np.zeros(2)
        revenue_DR_daily = np.zeros(2)
        is_last_run_day = (Day == run_days[-1])
        TheDate_Day0 = datetime(year, month, Day)   
        TheDate_Day0_Start = TheDate_Day0 + timedelta(seconds=0)
        TheDate_Day0_End = TheDate_Day0 + timedelta(hours=24)
        if len(Y_WeekdaysWOH[Y_WeekdaysWOH==TheDate_Day0])>0: 
            Order = Y_WeekdaysWOH[Y_WeekdaysWOH==TheDate_Day0].index  
            TheDate_Dayb1 = Y_WeekdaysWOH[Order-1].iloc[0]
            TheDate_Dayb2 = Y_WeekdaysWOH[Order-2].iloc[0]
            TheDate_Dayb3 = Y_WeekdaysWOH[Order-3].iloc[0]
        else:
            Order = Y_WeekendsWH[Y_WeekendsWH==TheDate_Day0].index  
            TheDate_Dayb1 = Y_WeekendsWH[Order-1].iloc[0]
            TheDate_Dayb2 = Y_WeekendsWH[Order-2].iloc[0]
            TheDate_Dayb3 = Y_WeekendsWH[Order-3].iloc[0]
        TheDate_Dayb1_Start = TheDate_Dayb1 + timedelta(seconds=0)
        TheDate_Dayb1_End   = TheDate_Dayb1 + timedelta(hours=24)
        TheDate_Dayb2_Start = TheDate_Dayb2 + timedelta(seconds=0)
        TheDate_Dayb2_End   = TheDate_Dayb2 + timedelta(hours=24)
        TheDate_Day1 = TheDate_Day0 + timedelta(days=1)
        TheDate_Day1_Start = TheDate_Day1
        TheDate_Day1_End = TheDate_Day1 + timedelta(hours=24)
        # Keep indices 0:2 unchanged for baseline logic; index 3 is the next
        # calendar day used only by the rolling 24-hour perfect EV forecast.
        TheDate        = [TheDate_Day0, TheDate_Dayb1, TheDate_Dayb2, TheDate_Day1]
        TheDates_Start = [TheDate_Day0_Start, TheDate_Dayb1_Start, TheDate_Dayb2_Start, TheDate_Day1_Start]
        TheDates_End   = [TheDate_Day0_End, TheDate_Dayb1_End, TheDate_Dayb2_End, TheDate_Day1_End]
        if DAM == 0:
            Bid_Pr_DA = np.zeros(96)
            Bid_Pr_RT = np.zeros(96)
            Baseline_Opt_avg = np.empty((3, 3, 24))
        # Read the LMP data
        dir_Input_DA = os.path.join("2025Data/LMP/2025/DA/")
        dir_Input_RT = os.path.join("2025Data/LMP/2025/FM/")
        filename_Input = glob.glob(dir_Input_DA + TheDate_Day0_Start.strftime('%Y%m%d') + '*.csv')
        Data_LMP_DA = pd.read_csv(''.join(filename_Input))
        filename_Input = glob.glob(dir_Input_RT + TheDate_Day0_Start.strftime('%Y%m%d') + '*.csv')
        Data_LMP_RT = pd.read_csv(''.join(filename_Input))
        Data_LMP_DA = Data_LMP_DA[Data_LMP_DA['LMP_TYPE']=='LMP']
        Data_LMP_RT = Data_LMP_RT[Data_LMP_RT['LMP_TYPE']=='LMP']
        # sort the 'INTERVALSTARTTIME_GMT'
        LMP_IntervalStartGMT_DA = Data_LMP_DA['INTERVALSTARTTIME_GMT']
        LMP_IntervalStartGMT_RT = Data_LMP_RT['INTERVALSTARTTIME_GMT']
        # sort LMP data based on the 'INTERVALSTARTTIME_GMT'    
        A = pd.to_datetime(LMP_IntervalStartGMT_DA)
        B = [k for k in range(len(A))]
        LMPOrder = [x for _, x in sorted(zip(A, B))]
        Data_LMP_DA_ = Data_LMP_DA.iloc[ LMPOrder, :]
        A = pd.to_datetime(LMP_IntervalStartGMT_RT)
        B = [k for k in range(len(A))]
        LMPOrder = [x for _, x in sorted(zip(A, B))]
        Data_LMP_RT_ = Data_LMP_RT.iloc[ LMPOrder, :]
        Bid_Pr_DA = np.array(Data_LMP_DA_['MW'].repeat(4)*0.001) #repeat 4 times of the 24 x 1 series, $/MWh ---> $/kWh   
        Bid_Pr_RT = (np.average(np.array(Data_LMP_RT_['VALUE']).reshape(-1, 3), axis=1)*0.001) #ave every 3 entries of the 288  x 1 series, $/MWh ---> $/kWh
        # Rolling RT extends past midnight, so load the next calendar day's known prices.
        filename_DA_next = glob.glob(dir_Input_DA + TheDate_Day1_Start.strftime('%Y%m%d') + '*.csv')
        filename_RT_next = glob.glob(dir_Input_RT + TheDate_Day1_Start.strftime('%Y%m%d') + '*.csv')
        if len(filename_DA_next) != 1 or len(filename_RT_next) != 1:
            raise FileNotFoundError(f'Rolling horizon requires unique next-day LMP files for {TheDate_Day1_Start:%Y-%m-%d}.')
        Data_LMP_DA_next = pd.read_csv(filename_DA_next[0])
        Data_LMP_RT_next = pd.read_csv(filename_RT_next[0])
        Data_LMP_DA_next = Data_LMP_DA_next[Data_LMP_DA_next['LMP_TYPE']=='LMP'].sort_values('INTERVALSTARTTIME_GMT')
        Data_LMP_RT_next = Data_LMP_RT_next[Data_LMP_RT_next['LMP_TYPE']=='LMP'].sort_values('INTERVALSTARTTIME_GMT')
        if len(Data_LMP_DA_next) != 24 or len(Data_LMP_RT_next) != 288:
            raise ValueError(f'Incomplete next-day LMP data for {TheDate_Day1_Start:%Y-%m-%d}: DA={len(Data_LMP_DA_next)}, RT={len(Data_LMP_RT_next)}.')
        Bid_Pr_DA_next = np.array(Data_LMP_DA_next['MW'].repeat(4)*0.001)
        Bid_Pr_RT_next = np.average(np.array(Data_LMP_RT_next['VALUE']).reshape(-1, 3), axis=1)*0.001
        # Read the Ancillary Service prices data
        Data_AS_DA = Data_AS_DA_month[Data_AS_DA_month["datetime"].dt.day == Day].reset_index(drop=True)
        Data_AS_RT = Data_AS_RT_month[Data_AS_RT_month["datetime"].dt.day == Day].reset_index(drop=True)
        _next_day = pd.Timestamp(TheDate_Day1)
        Data_AS_DA_next = Data_AS_DA_all[(Data_AS_DA_all['datetime'].dt.year == _next_day.year) & (Data_AS_DA_all['datetime'].dt.month == _next_day.month) & (Data_AS_DA_all['datetime'].dt.day == _next_day.day)].reset_index(drop=True)
        Data_AS_RT_next = Data_AS_RT_all[(Data_AS_RT_all['datetime'].dt.year == _next_day.year) & (Data_AS_RT_all['datetime'].dt.month == _next_day.month) & (Data_AS_RT_all['datetime'].dt.day == _next_day.day)].reset_index(drop=True)
        if len(Data_AS_DA_next) != 24 or len(Data_AS_RT_next) != 96:
            raise ValueError(f'Incomplete next-day AS data for {_next_day:%Y-%m-%d}: DA={len(Data_AS_DA_next)}, RT={len(Data_AS_RT_next)}.')
        # CAISO ASMP is reported in $/MW, but the settlement interval differs by market.
        # DA ASMP applies to one hourly award: repeat the hourly value and integrate with dt_h.
        # RT ASMP applies to one 15-min binding interval: divide by dt_h so every model
        # price is expressed in $/kWh and Revenue = dt_h * price * quantity remains exact.
        _as_da_scale = 0.001
        _as_rt_scale = 0.001 / dt_h
        AS_Pr_RU_DA = np.repeat(Data_AS_DA["RegUp"].values, 4) * _as_da_scale
        AS_Pr_RD_DA = np.repeat(Data_AS_DA["RegDown"].values, 4) * _as_da_scale
        AS_Pr_SP_DA = np.repeat(Data_AS_DA["Spin"].values, 4) * _as_da_scale
        AS_Pr_NSP_DA = np.repeat(Data_AS_DA["NonSpin"].values, 4) * _as_da_scale
        AS_Pr_RU_RT = Data_AS_RT["RegUp"].values * _as_rt_scale
        AS_Pr_RD_RT = Data_AS_RT["RegDown"].values * _as_rt_scale
        AS_Pr_SP_RT = Data_AS_RT["Spin"].values * _as_rt_scale
        AS_Pr_NSP_RT = Data_AS_RT["NonSpin"].values * _as_rt_scale
        # Two-day exogenous price vectors support every 96-step rolling horizon.
        Bid_Pr_DA_rolling = np.concatenate([Bid_Pr_DA, Bid_Pr_DA_next])
        Bid_Pr_RT_rolling = np.concatenate([Bid_Pr_RT, Bid_Pr_RT_next])
        AS_Pr_RU_DA_rolling = np.concatenate([AS_Pr_RU_DA, np.repeat(Data_AS_DA_next['RegUp'].values, 4) * _as_da_scale])
        AS_Pr_RD_DA_rolling = np.concatenate([AS_Pr_RD_DA, np.repeat(Data_AS_DA_next['RegDown'].values, 4) * _as_da_scale])
        AS_Pr_SP_DA_rolling = np.concatenate([AS_Pr_SP_DA, np.repeat(Data_AS_DA_next['Spin'].values, 4) * _as_da_scale])
        AS_Pr_NSP_DA_rolling = np.concatenate([AS_Pr_NSP_DA, np.repeat(Data_AS_DA_next['NonSpin'].values, 4) * _as_da_scale])
        AS_Pr_RU_RT_rolling = np.concatenate([AS_Pr_RU_RT, Data_AS_RT_next['RegUp'].values * _as_rt_scale])
        AS_Pr_RD_RT_rolling = np.concatenate([AS_Pr_RD_RT, Data_AS_RT_next['RegDown'].values * _as_rt_scale])
        AS_Pr_SP_RT_rolling = np.concatenate([AS_Pr_SP_RT, Data_AS_RT_next['Spin'].values * _as_rt_scale])
        AS_Pr_NSP_RT_rolling = np.concatenate([AS_Pr_NSP_RT, Data_AS_RT_next['NonSpin'].values * _as_rt_scale])
        # Calendar-aware SDG&E retail tariff vectors for the DA day and every shifted RT horizon.
        _tariff_day_index = pd.date_range(TheDate_Day0, periods=96, freq=interval)
        _tariff_rolling_index = pd.date_range(TheDate_Day0, periods=192, freq=interval)
        c_e_TOU_AL, tariff_period_DA, tariff_season_DA = get_sdge_dgr_tariff_profile(_tariff_day_index)
        c_e_TOU_AL_rolling, _, _ = get_sdge_dgr_tariff_profile(_tariff_rolling_index)
        c_PD = get_sdge_pd_rate(TheDate_Day0)
        # concat last loop's implementation results with baseline
        dir_Output = os.path.join(VERSION_DIR, 'Dispatch', str(year) +'_' +Fc_SessionkWh+'_'+Fc_NumbEV+'_'+Fc_AtArrival+'_'+WM_Mode  +'_implementation.csv')
        _trim_existing_csv_before_run_start(dir_Output, TheDate_Day0)
        path = Path(dir_Output)
        if path.exists():
            Dispatch_implemented_lastloop = pd.read_csv(dir_Output, low_memory=False, header = 0)
            Dispatch_implemented_lastloop['Interval start'] = pd.to_datetime(Dispatch_implemented_lastloop['Interval start'])
            Dispatch_2025 = pd.concat([Dispatch_2025_baseline, Dispatch_implemented_lastloop], axis=0)
            Dispatch_2025 = Dispatch_2025.drop_duplicates('Interval start',keep='last')
        else:
            if Day == run_days[0]:
                Dispatch_2025 = Dispatch_2025_baseline
            else:
                print("No previous implementation file, exit!\n")
                sys.exit()
        dispatch_day_cache = {}
        dispatch_day_hour_cache = {}
        def _get_dispatch_day(this_day):
            day_key = pd.Timestamp(this_day).normalize()
            if day_key not in dispatch_day_cache:
                start = day_key
                end = day_key + timedelta(days=1)
                dispatch_day_cache[day_key] = Dispatch_2025[(Dispatch_2025['Interval start']>=start)&(Dispatch_2025['Interval start']<end)].reset_index()
            return dispatch_day_cache[day_key]
        def _get_dispatch_day_hour(this_day, i_hour):
            key = (pd.Timestamp(this_day).normalize(), int(i_hour))
            if key not in dispatch_day_hour_cache:
                day_df = _get_dispatch_day(this_day)
                hour_df = day_df[day_df['Interval start'].dt.hour == i_hour]
                dispatch_day_hour_cache[key] = hour_df.drop_duplicates('Interval start',keep='last')
            return dispatch_day_hour_cache[key]
        # Read baseline data
        days = 3
        Baseline_cases = [f'{case} [kWh]' for case in Cases] + ['V0G [kWh]']  # active RT cases plus DA/V0G reference
        Baseline_Opt_TheDates = [[[] for _ in range(len(Baseline_cases))] for _ in range(days)]  # Base case (100% service level), Case1 (eta%), and V0G for Day0. Day-1. and Day-2
        Baseline_days = [[[] for _ in range(len(Baseline_cases))] for _ in range(days)] # Base case, Case1, and V0G for Day0. Day-1. and Day-2
        Baseline_Opt_all = [[pd.Series() for _ in range(len(Baseline_cases))] for _ in range(days)] 
        Baseline_Opt_avg = [[[] for _ in range(len(Baseline_cases))] for _ in range(days)] 
        for i_d in range(days): #Day0. Day-1. and Day-2
            # if TheDate is a weekend day, or a holiday---> previous 4 weekend days                
            if TheDate[i_d] in list(Y_WeekendsWH) :
                NumbBaselineDays = 4
                CatagoryBaselineDays = Y_WeekendsWH
            else: # TheDate[i_d] in list(Y_WeekdaysWOH)
                NumbBaselineDays = 10
                CatagoryBaselineDays = Y_WeekdaysWOH
            for i_case in range(len(Baseline_cases)): # Base case (100% service level), Case1 (eta%), and V0G
                if i_case < len(Cases): # RT cases with market participation; event days can differ by hour
                    Baseline_days[i_d][i_case] = [[] for _ in range(24)]
                    Baseline_Opt_all[i_d][i_case] = [[] for _ in range(24)]
                    for i_hour in range(24):
                        for j in range(45):
                            ThisDay = TheDate[i_d] - timedelta(days=j+1)
                            if ThisDay in list(CatagoryBaselineDays):
                                Dispatch_2025_ThisDate_ThisHour = _get_dispatch_day_hour(ThisDay, i_hour)
                                # check if this hour this day is an event hour, if NOT an event hour, count the value
                                # TODO: 2 cases baseline always same with DA results, but should be different using implementation as baseline after 45 days running
                                if i_case == 0:
                                    Event = 'event hour_Base'
                                else:
                                    Event = 'event hour_Case1'
                                if Dispatch_2025_ThisDate_ThisHour.empty or (Event not in Dispatch_2025_ThisDate_ThisHour.columns):
                                    continue
                                if Dispatch_2025_ThisDate_ThisHour[Event].iloc[0] == 0:
                                    Baseline_days[i_d][i_case][i_hour] = Baseline_days[i_d][i_case][i_hour] + [ThisDay]
                                    avg = sum(Dispatch_2025_ThisDate_ThisHour[Baseline_cases[i_case]])/4
                                    Baseline_Opt_all[i_d][i_case][i_hour] = Baseline_Opt_all[i_d][i_case][i_hour] + [avg]
                            if len(Baseline_Opt_all[i_d][i_case][i_hour]) == NumbBaselineDays:
                                Baseline_Opt_avg[i_d][i_case] = Baseline_Opt_avg[i_d][i_case] + [float(np.mean(Baseline_Opt_all[i_d][i_case][i_hour])) if len(Baseline_Opt_all[i_d][i_case][i_hour]) > 0 else 0.0]
                                break
                            # if after going thru 45 days still cannot find the 'non event hour day', 
                            # just average howmany hour days 'Baseline_Opt_all[i_d][i_case][i_hour]' has
                            if j==44:
                                Baseline_Opt_avg[i_d][i_case] = Baseline_Opt_avg[i_d][i_case] + [float(np.mean(Baseline_Opt_all[i_d][i_case][i_hour])) if len(Baseline_Opt_all[i_d][i_case][i_hour]) > 0 else 0.0]
                        # A continue on the final search day can bypass the legacy
                        # j == 44 fallback above. Guarantee exactly one baseline
                        # value for every hour before expanding to 15-min steps.
                        if len(Baseline_Opt_avg[i_d][i_case]) <= i_hour:
                            values = Baseline_Opt_all[i_d][i_case][i_hour]
                            Baseline_Opt_avg[i_d][i_case].append(
                                float(np.mean(values)) if len(values) > 0 else 0.0
                            )
                    if len(Baseline_Opt_avg[i_d][i_case]) != 24:
                        raise ValueError(
                            f'EV baseline must contain 24 hourly values for {TheDate[i_d]:%Y-%m-%d}; '
                            f'got {len(Baseline_Opt_avg[i_d][i_case])}'
                        )
                else: # for i_case==2 no market participation
                    for j in range(45):
                        ThisDay = TheDate[i_d] - timedelta(days=j+1)
                        if ThisDay in list(CatagoryBaselineDays):
                            Dispatch_2025_ThisDate = _get_dispatch_day(ThisDay)
                            Baseline_days[i_d][i_case] = Baseline_days[i_d][i_case] + [ThisDay]
                            Baseline_Opt_all[i_d][i_case] = pd.concat([Baseline_Opt_all[i_d][i_case],Dispatch_2025_ThisDate[Baseline_cases[i_case]]],axis = 1,ignore_index = True)
                        if len(Baseline_days[i_d][i_case]) == NumbBaselineDays:
                            break
                    _baseline_samples = len(Baseline_days[i_d][i_case])
                    _baseline_matrix = Baseline_Opt_all[i_d][i_case]
                    if _baseline_samples > 0 and not _baseline_matrix.empty:
                        _baseline_matrix = _baseline_matrix.drop(0, axis=1, errors='ignore')
                        if _baseline_matrix.shape[1] > 0:
                            _interval_avg = np.array(_baseline_matrix.sum(axis=1)) / _baseline_matrix.shape[1]
                            Baseline_Opt_avg[i_d][i_case] = np.nan_to_num(np.average(_interval_avg.reshape(-1, 4), axis=1), nan=0.0, posinf=0.0, neginf=0.0) # ave every 4 entries of the (96, ) array
                        else:
                            Baseline_Opt_avg[i_d][i_case] = np.zeros(24)
                    else:
                        Baseline_Opt_avg[i_d][i_case] = np.zeros(24)
        # Choose data for Day0, Dayb1, Dayb2, and the next calendar day.
        ThisDates = []
        Data_TheDates = []       
        TimeSeries_TheDates = []
        Time_table_TheDates = []
        EnergyDemand_Table_V0G_TheDates = []
        EnergyDemand_Table_V1G_TheDates = [] 
        EnergyDemand_Step_V0G_TheDates = []
        EnergyDemand_Step_V1G_TheDates = []
        ArrivalTime_table_TheDates = []
        ArrivalTime_table_1_TheDates = []
        ArrivalTime_table_2_TheDates = []
        Car_table_TheDates = []
        Car_table_1_TheDates = []
        Car_table_2_TheDates = []
        Eta_min_table_TheDates = []
        Eta_min_table_1_TheDates = []
        Eta_min_table_2_TheDates = []
        IntervalkWh_max_TheDates = []
        IntervalkWh_max_1_TheDates = []
        IntervalkWh_max_2_TheDates = []
        SessionkWh_table_TheDates = []
        SessionkWh_table_1_TheDates = []
        SessionkWh_table_2_TheDates = []        
        Type_table_TheDates = []
        Type_table_1_TheDates = []
        Type_table_2_TheDates = []
        D_Th_NCD_96 = [[],[]]
        D_Th_PD_96  = [[],[]]
        for i_d in range(4):
            # Create Time Series for the Day                  
            start_ind = TheDates_Start[i_d]
            end_ind = TheDates_End[i_d]
            TimeSeries_ThisDate = []
            while start_ind < end_ind:
                TimeSeries_ThisDate.append(start_ind)
                start_ind += interval
            # Filter the data based on the Date you choose
            Data_ThisDate = Data[(TheDates_Start[i_d] <= Data['Interval start']) & (Data['Interval start'] < TheDates_End[i_d])]
            # An empty site-day is a physical zero-EV day, not missing data.
            # Substituting another calendar day would fabricate arrivals and leak
            # energy into the executed trajectory, especially for sparse sites.
            if len(Data_ThisDate) == 0:
                print("No EV sessions for {0}; retaining the exact calendar day with zero EV load".format(TheDates_Start[i_d]))
            Data_TheDates = Data_TheDates + [Data_ThisDate]
            TimeSeries_TheDates = TimeSeries_TheDates + [TimeSeries_ThisDate]    
            Time_table_TheDates = Time_table_TheDates + [pd.DataFrame(TimeSeries_ThisDate, columns=["Interval start"]) ]  
            #################################################################################
            # Availability table (Time series x Vehicles)
            ################################################################################# 
            AllCar_table_max_charger = pd.DataFrame(columns=["Interval start"])
            AllCar_table_float_V0G = pd.DataFrame(columns=["Interval start"])
            AllCar_table_float_V1G = pd.DataFrame(columns=["Interval start"])
            Type_list = []
            SessionkWh_list = []
            ArrivalTime_list = []
            Eta_min_list = []
            Cars = Data_ThisDate["Car_"].unique()
            Cars2 = []
            if Fc_AtArrival == 'MLatArrival' :   
                #store Day 0, real session data for ML forecast inputs
                if i_d == 0:               
                    List = ['Car', 'User','Session start','Arrival Hour', 'Battery (kWh)','Weekday', 'Max Charging Power']
                    Sess_Car = []
                    Sess_User = []
                    Sess_SessionStart = []
                    Sess_ArrivalHr = []
                    Sess_BESS = []
                    Sess_Weekday = []
                    Sess_MaxPw = []
            for i in range(len(Cars)):
                CarName = Cars[i]      
                #Check for Users with no sessions creating empty dataframes 
                data_Merge_ThisDate_ThisCar_Test = Data_ThisDate[Data_ThisDate["Car_"] == CarName]
                rows, columns =  data_Merge_ThisDate_ThisCar_Test.shape
                if rows > 0:
                    data_Merge_ThisDate_ThisCar = Data_ThisDate[Data_ThisDate["Car_"] == CarName]
                    data_Merge_ThisDate_ThisCar = data_Merge_ThisDate_ThisCar.sort_values('Interval start')
                    # Round up/down the arrival/departure time for V1G, V0G, and Opt
                    # NOTE: if there are > 1 sessions for this car?
                    if data_Merge_ThisDate_ThisCar["Session start"].nunique() == 1:
                        filtered = data_Merge_ThisDate_ThisCar[
                            (data_Merge_ThisDate_ThisCar["Interval start"] >= data_Merge_ThisDate_ThisCar["Session start"]) &
                            (data_Merge_ThisDate_ThisCar["Interval end"] <= data_Merge_ThisDate_ThisCar["Session end"])
                        ]
                        data_Merge_ThisDate_ThisCar = filtered
                        AvailableSlots = len(data_Merge_ThisDate_ThisCar)
                    elif data_Merge_ThisDate_ThisCar["Session start"].nunique() >= 2:
                        for sess in range(data_Merge_ThisDate_ThisCar["Session start"].nunique()):
                            filtered = data_Merge_ThisDate_ThisCar[
                                (data_Merge_ThisDate_ThisCar["Interval start"] >= data_Merge_ThisDate_ThisCar["Session start"].unique()[sess]) &
                                (data_Merge_ThisDate_ThisCar["Interval end"] <= data_Merge_ThisDate_ThisCar["Session end"].unique()[sess])
                            ]
                            if sess == 0:
                                data_Merge_ThisDate_ThisCar_tmp = filtered
                            else:
                                data_Merge_ThisDate_ThisCar_tmp = pd.concat([data_Merge_ThisDate_ThisCar_tmp, filtered], axis=0)
                        data_Merge_ThisDate_ThisCar = data_Merge_ThisDate_ThisCar_tmp.sort_values('Interval start')
                        AvailableSlots = len(data_Merge_ThisDate_ThisCar)
                    if AvailableSlots == 0:
                        continue # if no available slots after filtering, skip this car
                    Cars2 = Cars2 + [CarName]
                    # NOTE: 2025 data doesnt have Tesla Type, but still have Tesla cars with higher charging power
                    if (data_Merge_ThisDate_ThisCar['Interval kWh'] > 1.664).any() or (data_Merge_ThisDate_ThisCar['Interval max demand kW'] > 1.664*4).any():
                        Type = 'Tesla'
                    elif (data_Merge_ThisDate_ThisCar['Interval kWh'] <= 1.664).all() and (data_Merge_ThisDate_ThisCar['Interval max demand kW'] <= 1.664*4).all():
                        Type = 'NonTesla'
                    else:
                        print("Error: not sure if Tesla or NonTesla, exit!")
                        sys.exit()
                    Type_list = Type_list + [Type]        
                    if Type == 'Tesla':
                        IntervalkWh_CH_max = 4.16
                    else:
                        IntervalkWh_CH_max = 1.664
                    # adjust intervalkWh_V1G if SessionkWh_V1G > numb of available intervals * IntervalkWh_CH_max 
                    SessionkWh = sum(data_Merge_ThisDate_ThisCar['Interval kWh'])
                    if SessionkWh > AvailableSlots*IntervalkWh_CH_max:
                        ratio = AvailableSlots*IntervalkWh_CH_max/SessionkWh
                        # modify interval data (decrease V1G)
                        data_Merge_ThisDate_ThisCar['Interval kWh'] = data_Merge_ThisDate_ThisCar['Interval kWh']*ratio
                        #also need to modify session data (decrease SessionkWh)
                        SessionkWh = SessionkWh*ratio
                    SessionkWh_list = SessionkWh_list + [SessionkWh]
                    # there might be two session start!!
                    ArrivalTime = data_Merge_ThisDate_ThisCar["Session start"].sort_values(ascending=True).iloc[0]
                    ArrivalTime_list = ArrivalTime_list + [ArrivalTime]
                    # Eta_min
                    Eta_min = data_Merge_ThisDate_ThisCar['Eta'].iloc[0]
                    Eta_min_list = Eta_min_list + [Eta_min]
                    # Interval time series during plug-in time
                    IntervalStart_ThisCar_round = pd.to_datetime(data_Merge_ThisDate_ThisCar["Interval start"])
                    # CHarger capacity: IntervalkWh_ThisCar_CH
                    IntervalkWh_ThisCar_CH = pd.Series(range(AvailableSlots))*0 + IntervalkWh_CH_max            
                    ThisCar_table_float_max_charger = pd.DataFrame(list(zip(IntervalStart_ThisCar_round, IntervalkWh_ThisCar_CH)), columns=["Interval start", CarName])            
                    ThisCar_table_float_max_charger_ = pd.merge(Time_table_TheDates[i_d], ThisCar_table_float_max_charger, how="left", on=["Interval start"])
                    AllCar_table_max_charger = AllCar_table_max_charger.merge(ThisCar_table_float_max_charger_,  on="Interval start", how='outer')
                    if Fc_AtArrival == 'MLatArrival' : 
                        #store Day 0, real session data for ML forecast inputs
                        if i_d == 0:
                            Sess_Car = Sess_Car + [CarName]
                            Sess_User = Sess_User + [data_Merge_ThisDate_ThisCar['User'].iloc[0]]
                            ArrivalTime = data_Merge_ThisDate_ThisCar['Session start'].iloc[0]
                            Sess_SessionStart = Sess_SessionStart + [ArrivalTime] 
                            Sess_ArrivalHr = Sess_ArrivalHr + [ArrivalTime.hour + ArrivalTime.minute/60]
                            # based on 'User', import BESS info from Byron's master sheet
                            BESS_kWh = UserBess['Battery (kWh)'][UserBess['Doe Id']==int(Sess_User[0])]
                            if len(BESS_kWh)==0:
                                BESS_kWh = 48 #which is the avg BESS kWh of the master sheet
                                UserNoBess = UserNoBess + [data_Merge_ThisDate_ThisCar['User'].iloc[0]]
                            Sess_BESS = Sess_BESS + [BESS_kWh]
                            Sess_Weekday = Sess_Weekday + [ArrivalTime.weekday()] #monday is 0
                            Sess_MaxPw = Sess_MaxPw + [IntervalkWh_CH_max*4]
                    # V0G
                    IntervalkWh_ThisCar_V0G = IntervalkWh_ThisCar_CH
                    IntervalkWh_left = SessionkWh + 0
                    for j in range(len(IntervalkWh_ThisCar_V0G)):
                        if IntervalkWh_left >= IntervalkWh_ThisCar_CH.iloc[j]:
                            IntervalkWh_left = IntervalkWh_left - IntervalkWh_ThisCar_CH.iloc[j]
                        else:
                            IntervalkWh_ThisCar_V0G.iloc[j] = IntervalkWh_left
                            IntervalkWh_ThisCar_V0G.iloc[j+1:] = 0
                            break
                    ThisCar_table_V0G = pd.DataFrame(list(zip(IntervalStart_ThisCar_round, IntervalkWh_ThisCar_V0G)), columns=["Interval start", CarName])
                    ThisCar_table_V0G_ = pd.merge(Time_table_TheDates[i_d], ThisCar_table_V0G, how="left", on=["Interval start"])
                    AllCar_table_float_V0G = AllCar_table_float_V0G.merge(ThisCar_table_V0G_,  on="Interval start", how='outer')
                    # V1G_real which is the real charging data
                    IntervalkWh_ThisCar_float_V1G = data_Merge_ThisDate_ThisCar["Interval kWh"]
                    ThisCar_table_float = pd.DataFrame(list(zip(IntervalStart_ThisCar_round, IntervalkWh_ThisCar_float_V1G)), columns=["Interval start", CarName])
                    ThisCar_table_float_ = pd.merge(Time_table_TheDates[i_d], ThisCar_table_float, how="left", on=["Interval start"])
                    AllCar_table_float_V1G = AllCar_table_float_V1G.merge(ThisCar_table_float_,  on="Interval start", how='outer')
            # Availability matrix with Max CHarger capacity
            AllCar_table_max_charger = AllCar_table_max_charger.fillna(0)
            IntervalkWh_max = AllCar_table_max_charger.drop("Interval start", axis=1)            
            # V0G
            AllCar_table_float_V0G = AllCar_table_float_V0G.fillna(0)
            EnergyDemand_Table_V0G = AllCar_table_float_V0G.drop("Interval start", axis=1)
            EnergyDemand_Step_V0G = EnergyDemand_Table_V0G.sum(axis=1)
            if len(EnergyDemand_Step_V0G) == 0:
                EnergyDemand_Step_V0G = pd.Series(np.zeros(len(TimeSeries_ThisDate)))
            # V1G_real
            AllCar_table_float_V1G = AllCar_table_float_V1G.fillna(0)
            EnergyDemand_Table_V1G =  AllCar_table_float_V1G.drop("Interval start", axis=1)  # tables without time series
            EnergyDemand_Step_V1G = EnergyDemand_Table_V1G.sum(axis=1)
            if len(EnergyDemand_Step_V1G) == 0:
                EnergyDemand_Step_V1G = pd.Series(np.zeros(len(TimeSeries_ThisDate)))
            #################################################################################
            # Sort all the list based on the car arrival time
            #################################################################################       
            Cars2 = np.array(Cars2)
            # Create dataframes for SessionkWh, Eta_min, Arrival time, Types, and Cars
            ArrivalTime_table = pd.DataFrame(ArrivalTime_list).T
            ArrivalTime_table.columns = Cars2
            Car_table = pd.DataFrame(columns=Cars2)
            Eta_min_table = pd.DataFrame(Eta_min_list).T
            Eta_min_table.columns = Cars2
            SessionkWh_table = pd.DataFrame(SessionkWh_list).T
            SessionkWh_table.columns = Cars2
            Type_table = pd.DataFrame(Type_list).T
            Type_table.columns = Cars2
            # sort all tables based on the arrival time (first come first serve)        
            A = pd.to_datetime(ArrivalTime_list)
            B = [k for k in range(len(A))]
            CarOrder = [x for _, x in sorted(zip(A, B))]
            # Sort the order                
            ArrivalTime_table = ArrivalTime_table.iloc[:, CarOrder]
            Car_table         = Car_table.iloc[:, CarOrder]
            Eta_min_table     = Eta_min_table.iloc[:, CarOrder]
            SessionkWh_table  = SessionkWh_table.iloc[:, CarOrder]
            Type_table        = Type_table.iloc[:, CarOrder]
            IntervalkWh_max   = IntervalkWh_max.iloc[:, CarOrder]
            # Final feasibility reconciliation for optimizer input.
            # Rare PowerFlex records can have session kWh slightly above the 15-min
            # interval max-demand envelope after rounding/merging. The MILP cannot
            # satisfy both equality service and interval upper bounds unless we cap
            # only these data-quality mismatches before constructing forecasts.
            if SessionkWh_table.shape[1] > 0:
                _max_feasible_session_kwh = IntervalkWh_max.sum(axis=0).reindex(SessionkWh_table.columns).fillna(0.0).astype(float)
                _session_kwh_raw = SessionkWh_table.iloc[0].astype(float)
                _excess_kwh = _session_kwh_raw - _max_feasible_session_kwh
                _needs_cap = _excess_kwh > 1e-6
                _large_mismatch = _excess_kwh > 0.10
                if bool(_large_mismatch.any()):
                    print("Warning: EV session energy exceeds interval envelope by more than 0.10 kWh; capping for feasibility:")
                    print(pd.DataFrame({
                        'session_kWh': _session_kwh_raw[_large_mismatch],
                        'interval_envelope_kWh': _max_feasible_session_kwh[_large_mismatch],
                        'excess_kWh': _excess_kwh[_large_mismatch],
                    }).round(4))
                if bool(_needs_cap.any()):
                    print(f"Adjusted {int(_needs_cap.sum())} session energy value(s) to interval max-feasible energy on {TheDates_Start[i_d].date()}")
                    SessionkWh_table.iloc[0, :] = np.minimum(_session_kwh_raw, _max_feasible_session_kwh).values
            #################################################################################
            # Seperate two types of cars: Tesla & non-Tesla
            #################################################################################                 
            if Type_table.empty or Type_table.shape[1] == 0:
                Type_table_1 = Type_table.iloc[:, 0:0].copy()
                Type_table_2 = Type_table.iloc[:, 0:0].copy()
            else:
                Type_table_1 = Type_table.loc[:,Type_table.iloc[0, :] == 'Tesla']
                Type_table_2 = Type_table.loc[:,Type_table.iloc[0, :] != 'Tesla']            
            Logic_table_select_car_1 = Car_table.columns.intersection(list(Type_table_1.columns))
            Logic_table_select_car_2 = Car_table.columns.intersection(list(Type_table_2.columns))
            ArrivalTime_table_1 = ArrivalTime_table[Logic_table_select_car_1]
            ArrivalTime_table_2 = ArrivalTime_table[Logic_table_select_car_2]
            Car_table_1 = Car_table[Logic_table_select_car_1]
            Car_table_2 = Car_table[Logic_table_select_car_2]
            Eta_min_table_1 = Eta_min_table[Logic_table_select_car_1]
            Eta_min_table_2 = Eta_min_table[Logic_table_select_car_2]
            IntervalkWh_max_1 = IntervalkWh_max[Logic_table_select_car_1]
            IntervalkWh_max_2 = IntervalkWh_max[Logic_table_select_car_2]
            SessionkWh_table_1 = SessionkWh_table[Logic_table_select_car_1]
            SessionkWh_table_2 = SessionkWh_table[Logic_table_select_car_2]
            #Put all variables of Day0, Dayb1, and Dayb2 in lists
            ArrivalTime_table_TheDates = ArrivalTime_table_TheDates + [ArrivalTime_table]
            ArrivalTime_table_1_TheDates = ArrivalTime_table_1_TheDates + [ArrivalTime_table_1]            
            ArrivalTime_table_2_TheDates = ArrivalTime_table_2_TheDates + [ArrivalTime_table_2]
            ArrivalTime_table_TheDates_ = [ArrivalTime_table_TheDates,ArrivalTime_table_1_TheDates,ArrivalTime_table_2_TheDates]
            Car_table_TheDates = Car_table_TheDates + [Car_table]
            Car_table_1_TheDates = Car_table_1_TheDates + [Car_table_1]
            Car_table_2_TheDates = Car_table_2_TheDates + [Car_table_2]
            Car_table_TheDates_ = [Car_table_TheDates,Car_table_1_TheDates,Car_table_2_TheDates]
            Eta_min_table_TheDates = Eta_min_table_TheDates + [Eta_min_table]
            Eta_min_table_1_TheDates = Eta_min_table_1_TheDates + [Eta_min_table_1]
            Eta_min_table_2_TheDates = Eta_min_table_2_TheDates + [Eta_min_table_2]  
            Eta_min_table_TheDates_ = [Eta_min_table_TheDates,Eta_min_table_1_TheDates,Eta_min_table_2_TheDates]
            IntervalkWh_max_TheDates = IntervalkWh_max_TheDates + [IntervalkWh_max]
            IntervalkWh_max_1_TheDates = IntervalkWh_max_1_TheDates + [IntervalkWh_max_1]
            IntervalkWh_max_2_TheDates = IntervalkWh_max_2_TheDates + [IntervalkWh_max_2]
            IntervalkWh_max_TheDates_ = [IntervalkWh_max_TheDates,IntervalkWh_max_1_TheDates,IntervalkWh_max_2_TheDates]
            SessionkWh_table_TheDates = SessionkWh_table_TheDates + [SessionkWh_table]
            SessionkWh_table_1_TheDates = SessionkWh_table_1_TheDates + [SessionkWh_table_1]
            SessionkWh_table_2_TheDates = SessionkWh_table_2_TheDates + [SessionkWh_table_2]
            SessionkWh_table_TheDates_ = [SessionkWh_table_TheDates,SessionkWh_table_1_TheDates,SessionkWh_table_2_TheDates]
            Type_table_TheDates = Type_table_TheDates + [Type_table]
            Type_table_1_TheDates = Type_table_1_TheDates + [Type_table_1]
            Type_table_2_TheDates = Type_table_2_TheDates + [Type_table_2]
            EnergyDemand_Table_V0G_TheDates = EnergyDemand_Table_V0G_TheDates + [EnergyDemand_Table_V0G] 
            EnergyDemand_Step_V0G_TheDates = EnergyDemand_Step_V0G_TheDates + [EnergyDemand_Step_V0G]
            EnergyDemand_Table_V1G_TheDates = EnergyDemand_Table_V1G_TheDates + [EnergyDemand_Table_V1G] 
            EnergyDemand_Step_V1G_TheDates = EnergyDemand_Step_V1G_TheDates + [EnergyDemand_Step_V1G]    
        if Fc_AtArrival == 'MLatArrival' : 
            #List ['Car', 'User', 'Session start', 'Arrival Hour', 'Battery (kWh)', 'Weekday', 'Max Charging Power']
            All_Sess = pd.concat([pd.Series(Sess_Car),  pd.Series(Sess_User),    pd.Series(Sess_SessionStart), pd.Series(Sess_ArrivalHr),\
                                    pd.Series(Sess_BESS), pd.Series(Sess_Weekday), pd.Series(Sess_MaxPw)], axis=1)
            All_Sess.columns = List
        else:
            All_Sess = pd.DataFrame()
        # Choose the Real and forecasted variables from Day0, or Day 1 based on the forecast
        i_ = len(Cases)+1
        EmptyList = [[[] for _ in range(i_)] for _ in range(3)] # for All EVS, Tesla, non-Tesla, each has three lists 
        ArrivalTime_table_real =  [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy() 
        ArrivalTime_table_fc   = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        Car_table_real         = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        Car_table_fc           = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        Eta_min_table_real     = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        Eta_table_RT           = [[[] for _ in range(i_)] for _ in range(3)] 
        IntervalkWh_max_real   = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        IntervalkWh_max_fc     = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        SessionkWh_table_real  = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()
        SessionkWh_table_fc    = [[[] for _ in range(i_)] for _ in range(3)]  #EmptyList.copy()    
        Time_table_real        = [[] for _ in range(i_)]
        Time_table_fc          = [[] for _ in range(i_)]
        TimeSeries_real        = [[] for _ in range(i_)]
        TimeSeries_fc          = [[] for _ in range(i_)]
        Baseline_Opt_real = [[] for _ in range(i_)] #Base, Case1, and V0G
        Baseline_Opt_fc   = [[] for _ in range(i_)]
        # unlike other variables, Base and Case1 have diff baseline values
        for i in range(len(Cases)+1): #Base, Case1, V0G
            Baseline_Opt_real[i] = np.array(Baseline_Opt_avg[0][i])
            if Fc_SessionkWh == 'PerfectSessionkWh'  :
                Baseline_Opt_fc[i] = np.array(Baseline_Opt_avg[0][i]) 
            elif Fc_SessionkWh == 'PersistenceSessionkWh'  :
                Baseline_Opt_fc[i] = np.array(Baseline_Opt_avg[1][i]) 
        #Assign real values for RT (2 cases) amd DA
        for i in range(len(Cases)+1):
            if i < len(Cases):
                i_ = 0 # Day0 assigned to active RT cases
            else:
                i_ = 1 # Dayb1 assigned to DA/V0G reference
            for j in range(3):      #All EVs, Tela, and Non-Tesla
                ArrivalTime_table_real[j][i] = (ArrivalTime_table_TheDates_[j][i_]).copy()
                Car_table_real[j][i] = Car_table_TheDates_[j][i_].copy()
                Eta_min_table_real[j][i] = Eta_min_table_TheDates_[j][i_].copy() 
                if i == 0: #base case has Eta=100*
                    Eta_min_table_real[j][i][Eta_min_table_real[j][i]>0]=1
                Eta_table_RT[j][i] = Eta_min_table_real[j][i].copy()
                #Eta_table_RT[j][i][Eta_table_RT[j][i]>0] = 1
                IntervalkWh_max_real[j][i] = IntervalkWh_max_TheDates_[j][i_].copy()
                SessionkWh_table_real[j][i]   =  SessionkWh_table_TheDates_[j][i_].copy()    
            Time_table_real[i] = Time_table_TheDates[i_].copy() 
            TimeSeries_real[i] = TimeSeries_TheDates[i_].copy()            
            if Fc_SessionkWh == 'PerfectSessionkWh' and WM_Mode != 'retail_only':
                i_ = 0
            elif Fc_SessionkWh == 'PersistenceSessionkWh':
                if i < len(Cases):
                    i_ = 1 # Dayb1 assigned to active RT cases
                else:
                    i_ = 2 # Dayb2 assigned to DA/V0G reference
            for j in range(3): #All EVs, Tela, and Non-Tesla
                ArrivalTime_table_fc[j][i] = ArrivalTime_table_TheDates_[j][i_].copy()
                Car_table_fc[j][i] = Car_table_TheDates_[j][i_].copy()
                IntervalkWh_max_fc[j][i] = IntervalkWh_max_TheDates_[j][i_].copy()
                SessionkWh_table_fc[j][i]   =  SessionkWh_table_TheDates_[j][i_].copy()                
            Time_table_fc[i] = Time_table_TheDates[i_].copy()                
            TimeSeries_fc[i] = TimeSeries_TheDates[i_].copy() 
        #################################################################################
        # make the forecasted SessionkWh and IntervalkWh_max the same length as the real ones
        #################################################################################
        #For Real-time and Day-ahead optimization, and Real-time optimization has two cases: with Eta= 100% and Eta_min
        IntervalkWh_max_fc_fix = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        SessionkWh_table_fc_fix = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        #SessionkWh_left           =  [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        Car_table_fc_fix          =  [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        for i in range(len(Cases)+1):
            if (Fc_SessionkWh == 'PersistenceSessionkWh') & (Fc_NumbEV == 'PersistenceNumbEV'):
                for j in np.array(range(2))+1:
                    IntervalkWh_max_fc_fix[j][i] = IntervalkWh_max_fc[j][i].copy()
                    SessionkWh_table_fc_fix[j][i] = SessionkWh_table_fc[j][i].copy()
                    Car_table_fc_fix[j][i] = Car_table_fc[j][i].copy()
            elif Fc_NumbEV == 'PerfectNumbEV':
                # in this version, with perfect forecasted SessionkWh, Day-1 uses Day0 data, 
                # Careful! Don't trim the fc_fix variables with Day-1 real data
                # instead, just make '_fc_fix' = '_fc' given that '_fc' already equals to '_real'
                if Fc_SessionkWh == 'PerfectSessionkWh'  :
                    for j in np.array(range(2))+1:
                        IntervalkWh_max_fc_fix[j][i] = IntervalkWh_max_fc[j][i].copy()
                        SessionkWh_table_fc_fix[j][i] = SessionkWh_table_fc[j][i].copy()
                        Car_table_fc_fix[j][i] = Car_table_fc[j][i].copy()
                else:            
                    for j in np.array(range(2))+1: # Tela, and Non-Tesla
                        # trim the forecast tables
                        if len(Car_table_fc[j][i].columns) > len(Car_table_real[j][i].columns):  
                            SessionkWh_table_fc_fix[j][i] = SessionkWh_table_fc[j][i].iloc[:,0:len(Car_table_real[j][i].columns)]
                            IntervalkWh_max_fc_fix[j][i] = IntervalkWh_max_fc[j][i].iloc[:,0:len(Car_table_real[j][i].columns)]
                            Car_table_fc_fix[j][i] = Car_table_fc[j][i].iloc[:,0:len(Car_table_real[j][i].columns)]
                        # repeat the last n1 rows
                        # note that after repeating the last n rows, these n column names are repeated 
                            # so when calling columns, donnot call the names, but call the order
                        # note that if there was no cars, ie len(Car_table_1_fc)==0, can't repeat 
                        # note if n1_=0
                        elif (len(Car_table_fc[j][i].columns) < len(Car_table_real[j][i].columns)) & (len(Car_table_fc[j][i].columns) > 0):  
                            n1 = len(Car_table_real[j][i].columns)//len(Car_table_fc[j][i].columns)  # qoutient
                            n1_ = len(Car_table_real[j][i].columns)%len(Car_table_fc[j][i].columns)  # remainder
                            Aux1 = pd.concat([SessionkWh_table_fc[j][i]]*n1,  axis=1)
                            Aux2 = pd.concat([IntervalkWh_max_fc[j][i]]*n1,  axis=1)
                            if n1_ == 0:
                                SessionkWh_table_fc_fix[j][i] = Aux1
                                IntervalkWh_max_fc_fix[j][i] = Aux2
                            else:
                                SessionkWh_table_fc_fix[j][i] = pd.concat([Aux1, SessionkWh_table_fc[j][i].iloc[:, -n1_:]], axis=1) 
                                IntervalkWh_max_fc_fix[j][i] = pd.concat([Aux2, IntervalkWh_max_fc[j][i].iloc[:, -n1_:]], axis=1)
                            Car_table_fc_fix[j][i] = pd.DataFrame(columns=IntervalkWh_max_fc_fix[j][i].columns)
                        # have to use perfect fc if no data from yesterday!!!   
                        # ex: on Dec 27th, no Tesla arrived yesterday(Dec. 26th)
                        elif len(Car_table_fc[j][i]) == 0:
                            SessionkWh_table_fc_fix[j][i] = SessionkWh_table_real[j][i].copy()
                            IntervalkWh_max_fc_fix[j][i] = IntervalkWh_max_real[j][i].copy()
                            Car_table_fc_fix[j][i] = Car_table_real[j][i].copy()
                        else:
                            SessionkWh_table_fc_fix[j][i] = SessionkWh_table_fc[j][i].copy()
                            IntervalkWh_max_fc_fix[j][i] = IntervalkWh_max_fc[j][i].copy()
                            Car_table_fc_fix[j][i] = Car_table_fc[j][i].copy()
            # Combine Type1 (Tesla) and Type2 (Non-Tesla) for optimization    
            SessionkWh_table_fc_fix[0][i] = pd.concat([SessionkWh_table_fc_fix[1][i], SessionkWh_table_fc_fix[2][i]], axis=1)   
            IntervalkWh_max_fc_fix[0][i] = pd.concat([IntervalkWh_max_fc_fix[1][i], IntervalkWh_max_fc_fix[2][i]], axis=1)
            Car_table_fc_fix[0][i] = pd.concat([Car_table_fc_fix[1][i], Car_table_fc_fix[2][i]], axis=1)
            Eta_table_RT[0][i] = pd.concat([Eta_table_RT[1][i], Eta_table_RT[2][i]], axis=1)
        #################################################################################
        # VERSION: DA/Offline CVX Optimization
        #################################################################################
        H = 96
        _n_ev_physical_da = Car_table_fc_fix[0][DA_CASE_IDX].shape[1]
        if _n_ev_physical_da == 0:
            _dummy_ev_col = '__dummy_ev_zero__'
            Car_table_fc_fix[0][DA_CASE_IDX] = pd.DataFrame(columns=[_dummy_ev_col])
            SessionkWh_table_fc_fix[0][DA_CASE_IDX] = pd.DataFrame([[0.0]], columns=[_dummy_ev_col])
            IntervalkWh_max_fc_fix[0][DA_CASE_IDX] = pd.DataFrame(0.0, index=range(H), columns=[_dummy_ev_col])
        H_Start_DA = TheDate_Day0 + timedelta(minutes=0)
        Operator_SumColumn = np.ones((Car_table_fc_fix[0][DA_CASE_IDX].shape[1], 1))  # size: num_cars x 1
        Operator_SumRow = np.ones((1, H)) # size: 1 x num_step
        c_e_TOU_AL_DA = np.asarray(c_e_TOU_AL, dtype=float).copy() # $/kWh
        Bid_Pr_DA_i = Bid_Pr_DA[:]
        Bid_Pr_RT_i = Bid_Pr_RT[:]
        AS_Pr_RU_DA_i = AS_Pr_RU_DA[:]
        AS_Pr_RD_DA_i = AS_Pr_RD_DA[:]
        AS_Pr_RU_RT_i = AS_Pr_RU_RT[:]
        AS_Pr_RD_RT_i = AS_Pr_RD_RT[:]
        AS_Pr_SP_DA_i = AS_Pr_SP_DA[:]
        AS_Pr_NSP_DA_i = AS_Pr_NSP_DA[:]
        AS_Pr_SP_RT_i = AS_Pr_SP_RT[:]
        AS_Pr_NSP_RT_i = AS_Pr_NSP_RT[:]
        # EV service revenue with a EV charging rate
        # https://transportation.ucsd.edu/commute/ev-stations.html
        c_EV_service = np.ones(H) * 0.4 # $/kWh
        # Filter for Peak demand Period
        PP_start = 16 * 4
        PP_end = 21 * 4
        Filter = 1
        # output variables for All EVs, Tesla, and Non-Tesla, each for RT (2 cases) and DA optimization
        EnergyDemand_Table_Opt_cap = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        EnergyDemand_Step_Opt_cap = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        # Construct the problem
        Cost_Opt = cp.Variable()
        penalty_BESS = 0
        # EV Variables
        EnergyDemand_Table_Opt = cp.Variable((H, Car_table_fc_fix[0][DA_CASE_IDX].shape[1]))
        EnergyDemand_Step_Opt = cp.Variable(H)
        p_EV = cp.Variable(H, nonneg=True)  # total EV charging power, kW
        p_GI = cp.Variable(H)  # grid import power, kW
        # BESS variables
        p_BESS = cp.Variable(H)
        p_ch_BESS = cp.Variable(H, nonneg=True)
        p_dch_BESS = cp.Variable(H, nonneg=True)
        p_ch_BESS_WM = cp.Variable(H, nonneg=True)
        p_dch_BESS_WM = cp.Variable(H, nonneg=True)
        p_ch_BESS_NWM = cp.Variable(H, nonneg=True)
        p_dch_BESS_NWM = cp.Variable(H, nonneg=True)
        soc_BESS = cp.Variable(H+1) # SOC from 0 to H
        if Day == run_days[0]:
            SOC_BESS_initial = 0.5
        else:
            SOC_BESS_initial = SOC_BESS_daily_end[0]  # use the final SOC of previous day in 100% eta as initial SOC of current day
        # BESS constraints
        constraints_bess = [
            p_BESS == p_ch_BESS - p_dch_BESS, # either positive or negative
            p_ch_BESS == p_ch_BESS_WM + p_ch_BESS_NWM,
            p_dch_BESS == p_dch_BESS_WM + p_dch_BESS_NWM,
            p_BESS <= P_BESS_max,
            p_BESS >= -P_BESS_max,
            soc_BESS[0] == SOC_BESS_initial,
            soc_BESS <= SOC_BESS_max,
            soc_BESS >= SOC_BESS_min,
            soc_BESS[1:] == soc_BESS[:-1] + (dt_h / C_BESS) * (cp.multiply(p_ch_BESS, np.sqrt(gamma)) - cp.multiply(p_dch_BESS, 1 / np.sqrt(gamma))),
            dt_h * cp.sum(p_ch_BESS + p_dch_BESS) <= 2 * C_BESS * (SOC_BESS_max - SOC_BESS_min)  # limit total throughput in one day
        ]
        if is_last_run_day:
            constraints_bess += [soc_BESS[-1] == 0.5]
            penalty_BESS_target = 0.5
            penalty_BESS = (cp.norm(soc_BESS[-1] - penalty_BESS_target, 2)) * c_BESS_penalty # penalty coefficient
        else:
            penalty_BESS = cp.Constant(0.0)
        p_load_DA, p_PV_DA = _get_passive_meter_inputs(
            pd.date_range(TheDate_Day0, periods=H, freq=interval)
        )
        # Power balance
        constraints_balance = [
            p_GI == p_load_DA + p_EV + p_BESS - p_PV_DA
        ]
        Baseline_Opt_i = np.nan_to_num(Baseline_Opt_fc[0].repeat(4), nan=0.0, posinf=0.0, neginf=0.0) # TODO: current Baseline use the 100% perfect, need to update from DRAM
        B_EV = Baseline_Opt_i / dt_h  # convert kWh to kW for constraints and cost calculation
        P_EV_max_profile = np.asarray(IntervalkWh_max_fc_fix[0][DA_CASE_IDX], dtype=float).sum(axis=1) / dt_h
        P_EV_max = float(np.max(P_EV_max_profile)) if P_EV_max_profile.size else 0.0
        M_EV = 1.05 * max(float(P_EV_max), float(np.nanmax(B_EV)), 1e-6)  # big-M must cover EV deviation from baseline, especially low-EV days
        # WM constraints
        p_DA = cp.Variable(H)
        p_RT = cp.Variable(H)
        c_RU_DA = cp.Variable(H, nonneg=True)
        c_RD_DA = cp.Variable(H, nonneg=True)
        c_SP_DA = cp.Variable(H, nonneg=True)
        c_NSP_DA = cp.Variable(H, nonneg=True)
        # Signed RT AS adjustments: actual delivered capacity is c_DA + c_RT.
        c_RU_RT = cp.Variable(H)
        c_RD_RT = cp.Variable(H)
        c_SP_RT = cp.Variable(H)
        c_NSP_RT = cp.Variable(H)
        p_up = P_BESS_max + B_EV
        p_down = np.maximum(P_BESS_max - B_EV + P_EV_max_profile, 0.0)
        p_ch_EV_WM = cp.Variable(H, nonneg=True)      # EV baseline deviation used for WM products
        p_dch_EV_WM = cp.Variable(H, nonneg=True)     # EV baseline deviation used for WM products
        p_ch_EV_NWM = cp.Variable(H, nonneg=True)  # EV baseline deviation needed for non-WM service feasibility
        p_dch_EV_NWM = cp.Variable(H, nonneg=True)
        p_ch_net = cp.Variable(H, nonneg=True)
        p_dch_net = cp.Variable(H, nonneg=True)
        b_ch_BESS = cp.Variable(H, boolean=True)
        b_dch_BESS = cp.Variable(H, boolean=True)
        b_ch_EV = cp.Variable(H, boolean=True)
        b_dch_EV = cp.Variable(H, boolean=True)
        b_ch_net = cp.Variable(H, boolean=True)
        b_dch_net = cp.Variable(H, boolean=True)
        alpha_RU, alpha_RD, alpha_SP, alpha_NSP = _get_as_activation_profile(
            pd.date_range(H_Start_DA, periods=H, freq=interval)
        )
        p_wm_actual_DA = p_DA + p_RT
        # Only controllable EV+BESS can create wholesale energy or AS positions.
        # Passive building/PV affect p_GI and retail cost only.
        p_wm_ctrl_DA = p_wm_actual_DA
        p_as_deployed_DA = (
            cp.multiply(alpha_RU, c_RU_DA + c_RU_RT)
            - cp.multiply(alpha_RD, c_RD_DA + c_RD_RT)
            + cp.multiply(alpha_SP, c_SP_DA + c_SP_RT)
            + cp.multiply(alpha_NSP, c_NSP_DA + c_NSP_RT)
        )
        if Enable_WM:
            constraints_WM = [
                c_RU_DA + c_RU_RT >= 0,
                c_RD_DA + c_RD_RT >= 0,
                c_SP_DA + c_SP_RT >= 0,
                c_NSP_DA + c_NSP_RT >= 0,
                c_RU_DA + c_SP_DA + c_NSP_DA <= p_up,
                c_RD_DA <= p_down,
                # Capacity
                # p_DA + p_RT + c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= p_up,
                # -(p_DA + p_RT) + c_RD_DA + c_RD_RT <= p_down,
                cp.pos(p_wm_ctrl_DA) + c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= p_up,
                cp.pos(-p_wm_ctrl_DA) + c_RD_DA + c_RD_RT <= p_down,
                p_DA <= p_up,
                p_RT <= p_up,
                p_DA >= -p_down,
                p_RT >= -p_down,
                # Energy
                b_ch_BESS + b_dch_BESS <= 1,
                b_ch_EV + b_dch_EV <= 1,
                b_ch_net + b_dch_net <= 1,
                p_EV - B_EV == (p_ch_EV_WM - p_dch_EV_WM) + (p_ch_EV_NWM - p_dch_EV_NWM),
                # p_dch_EV_WM == (B_EV - p_EV) * b_dch_EV,
                p_dch_EV_WM + p_dch_EV_NWM <= M_EV * b_dch_EV,
                # p_ch_EV_WM == -(B_EV - p_EV) * b_ch_EV,
                p_ch_EV_WM + p_ch_EV_NWM <= M_EV * b_ch_EV,
                p_dch_net <= p_wm_ctrl_DA + p_as_deployed_DA + cp.multiply(p_up, b_ch_net),
                p_dch_net >= p_wm_ctrl_DA + p_as_deployed_DA,
                p_dch_BESS <= P_BESS_max * b_dch_BESS,
                p_ch_net <= -(p_wm_ctrl_DA + p_as_deployed_DA) + cp.multiply(p_down, b_dch_net),
                p_ch_net >= -(p_wm_ctrl_DA + p_as_deployed_DA),
                p_ch_BESS <= P_BESS_max * b_ch_BESS,
                p_ch_net <= cp.multiply(p_down, b_ch_net),
                p_dch_net <= cp.multiply(p_up, b_dch_net),
                p_ch_net - p_dch_net == p_ch_BESS_WM - p_dch_BESS_WM + p_ch_EV_WM - p_dch_EV_WM,
                # Duration
                soc_BESS[1:] >= SOC_BESS_min + (2 * dt_h) / (C_BESS * np.sqrt(gamma)) * (c_RU_DA + c_RU_RT + c_SP_DA + c_SP_RT + c_NSP_DA + c_NSP_RT),
                soc_BESS[1:] <= SOC_BESS_max - (2 * dt_h * np.sqrt(gamma)) / C_BESS * (c_RD_DA + c_RD_RT)
            ]
            # 2025 CAISO DAM awards are hourly. The DA model uses 15-min physics,
            # so all DA energy and AS quantities are held constant within each hour.
            _da_block = int(round(1.0 / dt_h))
            for _h0 in range(0, H, _da_block):
                _h1 = min(_h0 + _da_block, H)
                if _h1 - _h0 > 1:
                    constraints_WM += [
                        p_DA[_h0 + 1:_h1] == p_DA[_h0],
                        c_RU_DA[_h0 + 1:_h1] == c_RU_DA[_h0],
                        c_RD_DA[_h0 + 1:_h1] == c_RD_DA[_h0],
                        c_SP_DA[_h0 + 1:_h1] == c_SP_DA[_h0],
                        c_NSP_DA[_h0 + 1:_h1] == c_NSP_DA[_h0],
                    ]
            # VERSION: Add-ons when direction aligned
            if Direction_Aligned_AddOn:
                constraints_WM += [
                    c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= cp.multiply(p_up, b_dch_net),
                    c_RD_DA + c_RD_RT <= cp.multiply(p_down, b_ch_net),
                ]
            else:
                print(f"--> Current Direction_Aligned_AddOn Status: {Direction_Aligned_AddOn}")
            # VERSION: WM only mode (disable non-WM BESS channel)
            if WM_Mode == 'wm_only':
                constraints_WM += [
                    p_ch_BESS_NWM == 0,
                    p_dch_BESS_NWM == 0,
                ]
            # WM Revenue calculation
            Revenue_WM = (
                dt_h * (
                    Bid_Pr_RT_i @ p_as_deployed_DA
                    + Bid_Pr_DA_i @ p_DA + Bid_Pr_RT_i @ p_RT
                    + AS_Pr_RU_DA_i @ c_RU_DA + AS_Pr_RU_RT_i @ c_RU_RT + AS_Pr_RD_DA_i @ c_RD_DA + AS_Pr_RD_RT_i @ c_RD_RT
                    + AS_Pr_SP_DA_i @ c_SP_DA + AS_Pr_SP_RT_i @ c_SP_RT + AS_Pr_NSP_DA_i @ c_NSP_DA + AS_Pr_NSP_RT_i @ c_NSP_RT
                )
            )
        else:
            constraints_WM = [
                p_DA == 0, p_RT == 0,
                c_RU_DA == 0, c_RD_DA == 0, c_SP_DA == 0, c_NSP_DA == 0,
                c_RU_RT == 0, c_RD_RT == 0, c_SP_RT == 0, c_NSP_RT == 0,
                p_ch_BESS_WM == 0, p_dch_BESS_WM == 0,
                p_ch_EV_WM == 0, p_dch_EV_WM == 0, p_ch_net == 0, p_dch_net == 0,
            ]
            Revenue_WM = 0
        # Cost-related constraints
        constraints_cost = [
            Cost_Opt >= (
                c_NCD * cp.maximum(cp.max(p_GI) - M_Th_NCD[DA_CASE_IDX], 0)
                + c_PD * cp.maximum(cp.max(Filter * p_GI[PP_start:PP_end]) - M_Th_PD[DA_CASE_IDX], 0)
                + c_e_TOU_AL_DA @ p_GI * dt_h
                - c_EV_service @ p_EV * dt_h
                - Revenue_WM
            )
        ]
        # EV constraints
        constraints_EV = [
            p_EV == EnergyDemand_Step_Opt / dt_h,  # kW
            EnergyDemand_Table_Opt @ Operator_SumColumn == cp.reshape(EnergyDemand_Step_Opt, (H, 1), order='C'),
            (Operator_SumRow @ EnergyDemand_Table_Opt == SessionkWh_table_fc_fix[0][DA_CASE_IDX]) if SERVICE_LEVEL_MIN == 1.0 else (Operator_SumRow @ EnergyDemand_Table_Opt >= SERVICE_LEVEL_MIN * SessionkWh_table_fc_fix[0][DA_CASE_IDX]),
            EnergyDemand_Table_Opt >= 0,
            EnergyDemand_Table_Opt <= IntervalkWh_max_fc_fix[0][DA_CASE_IDX].iloc[:, :]
        ]
        if SERVICE_LEVEL_MIN < 1.0:
            constraints_EV += [Operator_SumRow @ EnergyDemand_Table_Opt <= SessionkWh_table_fc_fix[0][DA_CASE_IDX]]
        # NOTE: Combine all constraints
        constraints = constraints_cost + constraints_EV + constraints_bess + constraints_WM + constraints_balance
        # Solve the problem            
        # solver_opts = {"Method": 2, "Seed": 0, "Threads": 1}
        objective = cp.Minimize(Cost_Opt + penalty_BESS)
        prob = cp.Problem(objective, constraints)
        result = prob.solve(solver=cp.GUROBI, verbose=False, MIPGap=GUROBI_MIPGAP, Threads=SOLVER_THREADS, Presolve=1, ignore_dpp=True)
        status = prob.status
        Status_list_DA = Status_list_DA + [status]
        if status != 'optimal':
            print("DA optimization does not have an optimal solution!")
            sys.exit()
        # Save DA results            
        Solver_Outputs_DA = {
            'Cost_Opt': [],
            'Cost_PD': [],
            'Cost_NCD': [],
            'Cost_TOU': [],
            'Revenue_EV': [],
            'penalty_BESS': [],
            'Revenue_WM': [],
            'p_GI': [],
            'p_EV': [],
            'p_ch_BESS': [],
            'p_dch_BESS': [],
            'p_ch_BESS_WM': [],
            'p_dch_BESS_WM': [],
            'p_ch_BESS_NWM': [],
            'p_dch_BESS_NWM': [],
            'p_ch_EV': [],
            'p_dch_EV': [],
            'p_ch_EV_WM': [],
            'p_dch_EV_WM': [],
            'p_ch_EV_NWM': [],
            'p_dch_EV_NWM': [],
            'p_ch_net': [],
            'p_dch_net': [],
            'soc_BESS': [],
            'p_BESS': [],
            'c_RU_DA': [],
            'c_RD_DA': [],
            'c_SP_DA': [],
            'c_NSP_DA': [],
            'p_DA': [],
            'c_RU_RT': [],
            'c_RD_RT': [],
            'c_SP_RT': [],
            'c_NSP_RT': [],
            'p_RT': [],
            'LMP_DA': Bid_Pr_DA,
            'LMP_RT': Bid_Pr_RT,
            'AS_Pr_RU_DA': AS_Pr_RU_DA,
            'AS_Pr_RU_RT': AS_Pr_RU_RT,
            'AS_Pr_RD_DA': AS_Pr_RD_DA,
            'AS_Pr_RD_RT': AS_Pr_RD_RT,
            'AS_Pr_SP_DA': AS_Pr_SP_DA,
            'AS_Pr_NSP_DA': AS_Pr_NSP_DA,
            'AS_Pr_SP_RT': AS_Pr_SP_RT,
            'AS_Pr_NSP_RT': AS_Pr_NSP_RT,
            'Baseline(kW)': Baseline_Opt_i / dt_h,
            'p_up_bound(kW)': p_up,
            'p_down_bound(kW)': p_down,
            'P_EV_max': P_EV_max_profile,
        }
        c_RU_DA_value_DA = c_RU_DA.value
        c_RD_DA_value_DA = c_RD_DA.value
        c_SP_DA_value_DA = c_SP_DA.value
        c_NSP_DA_value_DA = c_NSP_DA.value
        p_DA_value_DA = p_DA.value
        p_RT_value_DA = p_RT.value
        EnergyDemand_Table_Opt_value = EnergyDemand_Table_Opt.value
        EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX] = pd.DataFrame(EnergyDemand_Table_Opt_value, columns=list(Car_table_fc_fix[0][DA_CASE_IDX].columns))
        EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX][EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX] <= 0.0001] = 0 
        EnergyDemand_Table_Opt_cap[1][DA_CASE_IDX] = EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX].iloc[:, :len(Car_table_fc_fix[1][DA_CASE_IDX])]
        EnergyDemand_Table_Opt_cap[2][DA_CASE_IDX] = EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX].iloc[:, len(Car_table_fc_fix[2][DA_CASE_IDX]):]
        EnergyDemand_Step_Opt_cap[0][DA_CASE_IDX] = EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX].sum(axis=1)
        # dispatch_Offline = EnergyDemand_Table_Opt_cap[0][DA_CASE_IDX]
        Solver_Outputs_DA['Cost_Opt'] = Cost_Opt.value.item()
        Solver_Outputs_DA['Cost_PD'] = (c_PD * np.maximum(p_GI.value[PP_start:PP_end].max() - M_Th_PD[DA_CASE_IDX], 0)).item()
        Solver_Outputs_DA['Cost_NCD'] = (c_NCD * np.maximum(np.max(p_GI.value) - M_Th_NCD[DA_CASE_IDX], 0)).item()
        Solver_Outputs_DA['Cost_TOU'] = (np.sum(c_e_TOU_AL_DA * p_GI.value) * dt_h).item()
        Solver_Outputs_DA['Revenue_EV'] = (np.sum(c_EV_service * p_EV.value) * dt_h).item()
        Solver_Outputs_DA['penalty_BESS'] = penalty_BESS.value.item()
        Solver_Outputs_DA['Revenue_WM'] = Revenue_WM.value.item() if Enable_WM else 0.0
        Solver_Outputs_DA['p_GI'] = p_GI.value.flatten()
        Solver_Outputs_DA['p_EV'] = p_EV.value.flatten()
        Solver_Outputs_DA['p_ch_BESS'] = p_ch_BESS.value.flatten()
        Solver_Outputs_DA['p_dch_BESS'] = p_dch_BESS.value.flatten()
        Solver_Outputs_DA['p_ch_BESS_WM'] = p_ch_BESS_WM.value.flatten()
        Solver_Outputs_DA['p_dch_BESS_WM'] = p_dch_BESS_WM.value.flatten()
        Solver_Outputs_DA['p_ch_BESS_NWM'] = p_ch_BESS_NWM.value.flatten()
        Solver_Outputs_DA['p_dch_BESS_NWM'] = p_dch_BESS_NWM.value.flatten()
        # Backward-compatible output keys: p_ch_EV/p_dch_EV now store the EV WM channel.
        Solver_Outputs_DA['p_ch_EV'] = p_ch_EV_WM.value.flatten()
        Solver_Outputs_DA['p_dch_EV'] = p_dch_EV_WM.value.flatten()
        Solver_Outputs_DA['p_ch_EV_WM'] = p_ch_EV_WM.value.flatten()
        Solver_Outputs_DA['p_dch_EV_WM'] = p_dch_EV_WM.value.flatten()
        _ev_wm_dev_DA = np.asarray(p_ch_EV_WM.value).flatten() - np.asarray(p_dch_EV_WM.value).flatten()
        _ev_resid_DA = np.asarray(p_EV.value).flatten() - np.asarray(B_EV).flatten() - _ev_wm_dev_DA
        if p_ch_EV_NWM.value is None or p_dch_EV_NWM.value is None:
            _p_ch_EV_NWM_DA = np.maximum(_ev_resid_DA, 0.0)
            _p_dch_EV_NWM_DA = np.maximum(-_ev_resid_DA, 0.0)
        else:
            _p_ch_EV_NWM_DA = p_ch_EV_NWM.value.flatten()
            _p_dch_EV_NWM_DA = p_dch_EV_NWM.value.flatten()
        Solver_Outputs_DA['p_ch_EV_NWM'] = _p_ch_EV_NWM_DA
        Solver_Outputs_DA['p_dch_EV_NWM'] = _p_dch_EV_NWM_DA
        Solver_Outputs_DA['p_ch_net'] = p_ch_net.value.flatten()
        Solver_Outputs_DA['p_dch_net'] = p_dch_net.value.flatten()
        Solver_Outputs_DA['soc_BESS'] = soc_BESS.value.flatten()[1:]  # exclude the initial SOC
        Solver_Outputs_DA['p_BESS'] = p_BESS.value.flatten()
        Solver_Outputs_DA['c_RU_DA'] = c_RU_DA.value.flatten()
        Solver_Outputs_DA['c_RD_DA'] = c_RD_DA.value.flatten()
        Solver_Outputs_DA['c_SP_DA'] = c_SP_DA.value.flatten()
        Solver_Outputs_DA['c_NSP_DA'] = c_NSP_DA.value.flatten()
        Solver_Outputs_DA['p_DA'] = p_DA.value.flatten()
        Solver_Outputs_DA['c_RU_RT'] = c_RU_RT.value.flatten()
        Solver_Outputs_DA['c_RD_RT'] = c_RD_RT.value.flatten()
        Solver_Outputs_DA['c_SP_RT'] = c_SP_RT.value.flatten()
        Solver_Outputs_DA['c_NSP_RT'] = c_NSP_RT.value.flatten()
        Solver_Outputs_DA['p_RT'] = p_RT.value.flatten()
        Solver_Outputs_DA['P_EV_max'] = P_EV_max_profile
        # before accepting DA market commitments. This is a pre-submission screen,
        # not an RT ex-post reversal.
        # DA per-timestep WM profit breakdown for each c/p product
        bid_pr_da_i = np.array(Bid_Pr_DA).flatten()
        bid_pr_rt_i = np.array(Bid_Pr_RT).flatten()
        as_ru_da_i = np.array(AS_Pr_RU_DA_i).flatten()
        as_ru_rt_i = np.array(AS_Pr_RU_RT_i).flatten()
        as_rd_da_i = np.array(AS_Pr_RD_DA_i).flatten()
        as_rd_rt_i = np.array(AS_Pr_RD_RT_i).flatten()
        as_sp_da_i = np.array(AS_Pr_SP_DA_i).flatten()
        as_sp_rt_i = np.array(AS_Pr_SP_RT_i).flatten()
        as_nsp_da_i = np.array(AS_Pr_NSP_DA_i).flatten()
        as_nsp_rt_i = np.array(AS_Pr_NSP_RT_i).flatten()
        if Enable_WM:
            wm_profit_table = pd.DataFrame({
                'p_DA_profit': dt_h * bid_pr_da_i * Solver_Outputs_DA['p_DA'], # 注意这里，DA处是 Solver_Outputs_DA，RT处是 Solver_Outputs_RT_Base
                'p_RT_profit': dt_h * bid_pr_rt_i * Solver_Outputs_DA['p_RT'],
                # 向上调节 RU
                'c_RU_DA_energy': dt_h * (alpha_RU * bid_pr_rt_i) * Solver_Outputs_DA['c_RU_DA'],
                'c_RU_DA_capacity': dt_h * as_ru_da_i * Solver_Outputs_DA['c_RU_DA'],
                'c_RU_RT_energy': dt_h * (alpha_RU * bid_pr_rt_i) * Solver_Outputs_DA['c_RU_RT'],
                'c_RU_RT_capacity': dt_h * as_ru_rt_i * Solver_Outputs_DA['c_RU_RT'],
                # 向下调节 RD (注意公式中的负号)
                'c_RD_DA_energy': dt_h * (-alpha_RD * bid_pr_rt_i) * Solver_Outputs_DA['c_RD_DA'],
                'c_RD_DA_capacity': dt_h * as_rd_da_i * Solver_Outputs_DA['c_RD_DA'],
                'c_RD_RT_energy': dt_h * (-alpha_RD * bid_pr_rt_i) * Solver_Outputs_DA['c_RD_RT'],
                'c_RD_RT_capacity': dt_h * as_rd_rt_i * Solver_Outputs_DA['c_RD_RT'],
                # 旋转备用 SP
                'c_SP_DA_energy': dt_h * (alpha_SP * bid_pr_rt_i) * Solver_Outputs_DA['c_SP_DA'],
                'c_SP_DA_capacity': dt_h * as_sp_da_i * Solver_Outputs_DA['c_SP_DA'],
                'c_SP_RT_energy': dt_h * (alpha_SP * bid_pr_rt_i) * Solver_Outputs_DA['c_SP_RT'],
                'c_SP_RT_capacity': dt_h * as_sp_rt_i * Solver_Outputs_DA['c_SP_RT'],
                # 非旋转备用 NSP
                'c_NSP_DA_energy': dt_h * (alpha_NSP * bid_pr_rt_i) * Solver_Outputs_DA['c_NSP_DA'],
                'c_NSP_DA_capacity': dt_h * as_nsp_da_i * Solver_Outputs_DA['c_NSP_DA'],
                'c_NSP_RT_energy': dt_h * (alpha_NSP * bid_pr_rt_i) * Solver_Outputs_DA['c_NSP_RT'],
                'c_NSP_RT_capacity': dt_h * as_nsp_rt_i * Solver_Outputs_DA['c_NSP_RT'],
            })
            wm_tou_table = pd.DataFrame({
                'p_DA_TOU_Cost': -dt_h * c_e_TOU_AL * Solver_Outputs_DA['p_DA'],
                'p_RT_TOU_Cost': -dt_h * c_e_TOU_AL * Solver_Outputs_DA['p_RT'],
                'c_RU_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_RU * Solver_Outputs_DA['c_RU_DA'],
                'c_RU_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_RU * Solver_Outputs_DA['c_RU_RT'],
                'c_RD_DA_TOU_Cost': dt_h * c_e_TOU_AL * alpha_RD * Solver_Outputs_DA['c_RD_DA'],
                'c_RD_RT_TOU_Cost': dt_h * c_e_TOU_AL * alpha_RD * Solver_Outputs_DA['c_RD_RT'],
                'c_SP_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_SP * Solver_Outputs_DA['c_SP_DA'],
                'c_SP_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_SP * Solver_Outputs_DA['c_SP_RT'],
                'c_NSP_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_NSP * Solver_Outputs_DA['c_NSP_DA'],
                'c_NSP_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_NSP * Solver_Outputs_DA['c_NSP_RT'],
            })
        else:
            cols = [
                'p_DA_profit', 'p_RT_profit',
                'c_RU_DA_energy', 'c_RU_DA_capacity', 'c_RU_RT_energy', 'c_RU_RT_capacity',
                'c_RD_DA_energy', 'c_RD_DA_capacity', 'c_RD_RT_energy', 'c_RD_RT_capacity',
                'c_SP_DA_energy', 'c_SP_DA_capacity', 'c_SP_RT_energy', 'c_SP_RT_capacity',
                'c_NSP_DA_energy', 'c_NSP_DA_capacity', 'c_NSP_RT_energy', 'c_NSP_RT_capacity'
            ]
            wm_profit_table = pd.DataFrame(np.zeros((H, len(cols))), columns=cols)
            wm_tou_table = pd.DataFrame(np.zeros((H, 10)), columns=[
                'p_DA_TOU_Cost', 'p_RT_TOU_Cost',
                'c_RU_DA_TOU_Cost', 'c_RU_RT_TOU_Cost',
                'c_RD_DA_TOU_Cost', 'c_RD_RT_TOU_Cost',
                'c_SP_DA_TOU_Cost', 'c_SP_RT_TOU_Cost',
                'c_NSP_DA_TOU_Cost', 'c_NSP_RT_TOU_Cost'
            ])
        wm_profit_table['profit_sum_all_products'] = wm_profit_table.sum(axis=1)
        wm_tou_table['TOU_Cost_sum_all_products'] = wm_tou_table.sum(axis=1)
        wm_profit_total_from_table = float(wm_profit_table['profit_sum_all_products'].sum())
        wm_profit_check_diff = float(abs(wm_profit_total_from_table - Solver_Outputs_DA['Revenue_WM']))
        Solver_Outputs_DA['WM_Profit_Table'] = wm_profit_table
        Solver_Outputs_DA['WM_Profit_Total_From_Table'] = wm_profit_total_from_table
        Solver_Outputs_DA['WM_TOU_Table'] = wm_tou_table
        Solver_Outputs_DA['WM_TOU_Total_From_Table'] = float(wm_tou_table['TOU_Cost_sum_all_products'].sum())
        Solver_Outputs_DA['WM_Profit_Check_Diff'] = wm_profit_check_diff
        if wm_profit_check_diff > 1e-3:
            print(f"DA WM profit check -> table_sum={wm_profit_total_from_table:.6f}, Revenue_WM={Solver_Outputs_DA['Revenue_WM']:.6f}, abs_diff={wm_profit_check_diff:.6e}")
        p_GI_DA = Solver_Outputs_DA['p_GI']
        # Save threshold BEFORE updating, so fig(f) shows previous day's threshold
        M_Th_NCD_prev = list(M_Th_NCD)  # snapshot of previous day's threshold
        M_Th_PD_prev = list(M_Th_PD)
        # DA threshold is 2, RT 100% and eta% are 0 and 1, respectively
        M_Th_NCD[DA_CASE_IDX] = np.maximum(p_GI_DA.max(), M_Th_NCD[DA_CASE_IDX])
        M_Th_PD[DA_CASE_IDX] = np.maximum(p_GI_DA[PP_start:PP_end].max(), M_Th_PD[DA_CASE_IDX])
        # M_Th_NCD[DA_CASE_IDX] = np.maximum(max((dispatch_Offline.sum(axis=1))/dt_h),M_Th_NCD[DA_CASE_IDX])
        # M_Th_PD[DA_CASE_IDX] = np.maximum(max((dispatch_Offline.sum(axis=1)[PP_start:PP_end])/dt_h),M_Th_PD[DA_CASE_IDX])
        M_Th_NCD_list[DA_CASE_IDX] = M_Th_NCD_list[DA_CASE_IDX] + [M_Th_NCD[DA_CASE_IDX]]
        M_Th_PD_list[DA_CASE_IDX] = M_Th_PD_list[DA_CASE_IDX] + [M_Th_PD[DA_CASE_IDX]]
        # bidding event hours:
        Baseline_hr  = [[] for _ in range(len(Cases))] # Base, Case1
        Baseline_96  = [[] for _ in range(len(Cases))]
        EventHour    = [[] for _ in range(len(Cases))]   # Base and Case1 (no marlet participation for V0G)
        EventHour_96 = [[] for _ in range(len(Cases))] 
        Offline_hr   = np.average(np.array(p_GI_DA * dt_h).reshape(-1, 4), axis=1)
        # Offline_hr   = np.average(np.array(dispatch_Offline.sum(axis=1)).reshape(-1, 4), axis=1) #ave every 4 entries of the 96  x 1 series
        for i in range(len(Cases)): # Base, Case1
            Baseline_hr[i] = Baseline_Opt_fc[i].copy()
            Baseline_96[i] = Baseline_Opt_fc[i].repeat(4).reshape(96, 1) #repeat 4 times of the 24 x 1 series
            EventHour[i] = Baseline_hr[i] - Offline_hr
            EventHour[i][EventHour[i]<0] = 0
            EventHour[i][EventHour[i]>0] = 1
            EventHour_96[i] = EventHour[i].repeat(4).reshape(96, 1) #repeat 4 times of the 24 x 1 series
        #################################################################################
        # VERSION: Real Time Optimization Implementation
        #################################################################################
        # Inputs for RT optimization
        Dispatch = [[[] for _ in range(len(Cases))] for _ in range(3)]
        IntervalkWh_table_Opt_list  = [[] for _ in range(len(Cases))] 
        SessionkWh_nEta_list = [[] for _ in range(len(Cases))]
        dispatch_t0 = [[],[]]
        # Results saving variables
        Cost_Opt_value_RT = np.zeros((len(Cases), 96))
        Cost_PD_value_RT = np.zeros((len(Cases), 96))
        Cost_NCD_value_RT = np.zeros((len(Cases), 96))
        Cost_TOU_value_RT = np.zeros((len(Cases), 96))
        Revenue_EV_value_RT = np.zeros((len(Cases), 96))
        Revenue_WM_value_RT = np.zeros((len(Cases), 96))
        penalty_BESS_value_RT = np.zeros((len(Cases), 96))
        penalty_WM_value_RT = np.zeros((len(Cases), 96))
        p_GI_value_RT = np.zeros((len(Cases), 96))
        p_EV_value_RT = np.zeros((len(Cases), 96))
        p_ch_BESS_value_RT = np.zeros((len(Cases), 96))
        p_dch_BESS_value_RT = np.zeros((len(Cases), 96))
        p_ch_BESS_WM_value_RT = np.zeros((len(Cases), 96))
        p_dch_BESS_WM_value_RT = np.zeros((len(Cases), 96))
        p_ch_BESS_NWM_value_RT = np.zeros((len(Cases), 96))
        p_dch_BESS_NWM_value_RT = np.zeros((len(Cases), 96))
        p_BESS_value_RT = np.zeros((len(Cases), 96))
        p_ch_EV_value_RT = np.zeros((len(Cases), 96))
        p_dch_EV_value_RT = np.zeros((len(Cases), 96))
        p_ch_EV_NWM_value_RT = np.zeros((len(Cases), 96))
        p_dch_EV_NWM_value_RT = np.zeros((len(Cases), 96))
        p_ch_net_value_RT = np.zeros((len(Cases), 96))
        p_dch_net_value_RT = np.zeros((len(Cases), 96))
        p_EV_max_value_RT = np.zeros((len(Cases), 96))
        soc_BESS_value_RT = np.zeros((len(Cases), 96))
        c_RU_DA_value_RT = np.zeros((len(Cases), 96))
        c_RD_DA_value_RT = np.zeros((len(Cases), 96))
        c_SP_DA_value_RT = np.zeros((len(Cases), 96))
        c_NSP_DA_value_RT = np.zeros((len(Cases), 96))
        p_DA_value_RT = np.zeros((len(Cases), 96))
        c_RU_RT_value_RT = np.zeros((len(Cases), 96))
        c_RD_RT_value_RT = np.zeros((len(Cases), 96))
        c_SP_RT_value_RT = np.zeros((len(Cases), 96))
        c_NSP_RT_value_RT = np.zeros((len(Cases), 96))
        p_RT_value_RT = np.zeros((len(Cases), 96))
        E_BESS_throughput_RT = np.zeros((len(Cases), 97)) # store energy throughput, like SOC from 0 to 97
        Th_NCD = [0 for _ in range(len(Cases))]
        Th_PD = [0 for _ in range(len(Cases))]
        #############################################################################
        Count = [[0 for _ in range(len(Cases))] for _ in range(3)]  # All EVs, Tesla, and Non-Tesla for active RT cases
        def _append_aligned_column(frame, column_name, values):
            # New EVs arrive one at a time. Rebuild the small 96-row table with concat
            # so pandas does not accumulate hundreds of fragmented column blocks.
            if isinstance(values, pd.DataFrame):
                if values.shape[1] != 1:
                    raise ValueError(f'Expected one source column for {column_name}, got {values.shape[1]}')
                column = values.iloc[:, 0].copy()
            else:
                column = values.copy() if isinstance(values, pd.Series) else pd.Series(values, index=frame.index)
            column.name = column_name
            return pd.concat([frame, column], axis=1).copy()
        for i in range(len(Cases)):
            for j in np.array(range(3)): # All EVs, Tela, and Non-Tesla 
                Dispatch[j][i] = pd.DataFrame(np.zeros(shape=(96, (Car_table_real[j][i]).shape[1])),columns=Car_table_real[j][i].columns)   
            IntervalkWh_table_Opt_list[i]  = pd.DataFrame(columns=["Interval start"])             
        # create real-time variables
        IntervalkWh_max_RT = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        Car_table_RT = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        SessionkWh_RT= [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        SessionkWh_nEta = [[[] for _ in range(len(Cases)+1)] for _ in range(3)]
        UpperBound = [[] for _ in range(3)] # all, Tesla, and Non-Tesla
        for i in range(len(Cases)+1):      # active RT cases and DA/V0G reference
            for j in range(3):  # all, Tesla, and Non-Tesla
                IntervalkWh_max_RT[j][i] = IntervalkWh_max_fc_fix[j][i].copy()
                Car_table_RT[j][i] = Car_table_fc_fix[j][i].copy()  
                SessionkWh_RT[j][i] = SessionkWh_table_fc_fix[j][i].copy()
                SessionkWh_nEta[j][i] = SessionkWh_table_fc_fix[j][i].copy()
        # If Case1 is enabled, apply Eta_min logic for sessions whose layover does not overlap event hours.
        if RUN_CASE1 and Fc_SessionkWh == 'PerfectSessionkWh':    
            for j in np.array(range(2))+1: #Tesla, non-Tesla                   
                overlap = pd.DataFrame((IntervalkWh_max_RT[j][1]*EventHour_96[1]).sum())
                for car in range(len(overlap)):
                    if overlap.iloc[car,0] == 0:
                        Eta_table_RT[j][1].iloc[:,car] = 1 
                SessionkWh_nEta[j][1] = SessionkWh_table_fc_fix[j][1]*Eta_table_RT[j][1]
            Eta_table_RT[0][1] = pd.concat([Eta_table_RT[1][1], Eta_table_RT[2][1]], axis=1)        
            SessionkWh_nEta[0][1] = pd.concat([SessionkWh_nEta[1][1], SessionkWh_nEta[2][1]], axis=1)   
        for j in range(3):
            UpperBound[j] = SessionkWh_nEta[j][0].copy() #100% base case
        SessionkWh_RT_ML = [[[] for _ in range(len(Cases))] for _ in range(3)] #All EVs, Tesla, and Non-Tesla
        # Shrinking horizon implementation
        Status_list_imp = [[] for _ in range(len(Cases))]  # two cases
        Solver_Outputs_RT = [[] for _ in range(len(Cases))] # two cases
        for i in range(len(Cases)):
            Solver_Outputs_RT[i] = {
                'Cost_Opt': [],
                'Cost_PD': [],
                'Cost_NCD': [],
                'Cost_TOU': [],
                'Revenue_EV': [],
                'Revenue_WM': [],
                'penalty_BESS': [],
                'penalty_WM': [],
                'p_GI': [],
                'p_EV': [],
                'p_ch_BESS': [],
                'p_dch_BESS': [],
                'p_ch_BESS_WM': [],
                'p_dch_BESS_WM': [],
                'p_ch_BESS_NWM': [],
                'p_dch_BESS_NWM': [],
                'p_ch_EV': [],
                'p_dch_EV': [],
            'p_ch_EV_WM': [],
            'p_dch_EV_WM': [],
            'p_ch_EV_NWM': [],
            'p_dch_EV_NWM': [],
                'p_ch_net': [],
                'p_dch_net': [],
                'soc_BESS': [],
                'c_RU_DA': [],
                'c_RD_DA': [],
                'c_SP_DA': [],
                'c_NSP_DA': [],
                'p_DA': [],
                'c_RU_RT': [],
                'c_RD_RT': [],
                'c_SP_RT': [],
                'c_NSP_RT': [],
                'p_RT': [],
                'Baseline(kW)': [],
                'P_EV_max': []
            }
        # Pure RT MPC replacement block for the original notebook cell.
        # This version removes all regex / dynamic rewrite logic.
        # It is designed to replace the original RT optimization block directly.
        user_sessions_cache = {}
        def _roll_vec(arr, start_idx, horizon, fill_mode='edge'):
            arr = np.asarray(arr, dtype=float).flatten()
            if arr.size == 0:
                return np.zeros(horizon, dtype=float)
            tail = arr[start_idx:start_idx + horizon] if start_idx < arr.size else np.array([], dtype=float)
            if tail.size >= horizon:
                return tail[:horizon]
            if tail.size == 0:
                fill_value = arr[-1] if fill_mode == 'edge' else 0.0
                return np.full(horizon, fill_value, dtype=float)
            fill_value = tail[-1] if fill_mode == 'edge' else 0.0
            return np.pad(tail, (0, horizon - tail.size), constant_values=fill_value)
        def _roll_df_rows(df, start_idx, horizon, fill=0.0):
            vals = np.asarray(df.values, dtype=float)
            n_rows, n_cols = vals.shape
            tail = vals[start_idx:start_idx + horizon, :] if start_idx < n_rows else np.zeros((0, n_cols), dtype=float)
            if tail.shape[0] >= horizon:
                out = tail[:horizon, :]
            else:
                pad = np.full((horizon - tail.shape[0], n_cols), float(fill), dtype=float)
                out = np.vstack([tail, pad])
            return pd.DataFrame(out, columns=df.columns)
        def _refresh_nonml_forecast_step(
            Fc_AtArrival,
            Fc_SessionkWh,
            Count,
            SessionkWh_RT,
            SessionkWh_table_fc_fix,
            SessionkWh_table_real,
            IntervalkWh_max_RT,
            IntervalkWh_max_fc_fix,
            IntervalkWh_max_real,
        ):
            if Fc_AtArrival == 'MLatArrival':
                return
            for i_case in [0]:
                for j_type in [1, 2]:
                    ncol = SessionkWh_RT[j_type][i_case].shape[1]
                    c = int(Count[j_type][i_case])
                    if c >= ncol:
                        continue
                    if Fc_SessionkWh == 'PerfectSessionkWh':
                        src_sess = SessionkWh_table_real[j_type][i_case]
                        src_int = IntervalkWh_max_real[j_type][i_case]
                    else:
                        src_sess = SessionkWh_table_fc_fix[j_type][i_case]
                        src_int = IntervalkWh_max_fc_fix[j_type][i_case]
                    SessionkWh_RT[j_type][i_case].iloc[:, c:] = src_sess.iloc[:, c:]
                    IntervalkWh_max_RT[j_type][i_case].iloc[:, c:] = src_int.iloc[:, c:]
                SessionkWh_RT[0][i_case] = pd.concat([SessionkWh_RT[1][i_case], SessionkWh_RT[2][i_case]], axis=1)
                IntervalkWh_max_RT[0][i_case] = pd.concat([IntervalkWh_max_RT[1][i_case], IntervalkWh_max_RT[2][i_case]], axis=1)
        def _build_rt_mpc_case_model(H, n_cars_max, enable_wm, is_last_run_day, direction_aligned_addon, wm_mode):
            Operator_SumColumn_fix = np.ones((n_cars_max, 1))
            Operator_SumRow_fix = np.ones((1, H))
            c_ev_service_vec = np.ones(H) * 0.4
            Cost_Opt = cp.Variable()
            EnergyDemand_Table_Opt = cp.Variable((H, n_cars_max))
            EnergyDemand_Step_Opt = cp.Variable(H)
            p_EV = cp.Variable(H, nonneg=True)
            p_GI = cp.Variable(H)
            p_BESS = cp.Variable(H)
            p_ch_BESS = cp.Variable(H, nonneg=True)
            p_dch_BESS = cp.Variable(H, nonneg=True)
            p_ch_BESS_WM = cp.Variable(H, nonneg=True)
            p_dch_BESS_WM = cp.Variable(H, nonneg=True)
            p_ch_BESS_NWM = cp.Variable(H, nonneg=True)
            p_dch_BESS_NWM = cp.Variable(H, nonneg=True)
            soc_BESS = cp.Variable(H + 1)
            p_DA = cp.Variable(H)
            p_RT = cp.Variable(H)
            c_RU_DA = cp.Variable(H, nonneg=True)
            c_RD_DA = cp.Variable(H, nonneg=True)
            c_SP_DA = cp.Variable(H, nonneg=True)
            c_NSP_DA = cp.Variable(H, nonneg=True)
            # Signed RT AS adjustments: actual delivered capacity is c_DA + c_RT.
            c_RU_RT = cp.Variable(H)
            c_RD_RT = cp.Variable(H)
            c_SP_RT = cp.Variable(H)
            c_NSP_RT = cp.Variable(H)
            p_ch_EV_WM = cp.Variable(H, nonneg=True)      # EV baseline deviation used for WM products
            p_dch_EV_WM = cp.Variable(H, nonneg=True)     # EV baseline deviation used for WM products
            p_ch_EV_NWM = cp.Variable(H, nonneg=True)  # EV baseline deviation needed for non-WM service feasibility
            p_dch_EV_NWM = cp.Variable(H, nonneg=True)
            p_ch_net = cp.Variable(H, nonneg=True)
            p_dch_net = cp.Variable(H, nonneg=True)
            b_ch_BESS = cp.Variable(H, boolean=True)
            b_dch_BESS = cp.Variable(H, boolean=True)
            b_ch_EV = cp.Variable(H, boolean=True)
            b_dch_EV = cp.Variable(H, boolean=True)
            b_ch_net = cp.Variable(H, boolean=True)
            b_dch_net = cp.Variable(H, boolean=True)
            dev_ru_da = cp.Variable(H, nonneg=True)
            dev_rd_da = cp.Variable(H, nonneg=True)
            dev_sp_da = cp.Variable(H, nonneg=True)
            dev_nsp_da = cp.Variable(H, nonneg=True)
            dev_p_da = cp.Variable(H, nonneg=True)
            prm_soc0 = cp.Parameter()
            prm_soc_min_bound = cp.Parameter(H + 1)
            prm_soc_max_bound = cp.Parameter(H + 1)
            prm_throughput_used = cp.Parameter(nonneg=True)
            prm_peak_mask = cp.Parameter(H, nonneg=True)
            prm_mth_ncd = cp.Parameter()
            prm_mth_pd = cp.Parameter()
            prm_b_ev = cp.Parameter(H)
            prm_p_load = cp.Parameter(H, nonneg=True)
            prm_p_PV = cp.Parameter(H, nonneg=True)
            prm_p_up = cp.Parameter(H, nonneg=True)
            prm_p_down = cp.Parameter(H, nonneg=True)
            prm_p_ev_max = cp.Parameter(nonneg=True)
            prm_m_ev = cp.Parameter(H, nonneg=True)
            prm_tou = cp.Parameter(H)
            prm_ev_service = cp.Parameter(H, nonneg=True)
            prm_bid_da = cp.Parameter(H)
            prm_bid_rt = cp.Parameter(H)
            prm_bid_rt_abs = cp.Parameter(H, nonneg=True)
            prm_as_ru_da = cp.Parameter(H)
            prm_as_rd_da = cp.Parameter(H)
            prm_as_sp_da = cp.Parameter(H)
            prm_as_nsp_da = cp.Parameter(H)
            prm_as_ru_rt = cp.Parameter(H)
            prm_as_rd_rt = cp.Parameter(H)
            prm_as_sp_rt = cp.Parameter(H)
            prm_as_nsp_rt = cp.Parameter(H)
            prm_ref_ru_da = cp.Parameter(H)
            prm_ref_rd_da = cp.Parameter(H)
            prm_ref_sp_da = cp.Parameter(H)
            prm_ref_nsp_da = cp.Parameter(H)
            prm_ref_p_da = cp.Parameter(H)
            prm_da_lock_mask = cp.Parameter(H, nonneg=True)
            prm_ev_min = cp.Parameter(n_cars_max)
            prm_ev_max = cp.Parameter(n_cars_max)
            prm_ev_ub = cp.Parameter((H, n_cars_max), nonneg=True)
            prm_ev_today_min = cp.Parameter(n_cars_max)
            prm_today_mask = cp.Parameter(H, nonneg=True)
            prm_alpha_ru = cp.Parameter(H, nonneg=True)
            prm_alpha_rd = cp.Parameter(H, nonneg=True)
            prm_alpha_sp = cp.Parameter(H, nonneg=True)
            prm_alpha_nsp = cp.Parameter(H, nonneg=True)
            constraints = [
                p_BESS == p_ch_BESS - p_dch_BESS,
                p_ch_BESS == p_ch_BESS_WM + p_ch_BESS_NWM,
                p_dch_BESS == p_dch_BESS_WM + p_dch_BESS_NWM,
                p_BESS <= P_BESS_max,
                p_BESS >= -P_BESS_max,
                soc_BESS[0] == prm_soc0,
                soc_BESS <= prm_soc_max_bound,
                soc_BESS >= prm_soc_min_bound,
                soc_BESS[1:] == soc_BESS[:-1] + (dt_h / C_BESS) * (
                    cp.multiply(p_ch_BESS, np.sqrt(gamma)) - cp.multiply(p_dch_BESS, 1 / np.sqrt(gamma))
                ),
                # A 24-hour rolling horizon can cross midnight. Apply the daily
                # throughput budget separately to the current and next calendar day.
                dt_h * cp.sum(cp.multiply(prm_today_mask, p_ch_BESS + p_dch_BESS)) + prm_throughput_used <= 2 * C_BESS * (SOC_BESS_max - SOC_BESS_min),
                dt_h * cp.sum(cp.multiply(1 - prm_today_mask, p_ch_BESS + p_dch_BESS)) <= 2 * C_BESS * (SOC_BESS_max - SOC_BESS_min),
                p_GI == prm_p_load + p_EV + p_BESS - prm_p_PV,
                p_EV == EnergyDemand_Step_Opt / dt_h,
                EnergyDemand_Table_Opt @ Operator_SumColumn_fix == cp.reshape(EnergyDemand_Step_Opt, (H, 1), order='C'),
                Operator_SumRow_fix @ EnergyDemand_Table_Opt >= prm_ev_min,
                Operator_SumRow_fix @ EnergyDemand_Table_Opt <= prm_ev_max,
                prm_today_mask @ EnergyDemand_Table_Opt >= prm_ev_today_min,
                EnergyDemand_Table_Opt >= 0,
                EnergyDemand_Table_Opt <= prm_ev_ub,
            ]
            if RT_DA_COMMITMENT_MODE == 'penalty':
                constraints += [
                    dev_ru_da >= c_RU_DA - prm_ref_ru_da,
                    dev_ru_da >= -(c_RU_DA - prm_ref_ru_da),
                    dev_rd_da >= c_RD_DA - prm_ref_rd_da,
                    dev_rd_da >= -(c_RD_DA - prm_ref_rd_da),
                    dev_sp_da >= c_SP_DA - prm_ref_sp_da,
                    dev_sp_da >= -(c_SP_DA - prm_ref_sp_da),
                    dev_nsp_da >= c_NSP_DA - prm_ref_nsp_da,
                    dev_nsp_da >= -(c_NSP_DA - prm_ref_nsp_da),
                    dev_p_da >= p_DA - prm_ref_p_da,
                    dev_p_da >= -(p_DA - prm_ref_p_da),
                ]
            else:
                constraints += [
                    dev_ru_da == 0, dev_rd_da == 0, dev_sp_da == 0,
                    dev_nsp_da == 0, dev_p_da == 0,
                ]
            p_da_committed = cp.multiply(prm_da_lock_mask, prm_ref_p_da)
            # Historical settlement parameterization (fd0c8c3): internal p_RT is the actual RT position.
            # The paper/settlement p_RT is the increment or decrement from the fixed DA commitment.
            p_wm_actual = p_RT if RT_DA_COMMITMENT_MODE == 'settlement' else p_DA + p_RT
            p_wm_ctrl = p_wm_actual
            p_rt_settled = p_RT - p_da_committed if RT_DA_COMMITMENT_MODE == 'settlement' else p_RT
            p_as_deployed_RT = (
                cp.multiply(prm_alpha_ru, c_RU_DA + c_RU_RT)
                - cp.multiply(prm_alpha_rd, c_RD_DA + c_RD_RT)
                + cp.multiply(prm_alpha_sp, c_SP_DA + c_SP_RT)
                + cp.multiply(prm_alpha_nsp, c_NSP_DA + c_NSP_RT)
            )
            if enable_wm:
                constraints += [
                    c_RU_DA + c_RU_RT >= 0,
                    c_RD_DA + c_RD_RT >= 0,
                    c_SP_DA + c_SP_RT >= 0,
                    c_NSP_DA + c_NSP_RT >= 0,
                    # CAISO two-settlement logic: physical capability follows the net DA+RT position.
                    cp.pos(p_wm_ctrl) + c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= prm_p_up,
                    cp.pos(-p_wm_ctrl) + c_RD_DA + c_RD_RT <= prm_p_down,
                    b_ch_BESS + b_dch_BESS <= 1,
                    b_ch_EV + b_dch_EV <= 1,
                    b_ch_net + b_dch_net <= 1,
                    p_EV - prm_b_ev == (p_ch_EV_WM - p_dch_EV_WM) + (p_ch_EV_NWM - p_dch_EV_NWM),
                    p_dch_EV_WM + p_dch_EV_NWM <= cp.multiply(prm_m_ev, b_dch_EV),
                    p_ch_EV_WM + p_ch_EV_NWM <= cp.multiply(prm_m_ev, b_ch_EV),
                    p_dch_net <= p_wm_ctrl + p_as_deployed_RT + cp.multiply(prm_p_up, b_ch_net),
                    p_dch_net >= p_wm_ctrl + p_as_deployed_RT,
                    p_dch_BESS <= P_BESS_max * b_dch_BESS,
                    p_ch_net <= -(p_wm_ctrl + p_as_deployed_RT) + cp.multiply(prm_p_down, b_dch_net),
                    p_ch_net >= -(p_wm_ctrl + p_as_deployed_RT),
                    p_ch_BESS <= P_BESS_max * b_ch_BESS,
                    p_ch_net <= cp.multiply(prm_p_down, b_ch_net),
                    p_dch_net <= cp.multiply(prm_p_up, b_dch_net),
                    p_ch_net - p_dch_net == p_ch_BESS_WM - p_dch_BESS_WM + p_ch_EV_WM - p_dch_EV_WM,
                    soc_BESS[1:] >= SOC_BESS_min + (2 * dt_h) / (C_BESS * np.sqrt(gamma)) * (c_RU_DA + c_RU_RT + c_SP_DA + c_SP_RT + c_NSP_DA + c_NSP_RT),
                    soc_BESS[1:] <= SOC_BESS_max - (2 * dt_h * np.sqrt(gamma)) / C_BESS * (c_RD_DA + c_RD_RT),
                ]
                if direction_aligned_addon:
                    constraints += [
                        c_RU_DA + c_SP_DA + c_NSP_DA + c_RU_RT + c_SP_RT + c_NSP_RT <= cp.multiply(prm_p_up, b_dch_net),
                        c_RD_DA + c_RD_RT <= cp.multiply(prm_p_down, b_ch_net),
                    ]
                if wm_mode == 'wm_only':
                    constraints += [
                        p_ch_BESS_NWM == 0,
                        p_dch_BESS_NWM == 0,
                    ]
                if RT_DA_COMMITMENT_MODE == 'hard':
                    constraints += [
                        cp.multiply(prm_da_lock_mask, c_RU_DA - prm_ref_ru_da) == 0,
                        cp.multiply(prm_da_lock_mask, c_RD_DA - prm_ref_rd_da) == 0,
                        cp.multiply(prm_da_lock_mask, c_SP_DA - prm_ref_sp_da) == 0,
                        cp.multiply(prm_da_lock_mask, c_NSP_DA - prm_ref_nsp_da) == 0,
                        cp.multiply(prm_da_lock_mask, p_DA - prm_ref_p_da) == 0,
                    ]
                elif RT_DA_COMMITMENT_MODE == 'settlement':
                    # DA energy and DA AS awards are fixed inside today's commitment mask.
                    # This simplified case assumes availability; CAISO forced-buyback/no-pay
                    # settlement for outages is outside scope. Incremental awards use c_*_RT.
                    constraints += [
                        p_DA == 0,
                        c_RU_DA == cp.multiply(prm_da_lock_mask, prm_ref_ru_da),
                        c_RD_DA == cp.multiply(prm_da_lock_mask, prm_ref_rd_da),
                        c_SP_DA == cp.multiply(prm_da_lock_mask, prm_ref_sp_da),
                        c_NSP_DA == cp.multiply(prm_da_lock_mask, prm_ref_nsp_da),
                    ]
                elif RT_DA_COMMITMENT_MODE != 'penalty':
                    raise ValueError("RT_DA_COMMITMENT_MODE must be 'settlement', 'hard', or 'penalty'")
            else:
                constraints += [
                    p_DA == 0,
                    p_RT == 0,
                    c_RU_DA == 0,
                    c_RD_DA == 0,
                    c_SP_DA == 0,
                    c_NSP_DA == 0,
                    c_RU_RT == 0,
                    c_RD_RT == 0,
                    c_SP_RT == 0,
                    c_NSP_RT == 0,
                    p_ch_BESS_WM == 0,
                    p_dch_BESS_WM == 0,
                    p_ch_EV_WM == 0,
                    p_dch_EV_WM == 0,
                    p_ch_net == 0,
                    p_dch_net == 0,
                    b_ch_net == 0,
                    b_dch_net == 0,
                ]
            if enable_wm:
                if RT_DA_COMMITMENT_MODE in ['hard', 'settlement']:
                    penalty_WM = cp.Constant(0.0)
                else:
                    penalty_WM = M * (cp.multiply(prm_bid_rt_abs, prm_da_lock_mask) @ (dev_ru_da + dev_rd_da + dev_sp_da + dev_nsp_da + dev_p_da))
                Revenue_WM = (
                    dt_h * (
                        prm_bid_rt @ p_as_deployed_RT
                        + prm_bid_da @ (p_da_committed if RT_DA_COMMITMENT_MODE == 'settlement' else p_DA) + prm_bid_rt @ p_rt_settled
                        + prm_as_ru_da @ c_RU_DA + prm_as_ru_rt @ c_RU_RT + prm_as_rd_da @ c_RD_DA + prm_as_rd_rt @ c_RD_RT
                        + prm_as_sp_da @ c_SP_DA + prm_as_sp_rt @ c_SP_RT + prm_as_nsp_da @ c_NSP_DA + prm_as_nsp_rt @ c_NSP_RT
                    )
                )
            else:
                penalty_WM = cp.Constant(0.0)
                Revenue_WM = cp.Constant(0.0)
            constraints += [
                Cost_Opt >= (
                    c_NCD * cp.maximum(cp.max(p_GI) - prm_mth_ncd, 0)
                    + c_PD * cp.maximum(cp.max(cp.multiply(prm_peak_mask, p_GI)) - prm_mth_pd, 0)
                    + prm_tou @ p_GI * dt_h
                    - prm_ev_service @ p_EV * dt_h
                    - Revenue_WM
                )
            ]
            penalty_BESS = cp.Constant(0.0)
            objective = cp.Minimize(Cost_Opt + penalty_WM + penalty_BESS)
            prob = cp.Problem(objective, constraints)
            return {
                'prob': prob,
                'vars': {
                    'Cost_Opt': Cost_Opt,
                    'EnergyDemand_Table_Opt': EnergyDemand_Table_Opt,
                    'EnergyDemand_Step_Opt': EnergyDemand_Step_Opt,
                    'p_EV': p_EV,
                    'p_GI': p_GI,
                    'p_BESS': p_BESS,
                    'p_ch_BESS': p_ch_BESS,
                    'p_dch_BESS': p_dch_BESS,
                    'p_ch_BESS_WM': p_ch_BESS_WM,
                    'p_dch_BESS_WM': p_dch_BESS_WM,
                    'p_ch_BESS_NWM': p_ch_BESS_NWM,
                    'p_dch_BESS_NWM': p_dch_BESS_NWM,
                    'soc_BESS': soc_BESS,
                    'c_RU_DA': c_RU_DA,
                    'c_RD_DA': c_RD_DA,
                    'c_SP_DA': c_SP_DA,
                    'c_NSP_DA': c_NSP_DA,
                    'c_RU_RT': c_RU_RT,
                    'c_RD_RT': c_RD_RT,
                    'c_SP_RT': c_SP_RT,
                    'c_NSP_RT': c_NSP_RT,
                    'p_DA': p_DA,
                    'p_RT': p_RT,
                    'p_ch_EV': p_ch_EV_WM,
                    'p_dch_EV': p_dch_EV_WM,
                    'p_ch_EV_NWM': p_ch_EV_NWM,
                    'p_dch_EV_NWM': p_dch_EV_NWM,
                    'p_ch_net': p_ch_net,
                    'p_dch_net': p_dch_net,
                },
                'params': {
                    'prm_soc0': prm_soc0,
                    'prm_soc_min_bound': prm_soc_min_bound,
                    'prm_soc_max_bound': prm_soc_max_bound,
                    'prm_throughput_used': prm_throughput_used,
                    'prm_peak_mask': prm_peak_mask,
                    'prm_mth_ncd': prm_mth_ncd,
                    'prm_mth_pd': prm_mth_pd,
                    'prm_b_ev': prm_b_ev,
                    'prm_p_load': prm_p_load,
                    'prm_p_PV': prm_p_PV,
                    'prm_p_up': prm_p_up,
                    'prm_p_down': prm_p_down,
                    'prm_p_ev_max': prm_p_ev_max,
                    'prm_m_ev': prm_m_ev,
                    'prm_tou': prm_tou,
                    'prm_ev_service': prm_ev_service,
                    'prm_bid_da': prm_bid_da,
                    'prm_bid_rt': prm_bid_rt,
                    'prm_bid_rt_abs': prm_bid_rt_abs,
                    'prm_as_ru_da': prm_as_ru_da,
                    'prm_as_rd_da': prm_as_rd_da,
                    'prm_as_sp_da': prm_as_sp_da,
                    'prm_as_nsp_da': prm_as_nsp_da,
                    'prm_as_ru_rt': prm_as_ru_rt,
                    'prm_as_rd_rt': prm_as_rd_rt,
                    'prm_as_sp_rt': prm_as_sp_rt,
                    'prm_as_nsp_rt': prm_as_nsp_rt,
                    'prm_ref_ru_da': prm_ref_ru_da,
                    'prm_ref_rd_da': prm_ref_rd_da,
                    'prm_ref_sp_da': prm_ref_sp_da,
                    'prm_ref_nsp_da': prm_ref_nsp_da,
                    'prm_ref_p_da': prm_ref_p_da,
                    'prm_da_lock_mask': prm_da_lock_mask,
                    'prm_ev_min': prm_ev_min,
                    'prm_ev_max': prm_ev_max,
                    'prm_ev_ub': prm_ev_ub,
                    'prm_ev_today_min': prm_ev_today_min,
                    'prm_today_mask': prm_today_mask,
                    'prm_alpha_ru': prm_alpha_ru,
                    'prm_alpha_rd': prm_alpha_rd,
                    'prm_alpha_sp': prm_alpha_sp,
                    'prm_alpha_nsp': prm_alpha_nsp,
                },
                'exprs': {
                    'penalty_BESS': penalty_BESS,
                    'penalty_WM': penalty_WM,
                },
                'dpp_ok': prob.is_dpp(),
            }
        H = int(24 / dt_h)
        c_EV_service_Day0 = np.ones(96) * 0.4
        c_EV_service = np.ones(H) * 0.4
        rt_mpc_models = []
        for i_case in range(len(Cases)):
            day0_ev_cols = list(Car_table_real[0][i_case].columns)
            if Fc_SessionkWh == 'PerfectSessionkWh':
                day1_ev_cols = list(Car_table_TheDates_[0][3].columns)
                ev_cols_full = list(dict.fromkeys(day0_ev_cols + day1_ev_cols))
            else:
                ev_cols_full = day0_ev_cols
            if len(ev_cols_full) == 0:
                ev_cols_full = ['__dummy_ev__']
            model_i = _build_rt_mpc_case_model(
                H=H,
                n_cars_max=len(ev_cols_full),
                enable_wm=Enable_WM,
                is_last_run_day=is_last_run_day,
                direction_aligned_addon=Direction_Aligned_AddOn,
                wm_mode=WM_Mode,
            )
            model_i['ev_cols_full'] = ev_cols_full
            rt_mpc_models.append(model_i)
        time_prep_and_forecast = 0.0  # 记录：动态预测与参数更新时间
        time_gurobi_solve = 0.0       # 记录：纯 Gurobi 求解时间
        time_extract_results = 0.0    # 记录：提取结果和存数据时间
        time_daily_save = 0.0         # 记录：每天保存结果时间
        for i_t in range(int(24/dt_h)):
            # Retail-only is the controller-consistency benchmark. The final
            # experiment day also ends at midnight, so neither mode may optimize
            # against EV/load/price information outside the requested run period.
            truncate_at_midnight = (WM_Mode == 'retail_only') or is_last_run_day
            H = int(24/dt_h) - i_t if truncate_at_midnight else int(24/dt_h)
            c_EV_service = np.ones(H) * 0.4
            H_Start_RT = TheDate_Day0 + timedelta(minutes=i_t*dt_m_EV)
            H_End_RT = TheDate_Day0 + timedelta(hours=24)
            t_start_interval = H_Start_RT
            t_end_interval = H_Start_RT + timedelta(minutes=dt_m_EV)
            Time_table_RT = pd.DataFrame({'Interval start': [H_Start_RT + k * interval for k in range(H)]})
            p_load_RT, p_PV_RT = _get_passive_meter_inputs(Time_table_RT['Interval start'])
            if ArrivalTime_table_real[0][0].empty:
                Cars_arrived_LastInterval = np.array([], dtype=object)
            else:
                Cars_arrived_LastInterval = np.array(list(Car_table_real[0][0]))[
                    np.array(ArrivalTime_table_real[0][0].iloc[0, :] > t_start_interval - timedelta(minutes=dt_m_EV))
                    & np.array(ArrivalTime_table_real[0][0].iloc[0, :] <= t_start_interval)
                ]
            if len(Cars_arrived_LastInterval) > 0:
                for car_i in Cars_arrived_LastInterval:
                    for i in range(len(Cases)):
                        if car_i in Car_table_real[1][0]:
                            j = 1
                            IntervalkWh_CH_max = 4.16
                        elif car_i in Car_table_real[2][0]:
                            j = 2
                            IntervalkWh_CH_max = 1.664
                        else:
                            j = 0
                        Count[j][i] = Count[j][i] + 1
                        if Count[j][i] <= len(Car_table_RT[j][i].columns):
                            ColumnName_RT = Car_table_RT[j][i].columns[Count[j][i]-1]
                            ColumnName_real = Car_table_real[j][i].columns[Count[j][i]-1]
                            Car_table_RT[j][i].rename(columns={ColumnName_RT: ColumnName_real}, inplace=True)
                            IntervalkWh_max_RT[j][i].rename(columns={ColumnName_RT: ColumnName_real}, inplace=True)
                            SessionkWh_RT[j][i].rename(columns={ColumnName_RT: ColumnName_real}, inplace=True)
                            SessionkWh_nEta[j][i].rename(columns={ColumnName_RT: ColumnName_real}, inplace=True)
                            if i == 0:
                                UpperBound[j].rename(columns={ColumnName_RT: ColumnName_real}, inplace=True)
                            Eta_table_RT[j][i].rename(columns={ColumnName_RT: ColumnName_real}, inplace=True)
                            IntervalkWh_max_RT[j][i].iloc[:, Count[j][i]-1] = IntervalkWh_max_real[j][i].iloc[:, Count[j][i]-1]
                            SessionkWh_RT[j][i].iloc[:, Count[j][i]-1] = SessionkWh_table_real[j][i].iloc[:, Count[j][i]-1]
                            # 👇 🌟 新增：预测表也要跟着改名字，保持列名对齐！
                            IntervalkWh_max_fc_fix[j][i].rename(columns={ColumnName_RT: ColumnName_real}, inplace=True)
                            SessionkWh_table_fc_fix[j][i].rename(columns={ColumnName_RT: ColumnName_real}, inplace=True)
                        else:
                            ColumnName_real = Car_table_real[j][i].columns[Count[j][i]-1]
                            Car_table_RT[j][i] = _append_aligned_column(Car_table_RT[j][i], ColumnName_real, Car_table_real[j][i][ColumnName_real])
                            IntervalkWh_max_RT[j][i] = _append_aligned_column(IntervalkWh_max_RT[j][i], ColumnName_real, IntervalkWh_max_real[j][i][ColumnName_real])
                            SessionkWh_RT[j][i] = _append_aligned_column(SessionkWh_RT[j][i], ColumnName_real, SessionkWh_table_real[j][i][ColumnName_real])
                            SessionkWh_nEta[j][i] = _append_aligned_column(SessionkWh_nEta[j][i], ColumnName_real, SessionkWh_table_real[j][i][ColumnName_real])
                            if i == 0:
                                UpperBound[j] = _append_aligned_column(UpperBound[j], ColumnName_real, SessionkWh_table_real[j][i][ColumnName_real])
                            Eta_table_RT[j][i] = _append_aligned_column(Eta_table_RT[j][i], ColumnName_real, pd.Series(1.0, index=Eta_table_RT[j][i].index))
                            # 👇 🌟 新增：给预测表也硬塞进第 21 辆车的数据，撑大到 21 列！
                            IntervalkWh_max_fc_fix[j][i] = _append_aligned_column(IntervalkWh_max_fc_fix[j][i], ColumnName_real, IntervalkWh_max_real[j][i][ColumnName_real])
                            SessionkWh_table_fc_fix[j][i] = _append_aligned_column(SessionkWh_table_fc_fix[j][i], ColumnName_real, SessionkWh_table_real[j][i][ColumnName_real])
                        if Fc_AtArrival == 'MLatArrival':
                            if i == 0:
                                SessInfo_ThisUser = All_Sess[All_Sess['Car'] == car_i]
                                ThisUser = int(SessInfo_ThisUser['User'])
                                ThisUser_AT = SessInfo_ThisUser['Session start'].iloc[0]
                                if ThisUser in list(User_known):
                                    if ThisUser not in user_sessions_cache:
                                        Data_ThisUser = pd.read_csv('Sessions_Data/Sessions_Data_'+str(int(ThisUser))+'.csv')
                                        Data_ThisUser['Session start'] = pd.to_datetime(Data_ThisUser['Session start'])
                                        user_sessions_cache[ThisUser] = Data_ThisUser
                                    Data_ThisUser = user_sessions_cache[ThisUser]
                                    Data_ThisUser_Before = Data_ThisUser[Data_ThisUser['Session start'] < ThisUser_AT]
                                    if len(Data_ThisUser_Before) >= 10:
                                        ThisUser_PD_ED = KnownUser(ThisUser, SessInfo_ThisUser[['Session start', 'Arrival Hour']])
                                    else:
                                        ThisUser_PD_ED = UnKnownUser(SessInfo_ThisUser[['Session start','Arrival Hour', 'Battery (kWh)','Weekday', 'Max Charging Power']])
                                else:
                                    ThisUser_PD_ED = UnKnownUser(SessInfo_ThisUser[['Session start','Arrival Hour', 'Battery (kWh)','Weekday', 'Max Charging Power']])
                                ThisUser_DT = ThisUser_AT + timedelta(minutes=ThisUser_PD_ED[0])
                                Logic_ThisUser_Interval = (Time_table_real[0].iloc[:,0] >= ThisUser_AT) & (Time_table_real[0].iloc[:,0] <= ThisUser_DT)
                                ThisUser_Interval = np.array(Logic_ThisUser_Interval.astype(int)) * IntervalkWh_CH_max
                            SessionkWh_RT[j][i].iloc[:, Count[j][i]-1] = ThisUser_PD_ED[1]
                            IntervalkWh_max_RT[j][i].iloc[:, Count[j][i]-1] = ThisUser_Interval
                            SessionkWh_RT_ML[j][i] = SessionkWh_RT_ML[j][i] + [ThisUser_PD_ED[1]]
                        if sum(np.array(IntervalkWh_max_RT[j][i].iloc[:, Count[j][i]-1]).reshape(96,1) * EventHour_96[i]) == 0:
                            Eta_table_RT[j][i].iloc[:, Count[j][i]-1] = 1
                        SessionkWh_nEta[j][i].iloc[0, Count[j][i]-1] = SessionkWh_RT[j][i].iloc[0, Count[j][i]-1] * Eta_table_RT[j][i].iloc[0, Count[j][i]-1]
                        if i == 1:
                            UpperBound[j].iloc[0, Count[j][i]-1] = SessionkWh_nEta[j][0].iloc[0, Count[j][i]-1].copy()
                for i in range(len(Cases)):
                    IntervalkWh_max_RT[0][i] = pd.concat([IntervalkWh_max_RT[1][i], IntervalkWh_max_RT[2][i]], axis=1)
                    Car_table_RT[0][i] = pd.DataFrame(columns=IntervalkWh_max_RT[0][i].columns)
                    Eta_table_RT[0][i] = pd.concat([Eta_table_RT[1][i], Eta_table_RT[2][i]], axis=1)
                    SessionkWh_RT[0][i] = pd.concat([SessionkWh_RT[1][i], SessionkWh_RT[2][i]], axis=1)
                    SessionkWh_nEta[0][i] = pd.concat([SessionkWh_nEta[1][i], SessionkWh_nEta[2][i]], axis=1)
                    if i == 1:
                        UpperBound[0] = pd.concat([UpperBound[1], UpperBound[2]], axis=1)
                    # 👇 🌟 新增：必须把预测表也 Concat 起来！
                    IntervalkWh_max_fc_fix[0][i] = pd.concat([IntervalkWh_max_fc_fix[1][i], IntervalkWh_max_fc_fix[2][i]], axis=1)
                    SessionkWh_table_fc_fix[0][i] = pd.concat([SessionkWh_table_fc_fix[1][i], SessionkWh_table_fc_fix[2][i]], axis=1)
            if (i_t == 0) or (len(Cars_arrived_LastInterval) > 0):
                _refresh_nonml_forecast_step(Fc_AtArrival, Fc_SessionkWh, Count, SessionkWh_RT, SessionkWh_table_fc_fix, SessionkWh_table_real, IntervalkWh_max_RT, IntervalkWh_max_fc_fix, IntervalkWh_max_real)
            for i in range(len(Cases)):
                # Minimum-service runs must retain full requested-minus-delivered energy.
                # Clipping it to future availability would repeatedly erase the service floor.
                if SERVICE_LEVEL_MIN == 1.0 or i != 0:
                    SessionkWh_nEta[0][i] = pd.DataFrame(pd.concat([SessionkWh_nEta[0][i], pd.DataFrame(IntervalkWh_max_RT[0][i].iloc[i_t:].sum()).transpose()]).min(axis=0)).transpose()
            UpperBound[0] = pd.DataFrame(pd.concat([UpperBound[0], pd.DataFrame(IntervalkWh_max_RT[0][0].iloc[i_t:].sum()).transpose()]).min(axis=0)).transpose()
            if SAVE_STEP_DEBUG:
                var_list = [UpperBound[0], SessionkWh_nEta[0][1]]
                name_list = ['UpperBound', 'Session_nEta']
                for var in range(len(var_list)):
                    dir_Output = os.path.join(VERSION_DIR, 'Dispatch', '0_'+name_list[var]+'_'+WM_Mode+'.csv')
                    append_df_to_csv(var_list[var], dir_Output, index=False)
            c_e_TOU_AL_RT = _roll_vec(c_e_TOU_AL_rolling, i_t, H)
            Bid_Pr_DA_i = _roll_vec(Bid_Pr_DA_rolling, i_t, H)
            Bid_Pr_RT_i = _roll_vec(Bid_Pr_RT_rolling, i_t, H)
            AS_Pr_RU_DA_i = _roll_vec(AS_Pr_RU_DA_rolling, i_t, H)
            AS_Pr_RD_DA_i = _roll_vec(AS_Pr_RD_DA_rolling, i_t, H)
            AS_Pr_RU_RT_i = _roll_vec(AS_Pr_RU_RT_rolling, i_t, H)
            AS_Pr_RD_RT_i = _roll_vec(AS_Pr_RD_RT_rolling, i_t, H)
            AS_Pr_SP_DA_i = _roll_vec(AS_Pr_SP_DA_rolling, i_t, H)
            AS_Pr_NSP_DA_i = _roll_vec(AS_Pr_NSP_DA_rolling, i_t, H)
            AS_Pr_SP_RT_i = _roll_vec(AS_Pr_SP_RT_rolling, i_t, H)
            AS_Pr_NSP_RT_i = _roll_vec(AS_Pr_NSP_RT_rolling, i_t, H)
            # Calendar-based PD mask: a rolling horizon can contain pieces of
            # two different days, so an i_t-only slice can miss next-day on-peak hours.
            peak_mask = get_sdge_on_peak_mask(Time_table_RT['Interval start'])
            _peak_idx = np.flatnonzero(peak_mask > 0.5)
            Filter = int(_peak_idx.size > 0)
            PP_start = int(_peak_idx[0]) if Filter else 0
            PP_end = int(_peak_idx[-1] + 1) if Filter else 0
            track = np.zeros(H)
            for i in range(len(Cases)):
                t0 = time.time()
                EventHour_96_i = _roll_vec(EventHour_96[i].flatten(), i_t, H)
                Bid_Pr_cap = Bid_Pr_DA_i * EventHour_96_i
                Baseline_Opt_i = np.nan_to_num(_roll_vec(Baseline_96[i].flatten(), i_t, H), nan=0.0, posinf=0.0, neginf=0.0)
                NonEventHour_96_i = 1 - EventHour_96_i
                B_EV = Baseline_Opt_i / dt_h
                p_up = P_BESS_max + B_EV
                if i_t == 0:
                    if Day == run_days[0]:
                        SOC_BESS_initial = 0.5
                    else:
                        SOC_BESS_initial = SOC_BESS_daily_end[i]
                else:
                    SOC_BESS_initial = SOC_BESS_last_step[i]
                if truncate_at_midnight:
                    active_ev_cols = list(Car_table_real[0][i].columns)
                    if len(active_ev_cols) == 0:
                        active_ev_cols = ['__dummy_ev__']
                    model_i = _build_rt_mpc_case_model(
                        H=H,
                        n_cars_max=len(active_ev_cols),
                        enable_wm=Enable_WM,
                        is_last_run_day=is_last_run_day,
                        direction_aligned_addon=Direction_Aligned_AddOn,
                        wm_mode=WM_Mode,
                    )
                    model_i['ev_cols_full'] = active_ev_cols
                else:
                    model_i = rt_mpc_models[i]
                vars_i = model_i['vars']
                params_i = model_i['params']
                exprs_i = model_i['exprs']
                ev_cols_full = model_i['ev_cols_full']
                # ev_cols_full = IntervalkWh_max_RT[0][i].columns
                # 🌟 新增：打包当前数据，调用动态预测模块
                current_real_data = {
                    'ev_cols_full': ev_cols_full,
                    'forecast_method': 'Perfect' if Fc_SessionkWh == 'PerfectSessionkWh' else 'Persistence',
                    'interval_ub_real': IntervalkWh_max_RT[0][i],
                    'interval_ub_fc': IntervalkWh_max_fc_fix[0][i],
                    'tail_ub_realized': pd.concat([Dispatch[1][i], Dispatch[2][i]], axis=1).iloc[:i_t, :],
                    'session_min_real': SessionkWh_nEta[0][0] if i == 0 else SessionkWh_nEta[0][1],
                    'session_max_real': SessionkWh_nEta[0][0] if i == 0 else UpperBound[0],
                    'session_original_real': SessionkWh_table_real[0][i],
                    'session_min_fc': SessionkWh_table_fc_fix[0][i],
                    'session_max_fc': SessionkWh_table_fc_fix[0][i],
                    'next_interval_ub_real': IntervalkWh_max_TheDates_[0][3],
                    'next_session_real': SessionkWh_table_TheDates_[0][3],
                }
                ev_ub_pred, ev_min_pred, ev_max_pred, ev_today_min_pred = get_dynamic_ev_forecast(
                    i_t, H, current_real_data, forecast_method=current_real_data['forecast_method']
                )
                # The first MPC action is executed immediately, so it must only use
                # real availability at the current interval. Future rows may still
                # use persistence forecasts.
                if i_t < IntervalkWh_max_real[0][i].shape[0]:
                    current_real_ub = (
                        IntervalkWh_max_real[0][i]
                        .reindex(columns=ev_cols_full, fill_value=0.0)
                        .fillna(0.0)
                        .iloc[i_t]
                        .to_numpy(dtype=float)
                    )
                else:
                    current_real_ub = np.zeros(len(ev_cols_full), dtype=float)
                ev_ub_pred[0, :] = np.minimum(ev_ub_pred[0, :], np.maximum(current_real_ub, 0.0))
                # 找出两个预测表里列数比较小的那一个（比如一个是 20，一个是 21，只取前 20 辆车做削平，防止 NumPy 崩溃）
                safe_cars = min(ev_ub_pred.shape[1], ev_min_pred.shape[0] if ev_min_pred.ndim > 0 else len(ev_min_pred))
                max_deliverable = np.sum(ev_ub_pred[:, :safe_cars], axis=0)  
                ev_min_pred[:safe_cars] = np.minimum(ev_min_pred[:safe_cars], max_deliverable)  
                ev_max_pred[:safe_cars] = np.maximum(ev_min_pred[:safe_cars], ev_max_pred[:safe_cars])
                today_steps_left = max(0, min(H, 96 - i_t))
                today_mask = np.zeros(H, dtype=float)
                today_mask[:today_steps_left] = 1.0
                max_deliverable_today = np.sum(ev_ub_pred[:today_steps_left, :safe_cars], axis=0)
                ev_today_min = np.zeros_like(ev_today_min_pred, dtype=float)
                ev_today_min[:safe_cars] = np.minimum(ev_today_min_pred[:safe_cars], max_deliverable_today)
                soc_min_arr = np.full(H + 1, float(SOC_BESS_min))
                soc_max_arr = np.full(H + 1, float(SOC_BESS_max))
                if is_last_run_day:
                    target_idx = min(H, int(24/dt_h) - i_t)  # tonight 24:00 within the active horizon
                    if 0 <= target_idx <= H:
                        soc_min_arr[target_idx] = 0.5  # 锁死当天 24:00 的下限
                        soc_max_arr[target_idx] = 0.5  # 锁死当天 24:00 的上限
                _alpha_RU_RT, _alpha_RD_RT, _alpha_SP_RT, _alpha_NSP_RT = _get_as_activation_profile(Time_table_RT['Interval start'])
                params_i['prm_soc0'].value = float(SOC_BESS_initial)
                params_i['prm_soc_min_bound'].value = soc_min_arr
                params_i['prm_soc_max_bound'].value = soc_max_arr
                params_i['prm_soc0'].value = float(SOC_BESS_initial)
                params_i['prm_throughput_used'].value = float(E_BESS_throughput_RT[i, i_t])
                params_i['prm_peak_mask'].value = peak_mask
                params_i['prm_mth_ncd'].value = float(M_Th_NCD[i])
                params_i['prm_mth_pd'].value = float(M_Th_PD[i])
                _ev_agg_ub_kw = np.asarray(ev_ub_pred, dtype=float).sum(axis=1) / dt_h
                P_EV_max = float(np.max(_ev_agg_ub_kw)) if _ev_agg_ub_kw.size else 0.0
                p_down = np.maximum(P_BESS_max - B_EV + _ev_agg_ub_kw, 0.0)
                params_i['prm_b_ev'].value = np.asarray(B_EV, dtype=float)
                params_i['prm_p_load'].value = np.asarray(p_load_RT, dtype=float)
                params_i['prm_p_PV'].value = np.asarray(p_PV_RT, dtype=float)
                params_i['prm_p_up'].value = np.asarray(p_up, dtype=float)
                params_i['prm_p_down'].value = np.asarray(p_down, dtype=float)
                params_i['prm_p_ev_max'].value = float(P_EV_max)
                params_i['prm_m_ev'].value = 1.05 * np.maximum.reduce([_ev_agg_ub_kw, np.abs(np.asarray(B_EV, dtype=float)), np.full(H, 1e-6)])
                params_i['prm_tou'].value = np.asarray(c_e_TOU_AL_RT, dtype=float)
                params_i['prm_ev_service'].value = np.asarray(c_EV_service, dtype=float)
                params_i['prm_bid_da'].value = np.asarray(Bid_Pr_DA_i, dtype=float)
                params_i['prm_bid_rt'].value = np.asarray(Bid_Pr_RT_i, dtype=float)
                params_i['prm_bid_rt_abs'].value = np.asarray(np.abs(Bid_Pr_RT_i), dtype=float)
                params_i['prm_as_ru_da'].value = np.asarray(AS_Pr_RU_DA_i, dtype=float)
                params_i['prm_as_rd_da'].value = np.asarray(AS_Pr_RD_DA_i, dtype=float)
                params_i['prm_as_sp_da'].value = np.asarray(AS_Pr_SP_DA_i, dtype=float)
                params_i['prm_as_nsp_da'].value = np.asarray(AS_Pr_NSP_DA_i, dtype=float)
                params_i['prm_as_ru_rt'].value = np.asarray(AS_Pr_RU_RT_i, dtype=float)
                params_i['prm_as_rd_rt'].value = np.asarray(AS_Pr_RD_RT_i, dtype=float)
                params_i['prm_as_sp_rt'].value = np.asarray(AS_Pr_SP_RT_i, dtype=float)
                params_i['prm_as_nsp_rt'].value = np.asarray(AS_Pr_NSP_RT_i, dtype=float)
                params_i['prm_ref_ru_da'].value = np.maximum(np.asarray(_roll_vec(c_RU_DA_value_DA, i_t, H), dtype=float), 0.0)
                params_i['prm_ref_rd_da'].value = np.maximum(np.asarray(_roll_vec(c_RD_DA_value_DA, i_t, H), dtype=float), 0.0)
                params_i['prm_ref_sp_da'].value = np.maximum(np.asarray(_roll_vec(c_SP_DA_value_DA, i_t, H), dtype=float), 0.0)
                params_i['prm_ref_nsp_da'].value = np.maximum(np.asarray(_roll_vec(c_NSP_DA_value_DA, i_t, H), dtype=float), 0.0)
                params_i['prm_ref_p_da'].value = np.asarray(_roll_vec(p_DA_value_DA, i_t, H), dtype=float)
                params_i['prm_da_lock_mask'].value = (np.arange(H) + i_t < len(p_DA_value_DA)).astype(float)
                params_i['prm_ev_min'].value = ev_min_pred
                params_i['prm_ev_max'].value = ev_max_pred
                params_i['prm_ev_ub'].value = ev_ub_pred
                params_i['prm_ev_today_min'].value = ev_today_min
                params_i['prm_today_mask'].value = today_mask
                params_i['prm_alpha_ru'].value = _alpha_RU_RT
                params_i['prm_alpha_rd'].value = _alpha_RD_RT
                params_i['prm_alpha_sp'].value = _alpha_SP_RT
                params_i['prm_alpha_nsp'].value = _alpha_NSP_RT
                Cost_Opt = vars_i['Cost_Opt']
                EnergyDemand_Table_Opt = vars_i['EnergyDemand_Table_Opt']
                EnergyDemand_Step_Opt = vars_i['EnergyDemand_Step_Opt']
                p_EV = vars_i['p_EV']
                p_GI = vars_i['p_GI']
                p_BESS = vars_i['p_BESS']
                p_ch_BESS = vars_i['p_ch_BESS']
                p_dch_BESS = vars_i['p_dch_BESS']
                p_ch_BESS_WM = vars_i['p_ch_BESS_WM']
                p_dch_BESS_WM = vars_i['p_dch_BESS_WM']
                p_ch_BESS_NWM = vars_i['p_ch_BESS_NWM']
                p_dch_BESS_NWM = vars_i['p_dch_BESS_NWM']
                soc_BESS = vars_i['soc_BESS']
                c_RU_DA = vars_i['c_RU_DA']
                c_RD_DA = vars_i['c_RD_DA']
                c_SP_DA = vars_i['c_SP_DA']
                c_NSP_DA = vars_i['c_NSP_DA']
                c_RU_RT = vars_i['c_RU_RT']
                c_RD_RT = vars_i['c_RD_RT']
                c_SP_RT = vars_i['c_SP_RT']
                c_NSP_RT = vars_i['c_NSP_RT']
                p_DA = vars_i['p_DA']
                p_RT = vars_i['p_RT']
                p_ch_EV_WM = vars_i['p_ch_EV']
                p_dch_EV_WM = vars_i['p_dch_EV']
                p_ch_EV_NWM = vars_i['p_ch_EV_NWM']
                p_dch_EV_NWM = vars_i['p_dch_EV_NWM']
                p_ch_net = vars_i['p_ch_net']
                p_dch_net = vars_i['p_dch_net']
                t1 = time.time()
                time_prep_and_forecast += (t1 - t0)
                rt_solver_opts = dict(
                    MIPGap=GUROBI_MIPGAP,
                    Threads=SOLVER_THREADS,
                    Presolve=1,
                    MIPFocus=1,
                )
                _rt_watchdog_limit = RT_SOLVER_TIME_LIMIT if RT_SOLVER_TIME_LIMIT is not None else RT_SOLVER_EMERGENCY_TIME_LIMIT
                if _rt_watchdog_limit is not None:
                    rt_solver_opts['TimeLimit'] = _rt_watchdog_limit
                result = model_i['prob'].solve(
                    solver=cp.GUROBI,
                    verbose=False,
                    warm_start=True,
                    ignore_dpp=True,
                    reoptimize=True,
                    **rt_solver_opts,
                )
                t2 = time.time()
                time_gurobi_solve += (t2 - t1)
                status = model_i['prob'].status
                _record_rt_solver_status(model_i['prob'], H_Start_RT, i_t, Cases[i])
                Status_list_imp[i] = Status_list_imp[i] + [status]
                if (REQUIRE_RT_OPTIMAL and status != 'optimal') or status not in ['optimal', 'optimal_inaccurate', 'user_limit'] or p_GI.value is None or EnergyDemand_Table_Opt.value is None:
                    _ev_min_v = np.asarray(params_i['prm_ev_min'].value, dtype=float)
                    _ev_ub_v = np.asarray(params_i['prm_ev_ub'].value, dtype=float).sum(axis=0)
                    _ev_gap_v = _ev_min_v - _ev_ub_v
                    _bad_ev = np.where(_ev_gap_v > 1e-7)[0]
                    print(f'RT optimization failed: status={status}, step={i_t}, H_Start_RT={H_Start_RT}')
                    print(f'EV input feasibility: max(target-envelope)={_ev_gap_v.max():.6f} kWh, bad_count={len(_bad_ev)}, bad_indices={_bad_ev[:10].tolist()}')
                    _today_target = np.asarray(params_i['prm_ev_today_min'].value, dtype=float)
                    _today_mask = np.asarray(params_i['prm_today_mask'].value, dtype=float)
                    _today_env = _today_mask @ np.asarray(params_i['prm_ev_ub'].value, dtype=float)
                    _today_gap = _today_target - _today_env
                    print(f'Today EV feasibility: max(target-envelope)={_today_gap.max():.6f} kWh, bad_count={int(np.sum(_today_gap > 1e-7))}')
                    print(f'SOC0={params_i["prm_soc0"].value:.6f}, throughput_used={params_i["prm_throughput_used"].value:.6f}')
                    sys.exit()
                _p_da_commit_0 = float(params_i['prm_da_lock_mask'].value[0] * params_i['prm_ref_p_da'].value[0]) if RT_DA_COMMITMENT_MODE == 'settlement' else float(p_DA.value[0])
                _p_rt_deviation_0 = float(p_RT.value[0] - _p_da_commit_0) if RT_DA_COMMITMENT_MODE == 'settlement' else float(p_RT.value[0])
                revenue_WM_i_t = (
                    dt_h * (
                        Bid_Pr_RT_i[0] * (_alpha_RU_RT[0] * (c_RU_DA.value[0] + c_RU_RT.value[0]) - _alpha_RD_RT[0] * (c_RD_DA.value[0] + c_RD_RT.value[0]) + _alpha_SP_RT[0] * (c_SP_DA.value[0] + c_SP_RT.value[0]) + _alpha_NSP_RT[0] * (c_NSP_DA.value[0] + c_NSP_RT.value[0]))
                        + Bid_Pr_DA_i[0] * _p_da_commit_0 + Bid_Pr_RT_i[0] * _p_rt_deviation_0
                        + AS_Pr_RU_DA_i[0] * c_RU_DA.value[0] + AS_Pr_RU_RT_i[0] * c_RU_RT.value[0] + AS_Pr_RD_DA_i[0] * c_RD_DA.value[0] + AS_Pr_RD_RT_i[0] * c_RD_RT.value[0]
                        + AS_Pr_SP_DA_i[0] * c_SP_DA.value[0] + AS_Pr_SP_RT_i[0] * c_SP_RT.value[0] + AS_Pr_NSP_DA_i[0] * c_NSP_DA.value[0] + AS_Pr_NSP_RT_i[0] * c_NSP_RT.value[0]
                    )
                )
                SOC_BESS_last_step[i] = soc_BESS.value[1]
                Revenue_WM_value_RT[i, i_t] = revenue_WM_i_t.item()
                Revenue_EV_value_RT[i, i_t] = c_EV_service[0] * p_EV.value[0] * dt_h
                p_ch_BESS_value_RT[i, i_t] = p_ch_BESS.value[0]
                p_dch_BESS_value_RT[i, i_t] = p_dch_BESS.value[0]
                p_ch_BESS_WM_value_RT[i, i_t] = p_ch_BESS_WM.value[0]
                p_dch_BESS_WM_value_RT[i, i_t] = p_dch_BESS_WM.value[0]
                p_ch_BESS_NWM_value_RT[i, i_t] = p_ch_BESS_NWM.value[0]
                p_dch_BESS_NWM_value_RT[i, i_t] = p_dch_BESS_NWM.value[0]
                p_ch_EV_value_RT[i, i_t] = p_ch_EV_WM.value[0]
                p_dch_EV_value_RT[i, i_t] = p_dch_EV_WM.value[0]
                if p_ch_EV_NWM.value is None or p_dch_EV_NWM.value is None:
                    _b_ev0 = float(prm_b_ev.value[0]) if 'prm_b_ev' in locals() else float(B_EV[0])
                    _ev_resid0 = float(p_EV.value[0]) - _b_ev0 - (float(p_ch_EV_WM.value[0]) - float(p_dch_EV_WM.value[0]))
                    p_ch_EV_NWM_value_RT[i, i_t] = max(_ev_resid0, 0.0)
                    p_dch_EV_NWM_value_RT[i, i_t] = max(-_ev_resid0, 0.0)
                else:
                    p_ch_EV_NWM_value_RT[i, i_t] = p_ch_EV_NWM.value[0]
                    p_dch_EV_NWM_value_RT[i, i_t] = p_dch_EV_NWM.value[0]
                p_ch_net_value_RT[i, i_t] = p_ch_net.value[0]
                p_dch_net_value_RT[i, i_t] = p_dch_net.value[0]
                soc_BESS_value_RT[i, i_t] = soc_BESS.value[1]
                c_RU_DA_value_RT[i, i_t] = c_RU_DA.value[0]
                c_RD_DA_value_RT[i, i_t] = c_RD_DA.value[0]
                c_SP_DA_value_RT[i, i_t] = c_SP_DA.value[0]
                c_NSP_DA_value_RT[i, i_t] = c_NSP_DA.value[0]
                c_RU_RT_value_RT[i, i_t] = c_RU_RT.value[0]
                c_RD_RT_value_RT[i, i_t] = c_RD_RT.value[0]
                c_SP_RT_value_RT[i, i_t] = c_SP_RT.value[0]
                c_NSP_RT_value_RT[i, i_t] = c_NSP_RT.value[0]
                p_DA_value_RT[i, i_t] = _p_da_commit_0
                p_RT_value_RT[i, i_t] = _p_rt_deviation_0
                p_GI_value_RT[i, i_t] = p_GI.value[0]
                p_EV_value_RT[i, i_t] = p_EV.value[0]
                p_BESS_value_RT[i, i_t] = p_BESS.value[0]
                p_EV_max_value_RT[i, i_t] = float(_ev_agg_ub_kw[0]) if _ev_agg_ub_kw.size else 0.0
                penalty_BESS_value_RT[i, i_t] = float(exprs_i['penalty_BESS'].value) if exprs_i['penalty_BESS'].value is not None else 0.0
                penalty_WM_value_RT[i, i_t] = float(exprs_i['penalty_WM'].value) if Enable_WM and exprs_i['penalty_WM'].value is not None else 0.0
                E_BESS_throughput_RT[i, i_t+1] = E_BESS_throughput_RT[i, i_t] + dt_h * (p_ch_BESS.value[0] + p_dch_BESS.value[0])
                # only store the current time step cost/revenue values
                _projected_pd_values = np.asarray(p_GI.value)[peak_mask > 0.5]
                daily_pd = c_PD * np.maximum(np.max(_projected_pd_values) - M_Th_PD[i], 0) if _projected_pd_values.size else 0.0
                Cost_PD_value_RT[i, i_t] = daily_pd
                daily_ncd = c_NCD * np.maximum(np.max(p_GI.value) - M_Th_NCD[i], 0)
                Cost_NCD_value_RT[i, i_t] = daily_ncd
                Cost_TOU_value_RT[i, i_t] = c_e_TOU_AL_RT[0] * p_GI.value[0] * dt_h
                Cost_Opt_value_RT[i, i_t] = Cost_PD_value_RT[i, i_t] + Cost_NCD_value_RT[i, i_t] + Cost_TOU_value_RT[i, i_t] - Revenue_WM_value_RT[i, i_t] - Revenue_EV_value_RT[i, i_t]
                EnergyDemand_Table_Opt_value = np.asarray(EnergyDemand_Table_Opt.value, dtype=float)
                EnergyDemand_Table_Opt_value[EnergyDemand_Table_Opt_value <= 1e-4] = 0.0
                # 🌟 结合 Shrinking 逻辑的终极修正版
                # 1. 提取预编译模型中的完整名单，并强制转换为字符串，防范 KeyError
                model_ev_cols = [str(c) for c in model_i['ev_cols_full']]
                # 2. 将优化结果构建为一个全量 DataFrame（类似你 Shrinking 的做法）
                df_opt_full = pd.DataFrame(EnergyDemand_Table_Opt_value, columns=model_ev_cols)
                # 3. 提取当前实际在场的 Tesla 和 Non-Tesla 名单，并强转为字符串
                arrived_tesla = [str(c) for c in Car_table_RT[1][i].columns]
                arrived_nontesla = [str(c) for c in Car_table_RT[2][i].columns]
                # 4. 完美切片：点名提取在场车辆，且因为是 Rolling，我们只取第一步 [0, :] 喂给 Execution
                EnergyDemand_Table_Opt_cap[1][i] = df_opt_full.reindex(columns=arrived_tesla, fill_value=0.0).values[0, :]
                EnergyDemand_Table_Opt_cap[2][i] = df_opt_full.reindex(columns=arrived_nontesla, fill_value=0.0).values[0, :]
                # 总功率仍需计算全时段的 sum，用于给后面的追踪和绘图提供数据
                EnergyDemand_Step_Opt_cap[0][i] = EnergyDemand_Table_Opt_value.sum(axis=1)
                if i == 0:
                    track = np.array(EnergyDemand_Step_Opt_cap[0][0]) # kWh
                if i_t == 0:
                    dispatch_t0[i] = p_GI.value
                t3 = time.time()
                time_extract_results += (t3 - t2)
            # end of for i in range(len(Cases))
            #################################################################################
            # EXECUTION
            #################################################################################
            # Dispatch to all the arrived cars for both Type 1 and 2
            for i in range(len(Cases)):
                for j in np.array(range(2))+1:
                    if Count[j][i] == 0:
                        continue
                    ed_row_j = np.asarray(EnergyDemand_Table_Opt_cap[j][i], dtype=float)
                    if i == 0:
                        Dispatch[j][i].iloc[i_t, :Count[j][i]] = np.minimum(ed_row_j[:Count[j][i]], np.array(SessionkWh_nEta[j][0].iloc[0, :Count[j][i]]))
                    else:
                        Dispatch[j][i].iloc[i_t, :Count[j][i]] = np.minimum(ed_row_j[:Count[j][i]], np.array(UpperBound[j].iloc[0, :Count[j][i]]))  
                    # Update the Session left
                    SessionkWh_nEta[j][i].iloc[0, :Count[j][i]] = np.array(SessionkWh_nEta[j][i].iloc[0, :Count[j][i]]) - np.array(Dispatch[j][i].iloc[i_t, :Count[j][i]])
                    #sometimes SessionkWh_nEta = -1*10^-8 because MPC output is capped with 0.0001                                     
                    SessionkWh_nEta[j][i][SessionkWh_nEta[j][i] <= 0.0001] = 0                            
                    if i==1:
                            UpperBound[j].iloc[0, :Count[j][0]] = np.array(UpperBound[j].iloc[0, :Count[j][0]]) - np.array(Dispatch[j][1].iloc[i_t, :Count[j][0]])
                            UpperBound[j][UpperBound[j] <= 0.0001] = 0
                d1_rt = float(Dispatch[1][i].iloc[i_t, :Count[1][i]].sum())
                d2_rt = float(Dispatch[2][i].iloc[i_t, :Count[2][i]].sum())
                SessionkWh_nEta[0][i] = pd.concat([SessionkWh_nEta[1][i], SessionkWh_nEta[2][i]], axis=1)
                if i == 1:
                    UpperBound[0] = pd.concat([UpperBound[1], UpperBound[2]], axis=1)                         
                p_gi_curr = float((d1_rt + d2_rt) / dt_h + p_BESS_value_RT[i, i_t] + p_load_RT[0] - p_PV_RT[0])
                M_Th_NCD[i] = max(M_Th_NCD[i], p_gi_curr)
                if get_sdge_on_peak_mask([TheDate_Day0 + timedelta(minutes=i_t * dt_m_EV)])[0] > 0.5:
                    M_Th_PD[i] = max(M_Th_PD[i], p_gi_curr)
                M_Th_NCD_list_96[i] = M_Th_NCD_list_96[i] + [M_Th_NCD[i]]
                M_Th_PD_list_96[i] = M_Th_PD_list_96[i] + [M_Th_PD[i]]
                D_Th_NCD_96[i] = D_Th_NCD_96[i] + [M_Th_NCD[i]]
                D_Th_PD_96[i] = D_Th_PD_96[i] + [M_Th_PD[i]]
                if i_t == 95:
                    M_Th_NCD_list[i] = M_Th_NCD_list[i] + [M_Th_NCD[i]]
                    M_Th_PD_list[i] = M_Th_PD_list[i] + [M_Th_PD[i]]
            if SAVE_STEP_DEBUG:
                Dispatch[0][1] = pd.concat([Dispatch[1][1], Dispatch[2][1]], axis=1)
                var_list = [pd.DataFrame(EnergyDemand_Step_Opt_cap[0][1]).T,
                            UpperBound[0], 
                            SessionkWh_nEta[0][1], 
                            pd.DataFrame(Dispatch[0][1].iloc[i_t, :]).T]
                name_list = ['MPC', 'UpperBound', 'Session_nEta', 'Dispatch_withoutBESS']
                for var in range(len(var_list)):
                    dir_Output = os.path.join(VERSION_DIR, 'Dispatch', '1_'+name_list[var]+'_'+WM_Mode+'.csv')
                    append_df_to_csv(var_list[var], dir_Output, index=True)
            # end of for i in range(len(Cases)) 
        # end of for i_t in range(96)
        H = int(24 / dt_h)  # restore full-day length for daily output tables
        t4 = time.time()
        for i in range(len(Cases)):
            Dispatch[0][i] = pd.concat([Dispatch[1][i], Dispatch[2][i]], axis=1)
        executed_ev_kW = np.vstack([
            np.asarray(Dispatch[0][i].sum(axis=1), dtype=float) / dt_h
            for i in range(len(Cases))
        ])
        native_meter_profile_kW = np.asarray(p_load_DA, dtype=float) - np.asarray(p_PV_DA, dtype=float)
        execution_meter_error_kW = float(np.max(np.abs(
            p_GI_value_RT - (native_meter_profile_kW[None, :] + executed_ev_kW + p_BESS_value_RT)
        )))
        if execution_meter_error_kW > 1e-3:
            raise ValueError(
                f'Executed-dispatch/meter mismatch: {execution_meter_error_kW:.6g} kW. '
                'Check EV-ID mapping between the forecast optimization table and arrived vehicles.'
            )
        if SERVICE_LEVEL_MIN < 1.0:
            _service_request = (SessionkWh_table_real[0][0].iloc[0].astype(float) if not SessionkWh_table_real[0][0].empty else pd.Series(dtype=float))
            _service_delivered = Dispatch[0][0].sum(axis=0).reindex(_service_request.index, fill_value=0.0).astype(float)
            _service_shortfall = np.maximum(SERVICE_LEVEL_MIN * _service_request - _service_delivered, 0.0)
            _service_excess = np.maximum(_service_delivered - _service_request, 0.0)
            if float(_service_shortfall.max()) > 1e-3 or float(_service_excess.max()) > 1e-3:
                _bad_service = pd.DataFrame({'requested': _service_request, 'delivered': _service_delivered, 'minimum_shortfall': _service_shortfall, 'upper_excess': _service_excess})
                raise ValueError('Executed EV session service violation: ' + _bad_service.loc[(_service_shortfall > 1e-3) | (_service_excess > 1e-3)].to_string())
            _service_validation = pd.DataFrame({
                'session': _service_request.index, 'requested_kWh': _service_request.values,
                'minimum_service_fraction': SERVICE_LEVEL_MIN,
                'minimum_kWh': SERVICE_LEVEL_MIN * _service_request.values,
                'delivered_kWh': _service_delivered.values,
                'unserved_kWh': np.maximum(_service_request.values - _service_delivered.values, 0.0),
                'actual_service_fraction': np.divide(_service_delivered.values, _service_request.values, out=np.ones(len(_service_request)), where=_service_request.values > 0),
                'driver_revenue_USD': 0.4 * _service_delivered.values,
            })
            os.makedirs(os.path.join(VERSION_DIR, 'Dispatch'), exist_ok=True)
            _service_validation.to_csv(os.path.join(VERSION_DIR, 'Dispatch', TheDate_Day0.strftime('%Y%m%d') + '_service_validation.csv'), index=False)
            # Report EV revenue from executed energy rather than requested energy.
            _old_ev_revenue = Revenue_EV_value_RT.copy()
            Revenue_EV_value_RT[:] = 0.4 * executed_ev_kW * dt_h
            Cost_Opt_value_RT[:] += _old_ev_revenue - Revenue_EV_value_RT
            p_EV_value_RT[:] = executed_ev_kW
        real_session_total_base = float(np.asarray(SessionkWh_table_real[0][0], dtype=float).sum())
        dispatched_total_base = float(Dispatch[0][0].sum().sum())
        if SERVICE_LEVEL_MIN == 1.0 and not np.isclose(dispatched_total_base, real_session_total_base, atol=1e-4):
            print(
                "EV dispatch energy mismatch (base case): "
                f"dispatched={dispatched_total_base:.6f} kWh, "
                f"real_session={real_session_total_base:.6f} kWh, "
                f"diff={dispatched_total_base - real_session_total_base:.6f} kWh"
            )
            sys.exit()
        #################################################################################
        # Save daily results
        #################################################################################
        for i in range(len(Cases)):
            Solver_Outputs_RT[i]['Cost_Opt'] = Cost_Opt_value_RT[i].flatten()
            Solver_Outputs_RT[i]['penalty_BESS'] = penalty_BESS_value_RT[i].flatten()
            Solver_Outputs_RT[i]['penalty_WM'] = penalty_WM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Revenue_WM'] = Revenue_WM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_BESS'] = p_ch_BESS_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_BESS'] = p_dch_BESS_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_BESS_WM'] = p_ch_BESS_WM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_BESS_WM'] = p_dch_BESS_WM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_BESS_NWM'] = p_ch_BESS_NWM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_BESS_NWM'] = p_dch_BESS_NWM_value_RT[i].flatten()
            # Backward-compatible output keys: p_ch_EV/p_dch_EV now store the EV WM channel.
            Solver_Outputs_RT[i]['p_ch_EV'] = p_ch_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_EV'] = p_dch_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_EV_WM'] = p_ch_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_EV_WM'] = p_dch_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_EV_NWM'] = p_ch_EV_NWM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_EV_NWM'] = p_dch_EV_NWM_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_ch_net'] = p_ch_net_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_dch_net'] = p_dch_net_value_RT[i].flatten()
            Solver_Outputs_RT[i]['soc_BESS'] = soc_BESS_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_BESS'] = p_BESS_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_RU_DA'] = c_RU_DA_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_RD_DA'] = c_RD_DA_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_SP_DA'] = c_SP_DA_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_NSP_DA'] = c_NSP_DA_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_DA'] = p_DA_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_RU_RT'] = c_RU_RT_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_RD_RT'] = c_RD_RT_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_SP_RT'] = c_SP_RT_value_RT[i].flatten()
            Solver_Outputs_RT[i]['c_NSP_RT'] = c_NSP_RT_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_RT'] = p_RT_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_GI'] = p_GI_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_EV'] = p_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Cost_PD'] = Cost_PD_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Cost_NCD'] = Cost_NCD_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Cost_TOU'] = Cost_TOU_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Revenue_EV'] = Revenue_EV_value_RT[i].flatten()
            Solver_Outputs_RT[i]['Baseline(kW)'] = Baseline_96[i].flatten() / dt_h
            Solver_Outputs_RT[i]['P_EV_max'] = p_EV_max_value_RT[i].flatten()
            Solver_Outputs_RT[i]['p_up_bound(kW)'] = P_BESS_max + Baseline_96[i].flatten() / dt_h
            Solver_Outputs_RT[i]['p_down_bound(kW)'] = np.maximum(P_BESS_max - Baseline_96[i].flatten() / dt_h + p_EV_max_value_RT[i].flatten(), 0.0)
        Solver_Outputs_RT_Base = Solver_Outputs_RT[0]  # 100% eta for implementation
        Solver_Outputs_RT_Case1 = Solver_Outputs_RT[1] if len(Cases) > 1 else None
        # Save the 96 actually executed first-step decisions. Opt_RT_t0 in the
        # implementation CSV is the day-start plan and must not be used as the
        # executed RT trajectory in validation figures.
        _trace = Solver_Outputs_RT_Base
        _trace_p_da = np.asarray(_trace['p_DA'], dtype=float).reshape(-1)
        _trace_p_rt = np.asarray(_trace['p_RT'], dtype=float).reshape(-1)
        _trace_p_actual = _trace_p_da + _trace_p_rt
        _trace_baseline = np.asarray(_trace['Baseline(kW)'], dtype=float).reshape(-1)
        _trace_p_ev_max = np.asarray(_trace['P_EV_max'], dtype=float).reshape(-1)
        _trace_p_ctrl = _trace_p_actual
        _trace_p_up_ctrl = P_BESS_max + _trace_baseline
        _trace_p_down_ctrl = np.maximum(P_BESS_max - _trace_baseline + _trace_p_ev_max, 0.0)
        # Panel bounds are the physical EV+BESS wholesale capability limits.
        # Passive building/PV are visible at the retail meter but cannot shift them.
        _trace_p_up = _trace_p_up_ctrl
        _trace_p_down = _trace_p_down_ctrl
        _trace_ru_actual = np.asarray(_trace['c_RU_DA'], dtype=float) + np.asarray(_trace['c_RU_RT'], dtype=float)
        _trace_rd_actual = np.asarray(_trace['c_RD_DA'], dtype=float) + np.asarray(_trace['c_RD_RT'], dtype=float)
        _trace_sp_actual = np.asarray(_trace['c_SP_DA'], dtype=float) + np.asarray(_trace['c_SP_RT'], dtype=float)
        _trace_nsp_actual = np.asarray(_trace['c_NSP_DA'], dtype=float) + np.asarray(_trace['c_NSP_RT'], dtype=float)
        _trace_up_slack = _trace_p_up_ctrl - (np.maximum(_trace_p_ctrl, 0.0) + _trace_ru_actual + _trace_sp_actual + _trace_nsp_actual)
        _trace_down_slack = _trace_p_down_ctrl - (np.maximum(-_trace_p_ctrl, 0.0) + _trace_rd_actual)
        _trace_df = pd.DataFrame({
            'Interval start': pd.to_datetime(TimeSeries_real[0]),
            'p_EV_kW': np.asarray(_trace['p_EV'], dtype=float),
            'p_BESS_kW': np.asarray(_trace['p_BESS'], dtype=float),
            'p_load_kW': np.asarray(p_load_DA, dtype=float),
            'p_PV_kW': np.asarray(p_PV_DA, dtype=float),
            'p_GI_kW': np.asarray(_trace['p_GI'], dtype=float),
            'passive_WM_position_kW': np.zeros_like(_trace_p_actual),
            'SOC': np.asarray(_trace['soc_BESS'], dtype=float),
            'p_DA_kW': _trace_p_da,
            'p_RT_deviation_kW': _trace_p_rt,
            'p_actual_kW': _trace_p_actual,
            'p_controlled_residual_kW': _trace_p_ctrl,
            'c_RU_DA_kW': np.asarray(_trace['c_RU_DA'], dtype=float),
            'c_RU_RT_kW': np.asarray(_trace['c_RU_RT'], dtype=float),
            'c_RU_actual_kW': _trace_ru_actual,
            'c_RD_DA_kW': np.asarray(_trace['c_RD_DA'], dtype=float),
            'c_RD_RT_kW': np.asarray(_trace['c_RD_RT'], dtype=float),
            'c_RD_actual_kW': _trace_rd_actual,
            'c_SP_DA_kW': np.asarray(_trace['c_SP_DA'], dtype=float),
            'c_SP_RT_kW': np.asarray(_trace['c_SP_RT'], dtype=float),
            'c_SP_actual_kW': _trace_sp_actual,
            'c_NSP_DA_kW': np.asarray(_trace['c_NSP_DA'], dtype=float),
            'c_NSP_RT_kW': np.asarray(_trace['c_NSP_RT'], dtype=float),
            'c_NSP_actual_kW': _trace_nsp_actual,
            'baseline_kW': _trace_baseline,
            'P_EV_max_kW': _trace_p_ev_max,
            'p_up_bound_kW': _trace_p_up,
            'p_down_bound_kW': _trace_p_down,
            'meter_balance_residual_kW': np.asarray(_trace['p_GI'], dtype=float) - (np.asarray(p_load_DA, dtype=float) + np.asarray(_trace['p_EV'], dtype=float) + np.asarray(_trace['p_BESS'], dtype=float) - np.asarray(p_PV_DA, dtype=float)),
            'actual_up_capability_slack_kW': _trace_up_slack,
            'actual_down_capability_slack_kW': _trace_down_slack,
            'LMP_DA_$/kWh': np.asarray(Bid_Pr_DA, dtype=float),
            'LMP_RT_$/kWh': np.asarray(Bid_Pr_RT, dtype=float),
            'TOU_$/kWh': np.asarray(c_e_TOU_AL, dtype=float),
        })
        _trace_root = Path(VERSION_DIR) / 'Validation_Traces' / f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{WM_Mode}'
        _trace_root.mkdir(parents=True, exist_ok=True)
        _trace_path = _trace_root / f"rolling_validation_trace_{H_Start_RT:%Y%m%d}.csv"
        _trace_df.to_csv(_trace_path, index=False, float_format='%.10g')
        # Update the SOC for the next day
        for i in range(len(Cases)):
            SOC_BESS_daily_end[i] = Solver_Outputs_RT[i]['soc_BESS'][-1]
            if not np.isclose(SOC_BESS_daily_end[i], SOC_BESS_last_step[i], atol=1e-5):
                print("SOC continuity error")
                sys.exit()
            if is_last_run_day and (not np.isclose(SOC_BESS_daily_end[i], 0.5, atol=1e-4)):
                print("Final-day SOC terminal error: SOC must return to 0.5")
                sys.exit()
        # print(f"Day {Day} end SOC by case: {[float(SOC_BESS_daily_end[k]) for k in range(len(Cases))]}")
        M_ED_V0G = M_ED_V0G + [sum(EnergyDemand_Step_V0G_TheDates[0])]       
        M_ED_V1G = M_ED_V1G + [sum(EnergyDemand_Step_V1G_TheDates[0])] 
        M_ED_Opt_DA = M_ED_Opt_DA + [sum(p_GI_DA * dt_h)]
        M_ED_Opt_t0 = M_ED_Opt_t0 + [sum(dispatch_t0[0] * dt_h)] 
        M_ED_Opt_base = M_ED_Opt_base + [sum(Dispatch[0][0].sum(axis=1) + p_BESS_value_RT[0, :] * dt_h)]
        if RUN_CASE1:
            M_ED_Opt_case1 = M_ED_Opt_case1 + [sum(Dispatch[0][1].sum(axis=1) + p_BESS_value_RT[1, :] * dt_h)]
        #################################################################################
        # Save dispatch & baseline on a daily basis
        #################################################################################
        # TODO: do i need t0 dispatch and baseline DA here? this is just for next day initialization, and the results analysis is based on the Solver_Outputs df
        D_all = pd.DataFrame({'Interval start': pd.Series(TimeSeries_real[0]),
                              'V0G [kWh]': EnergyDemand_Step_V0G_TheDates[0],
                              'V1G_real [kWh]': EnergyDemand_Step_V1G_TheDates[0],
                              'Opt_DA [kWh]': p_GI_DA * dt_h,
                              'Opt_RT_t0 [kWh]': dispatch_t0[0] * dt_h}) # NOTE: V0G and V1G here are saved without BESS and DR/WM
        case_tables = []
        for i, case in enumerate(Cases):
            case_tables.append(
                pd.DataFrame({
                    # EV-only executed dispatch for baseline learning. Do not include BESS here;
                    # BESS/grid-meter effects are accounted in solver financial outputs.
                    f'{case} [kWh]': Dispatch[0][i].sum(axis=1),
                    f'{case}_withBESS [kWh]': Dispatch[0][i].sum(axis=1) + p_BESS_value_RT[i, :] * dt_h,
                    f'{case}_meter [kWh]': (p_load_DA + Dispatch[0][i].sum(axis=1) / dt_h + p_BESS_value_RT[i, :] - p_PV_DA) * dt_h,
                    'p_load [kW]': p_load_DA,
                    'p_PV [kW]': p_PV_DA,
                    f'NCD_{case}': D_Th_NCD_96[i],
                    f'PD_{case}': D_Th_PD_96[i],
                    f'baseline_{case} [kWh]': Baseline_96[i].reshape(96),
                    f'event hour_{case}': EventHour_96[i].reshape(96)
                })
            )
        lmp_tables = [
            pd.DataFrame({'LMP_DA': Bid_Pr_DA}),
            pd.DataFrame({'LMP_RT': Bid_Pr_RT}),
        ]
        D_all = pd.concat([D_all, *case_tables, *lmp_tables], axis=1)
        # Keep physical interval series untouched. DA, RT snapshot, and executed
        # base-load series use different accounting scopes, so do not reconcile
        # their totals by injecting residual energy into the last interval.
        total_energies = {
            'Opt_DA [kWh]': float(D_all['Opt_DA [kWh]'].sum()),
            'Opt_RT_t0 [kWh]': float(D_all['Opt_RT_t0 [kWh]'].sum()),
            f'{Cases[0]} [kWh]': float(D_all[f'{Cases[0]} [kWh]'].sum()),
        }
        dir_Output = os.path.join(VERSION_DIR, 'Dispatch', H_Start_RT.strftime("%Y") +'_' +Fc_SessionkWh+'_'+Fc_NumbEV+'_'+Fc_AtArrival+'_'+WM_Mode  +'_implementation.csv')
        append_df_to_csv(D_all, dir_Output, index=False)
        #################################################################################
        # Save forecasted and real SessionkWh for validation
        #################################################################################
        validation_blocks = [
            ('Fc EVs', pd.DataFrame(Car_table_fc_fix[0][0].columns).T),
            ('Fc SessionkWh', SessionkWh_table_fc_fix[0][0].T.reset_index(drop=True).T),
        ]
        if Fc_AtArrival == 'MLatArrival':
            validation_blocks.append(('ML SessionkWh', pd.DataFrame(SessionkWh_RT_ML[0][0]).T))
        validation_blocks += [
            ('Real EVs', pd.DataFrame(pd.concat([Car_table_real[1][0], Car_table_real[2][0]], axis=1).columns).T),
            ('Real SessionkWh', pd.concat([SessionkWh_table_real[1][0], SessionkWh_table_real[2][0]], axis=1).T.reset_index(drop=True).T),
            ('Dispatched_base_withoutBESS', pd.DataFrame((Dispatch[0][0].sum(axis=0)).reset_index(drop=True)).T),
        ]
        if RUN_CASE1:
            validation_blocks += [
                ('Real Eta_min', pd.concat([Eta_min_table_real[1][1], Eta_min_table_real[2][1]], axis=1).T.reset_index(drop=True).T),
                ('Dispatched_Eta', Eta_table_RT[0][1].T.reset_index(drop=True).T),
                ('Dispatched_case1_withoutBESS', pd.DataFrame((Dispatch[0][1].sum(axis=0)).reset_index(drop=True)).T),
            ]
        List = [name for name, _ in validation_blocks]
        A = pd.concat([block for _, block in validation_blocks])
        A = pd.concat([A.reset_index(drop=True), pd.DataFrame(List, columns=[H_Start_RT.strftime("%Y%m%d")])], axis=1).set_index(H_Start_RT.strftime("%Y%m%d"))
        A = A.T
        A.index = [H_Start_RT.strftime("%Y%m%d")] * len(A) 
        A.index.name = "Date"
        dir_Output_summary = os.path.join(VERSION_DIR, 'Dispatch', H_Start_RT.strftime("%Y") +'_' +Fc_SessionkWh+'_'+Fc_NumbEV+'_'+Fc_AtArrival+'_'+WM_Mode +'_daily_summary.csv')
        _trim_existing_csv_before_run_start(dir_Output_summary, TheDate_Day0)
        append_df_to_csv(A, dir_Output_summary, index=True)
        #################################################################################
        # Save WM profit table and the reconciliation check in output folder
        #################################################################################
        # Save interval tables for Base case only
        if Enable_WM:
            p_DA_arr = np.array(Solver_Outputs_RT_Base['p_DA']).flatten() # 注意这里，DA处是 Solver_Outputs_DA，RT处是 Solver_Outputs_RT_Base
            p_RT_arr = np.array(Solver_Outputs_RT_Base['p_RT']).flatten()
            c_RU_DA_arr = np.array(Solver_Outputs_RT_Base['c_RU_DA']).flatten()
            c_RU_RT_arr = np.array(Solver_Outputs_RT_Base['c_RU_RT']).flatten()
            c_RD_DA_arr = np.array(Solver_Outputs_RT_Base['c_RD_DA']).flatten()
            c_RD_RT_arr = np.array(Solver_Outputs_RT_Base['c_RD_RT']).flatten()
            c_SP_DA_arr = np.array(Solver_Outputs_RT_Base['c_SP_DA']).flatten()
            c_SP_RT_arr = np.array(Solver_Outputs_RT_Base['c_SP_RT']).flatten()
            c_NSP_DA_arr = np.array(Solver_Outputs_RT_Base['c_NSP_DA']).flatten()
            c_NSP_RT_arr = np.array(Solver_Outputs_RT_Base['c_NSP_RT']).flatten()
            wm_profit_table = pd.DataFrame({
                'p_DA_profit': dt_h * bid_pr_da_i * p_DA_arr, 
                'p_RT_profit': dt_h * bid_pr_rt_i * p_RT_arr,
                # 向上调节 RU
                'c_RU_DA_energy': dt_h * (alpha_RU * bid_pr_rt_i) * c_RU_DA_arr,
                'c_RU_DA_capacity': dt_h * as_ru_da_i * c_RU_DA_arr,
                'c_RU_RT_energy': dt_h * (alpha_RU * bid_pr_rt_i) * c_RU_RT_arr,
                'c_RU_RT_capacity': dt_h * as_ru_rt_i * c_RU_RT_arr,
                # 向下调节 RD (注意公式中的负号)
                'c_RD_DA_energy': dt_h * (-alpha_RD * bid_pr_rt_i) * c_RD_DA_arr,
                'c_RD_DA_capacity': dt_h * as_rd_da_i * c_RD_DA_arr,
                'c_RD_RT_energy': dt_h * (-alpha_RD * bid_pr_rt_i) * c_RD_RT_arr,
                'c_RD_RT_capacity': dt_h * as_rd_rt_i * c_RD_RT_arr,
                # 旋转备用 SP
                'c_SP_DA_energy': dt_h * (alpha_SP * bid_pr_rt_i) * c_SP_DA_arr,
                'c_SP_DA_capacity': dt_h * as_sp_da_i * c_SP_DA_arr,
                'c_SP_RT_energy': dt_h * (alpha_SP * bid_pr_rt_i) * c_SP_RT_arr,
                'c_SP_RT_capacity': dt_h * as_sp_rt_i * c_SP_RT_arr,
                # 非旋转备用 NSP
                'c_NSP_DA_energy': dt_h * (alpha_NSP * bid_pr_rt_i) * c_NSP_DA_arr,
                'c_NSP_DA_capacity': dt_h * as_nsp_da_i * c_NSP_DA_arr,
                'c_NSP_RT_energy': dt_h * (alpha_NSP * bid_pr_rt_i) * c_NSP_RT_arr,
                'c_NSP_RT_capacity': dt_h * as_nsp_rt_i * c_NSP_RT_arr,
            })
            wm_tou_table = pd.DataFrame({
                'p_DA_TOU_Cost': -dt_h * c_e_TOU_AL * p_DA_arr,
                'p_RT_TOU_Cost': -dt_h * c_e_TOU_AL * p_RT_arr,
                'c_RU_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_RU * c_RU_DA_arr,
                'c_RU_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_RU * c_RU_RT_arr,
                'c_RD_DA_TOU_Cost': dt_h * c_e_TOU_AL * alpha_RD * c_RD_DA_arr,
                'c_RD_RT_TOU_Cost': dt_h * c_e_TOU_AL * alpha_RD * c_RD_RT_arr,
                'c_SP_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_SP * c_SP_DA_arr,
                'c_SP_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_SP * c_SP_RT_arr,
                'c_NSP_DA_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_NSP * c_NSP_DA_arr,
                'c_NSP_RT_TOU_Cost': -dt_h * c_e_TOU_AL * alpha_NSP * c_NSP_RT_arr,
            })
        else:
            cols = [
                'p_DA_profit', 'p_RT_profit',
                'c_RU_DA_energy', 'c_RU_DA_capacity', 'c_RU_RT_energy', 'c_RU_RT_capacity',
                'c_RD_DA_energy', 'c_RD_DA_capacity', 'c_RD_RT_energy', 'c_RD_RT_capacity',
                'c_SP_DA_energy', 'c_SP_DA_capacity', 'c_SP_RT_energy', 'c_SP_RT_capacity',
                'c_NSP_DA_energy', 'c_NSP_DA_capacity', 'c_NSP_RT_energy', 'c_NSP_RT_capacity'
            ]
            wm_profit_table = pd.DataFrame(np.zeros((H, len(cols))), columns=cols)
            wm_tou_table = pd.DataFrame(np.zeros((H, 10)), columns=[
                'p_DA_TOU_Cost', 'p_RT_TOU_Cost',
                'c_RU_DA_TOU_Cost', 'c_RU_RT_TOU_Cost',
                'c_RD_DA_TOU_Cost', 'c_RD_RT_TOU_Cost',
                'c_SP_DA_TOU_Cost', 'c_SP_RT_TOU_Cost',
                'c_NSP_DA_TOU_Cost', 'c_NSP_RT_TOU_Cost'
            ])
        wm_profit_table['profit_sum_all_products'] = wm_profit_table.sum(axis=1)
        wm_tou_table['TOU_Cost_sum_all_products'] = wm_tou_table.sum(axis=1)
        wm_profit_total_from_table = float(wm_profit_table['profit_sum_all_products'].sum())
        wm_profit_check_diff = float(abs(wm_profit_total_from_table - np.sum(Solver_Outputs_RT_Base['Revenue_WM'])))
        Solver_Outputs_RT_Base['WM_Profit_Table'] = wm_profit_table
        Solver_Outputs_RT_Base['WM_Profit_Total_From_Table'] = wm_profit_total_from_table
        Solver_Outputs_RT_Base['WM_TOU_Table'] = wm_tou_table
        Solver_Outputs_RT_Base['WM_TOU_Total_From_Table'] = float(wm_tou_table['TOU_Cost_sum_all_products'].sum())
        if wm_profit_check_diff > 1e-3:
            print(f"DA WM profit check -> table_sum={wm_profit_total_from_table:.6f}, Revenue_WM={Solver_Outputs_RT_Base['Revenue_WM'].sum():.6f}, abs_diff={wm_profit_check_diff:.6e}")
        # =========================================================================
        # Save daily WM profit table and the reconciliation check in output folder
        run_tag_daily = H_Start_RT.strftime("%Y") + '_' + Fc_SessionkWh + '_' + Fc_NumbEV + '_' + Fc_AtArrival + '_' + WM_Mode
        profit_dir = Path(VERSION_DIR) / f"Plots/Solver_{TARGET_SAVE}_Choices" / run_tag_daily / pd.Timestamp(TheDate_Day0).strftime("%Y%m%d")
        profit_dir.mkdir(parents=True, exist_ok=True)
        if TARGET_SAVE == 'DA':
            Solver_Outputs_target = Solver_Outputs_DA
        else:
            Solver_Outputs_target = Solver_Outputs_RT_Base
        wm_profit_table_to_save = Solver_Outputs_target['WM_Profit_Table'].copy()
        wm_tou_table_to_save = Solver_Outputs_target['WM_TOU_Table'].copy()
        # =========================================================================
        # --- EV vs BESS 收益精确分配 (向上与向下严格解耦修正版) --- 
        # =========================================================================
        # 1. 动态获取当前表格的实际行数
        H_len = len(wm_profit_table_to_save)
        # 2. 提取物理出力，严格解耦放电(Up)与充电(Dn)，杜绝抵消漏洞
        p_up_bess = np.array(Solver_Outputs_target['p_dch_BESS_WM']).flatten()[:H_len]
        p_up_ev   = np.array(Solver_Outputs_target['p_dch_EV']).flatten()[:H_len]
        p_up_total = p_up_bess + p_up_ev
        p_dn_bess = np.array(Solver_Outputs_target['p_ch_BESS_WM']).flatten()[:H_len]
        p_dn_ev   = np.array(Solver_Outputs_target['p_ch_EV']).flatten()[:H_len]
        p_dn_total = p_dn_bess + p_dn_ev
        # 3. 动态容量边界（已由你在上一步正确计算）
        dynamic_P_BESS_max = np.full(H_len, P_BESS_max) 
        dynamic_B_EV = np.array(Solver_Outputs_target['Baseline(kW)']).flatten()[:H_len]
        dynamic_P_EV_max = np.array(Solver_Outputs_target['P_EV_max']).flatten()[:H_len]
        dynamic_up_max = dynamic_P_BESS_max + dynamic_B_EV
        dynamic_down_max = dynamic_P_BESS_max + dynamic_P_EV_max - dynamic_B_EV
        # =========================================================================
        # 4. 计算分配权重 (强制守恒修复版)
        # =========================================================================
        # 4b. 容量分配权重 (Capacity Weights) - 必须先算，作为保底
        w_cap_up_bess = np.divide(dynamic_P_BESS_max, dynamic_up_max, out=np.zeros_like(dynamic_up_max), where=dynamic_up_max>1e-6)
        w_cap_up_ev   = 1.0 - w_cap_up_bess
        w_cap_dn_bess = np.divide(dynamic_P_BESS_max, dynamic_down_max, out=np.zeros_like(dynamic_down_max), where=dynamic_down_max>1e-6)
        w_cap_dn_ev   = 1.0 - w_cap_dn_bess
        # =========================================================================
        # 4a. 能量权重：按代数净功率流占比 (支持对冲工况，守恒且无Warning)
        # =========================================================================
        # 分别计算 BESS 和 EV 的代数净功率流（放电为正，充电为负）
        p_net_bess = np.array(Solver_Outputs_target['p_dch_BESS_WM']).flatten()[:H_len] - np.array(Solver_Outputs_target['p_ch_BESS_WM']).flatten()[:H_len]
        p_net_ev   = np.array(Solver_Outputs_target['p_dch_EV']).flatten()[:H_len] - np.array(Solver_Outputs_target['p_ch_EV']).flatten()[:H_len]
        # 系统对外的总净功率流（等于两者代数相加，完美对应 p_dch_net - p_ch_net）
        p_net_system = p_net_bess + p_net_ev
        # 先计算原始的代数权重（允许破1或负数，用 where 规避系统不充不放时的除零错误）
        w_raw = np.divide(p_net_bess, p_net_system, out=w_cap_up_bess.copy(), where=np.abs(p_net_system) > 1e-3)
        # 核心：用你说的 min-max 逻辑将 BESS 强行锁死在 [0, 1] 之间
        w_energy_bess = np.clip(w_raw, 0.0, 1.0)
        w_energy_ev   = 1.0 - w_energy_bess
        # =========================================================================
        # 5. 执行拆分：将收益按【功能维度】与【物理方向】细分
        # =========================================================================
        cap_up_profit = (
            wm_profit_table_to_save['c_RU_DA_capacity'] + wm_profit_table_to_save['c_RU_RT_capacity'] + 
            wm_profit_table_to_save['c_SP_DA_capacity'] + wm_profit_table_to_save['c_SP_RT_capacity'] + 
            wm_profit_table_to_save['c_NSP_DA_capacity'] + wm_profit_table_to_save['c_NSP_RT_capacity']
        ).values
        cap_dn_profit = (
            wm_profit_table_to_save['c_RD_DA_capacity'] + wm_profit_table_to_save['c_RD_RT_capacity']
        ).values
        total_energy_profit = (
            wm_profit_table_to_save['p_DA_profit'] + wm_profit_table_to_save['p_RT_profit'] +
            wm_profit_table_to_save['c_RU_DA_energy'] + wm_profit_table_to_save['c_RU_RT_energy'] +
            wm_profit_table_to_save['c_SP_DA_energy'] + wm_profit_table_to_save['c_SP_RT_energy'] +
            wm_profit_table_to_save['c_NSP_DA_energy'] + wm_profit_table_to_save['c_NSP_RT_energy'] +
            wm_profit_table_to_save['c_RD_DA_energy'] + wm_profit_table_to_save['c_RD_RT_energy']
        ).values
        # Every wholesale dollar is attributable only to controllable EV+BESS.
        # Passive building/PV affect p_GI retail charges but have zero WM position.
        controlled_energy_profit = total_energy_profit
        # 汇总所有容量收益
        total_cap_profit = (
            wm_profit_table_to_save['c_RU_DA_capacity'] + wm_profit_table_to_save['c_RU_RT_capacity'] +
            wm_profit_table_to_save['c_SP_DA_capacity'] + wm_profit_table_to_save['c_SP_RT_capacity'] +
            wm_profit_table_to_save['c_NSP_DA_capacity'] + wm_profit_table_to_save['c_NSP_RT_capacity'] +
            wm_profit_table_to_save['c_RD_DA_capacity'] + wm_profit_table_to_save['c_RD_RT_capacity']
        ).values
        # =========================================================================
        # 6. 分配并计算最终结果
        # =========================================================================
        wm_profit_table_to_save['BESS_Energy_Profit'] = controlled_energy_profit * w_energy_bess
        wm_profit_table_to_save['EV_Energy_Profit']   = controlled_energy_profit * w_energy_ev
        wm_profit_table_to_save['BESS_Capacity_Profit'] = (cap_up_profit * w_cap_up_bess) + (cap_dn_profit * w_cap_dn_bess)
        wm_profit_table_to_save['EV_Capacity_Profit']   = (cap_up_profit * (1.0 - w_cap_up_bess)) + (cap_dn_profit * (1.0 - w_cap_dn_bess))
        wm_profit_table_to_save['BESS_Total_WM_Profit'] = wm_profit_table_to_save['BESS_Energy_Profit'] + wm_profit_table_to_save['BESS_Capacity_Profit']
        wm_profit_table_to_save['EV_Total_WM_Profit']   = wm_profit_table_to_save['EV_Energy_Profit'] + wm_profit_table_to_save['EV_Capacity_Profit']
        # 7. 最后再保存到 CSV
        wm_profit_check_separation_diff = float(abs(
            wm_profit_table_to_save['BESS_Total_WM_Profit'].sum()
            + wm_profit_table_to_save['EV_Total_WM_Profit'].sum()
            - wm_profit_total_from_table
        ))
        if wm_profit_check_separation_diff > 1e-3:
            print(f"WM profit separation check failed: original_total={wm_profit_total_from_table:.6f}, abs_diff={wm_profit_check_separation_diff:.6e}")
        wm_profit_table_to_save.insert(0, 'Interval start', pd.date_range(start=pd.Timestamp(TheDate_Day0), periods=len(wm_profit_table_to_save), freq=f"{max(1, int(round(dt_h * 60)))}min"))
        wm_profit_file = profit_dir / f"{TARGET_SAVE}_WM_profit_breakdown_{WM_Mode}.csv"
        wm_profit_table_to_save.to_csv(wm_profit_file, index=False, float_format='%.6f')
        wm_tou_table_to_save.insert(0, 'Interval start', pd.date_range(start=pd.Timestamp(TheDate_Day0), periods=len(wm_tou_table_to_save), freq=f"{max(1, int(round(dt_h * 60)))}min"))
        wm_tou_file = profit_dir / f"{TARGET_SAVE}_WM_TOU_breakdown_{WM_Mode}.csv"
        wm_tou_table_to_save.to_csv(wm_tou_file, index=False, float_format='%.2f')
        # Reproducible market audit tables with the price signal that creates each value.
        # These use the 96 executed first-step Rolling decisions, never the t0 look-ahead plan.
        _audit_time = pd.to_datetime(wm_profit_table_to_save['Interval start'])
        _product_inputs = {
            'Energy': {
                'q_DA': np.asarray(Solver_Outputs_target['p_DA'], dtype=float)[:H_len],
                'q_RT': np.asarray(Solver_Outputs_target['p_RT'], dtype=float)[:H_len],
                'alpha': np.ones(H_len), 'direction': 1.0,
                'cap_DA': np.zeros(H_len), 'cap_RT': np.zeros(H_len),
            },
            'RU': {'q_DA': np.asarray(Solver_Outputs_target['c_RU_DA'], dtype=float)[:H_len], 'q_RT': np.asarray(Solver_Outputs_target['c_RU_RT'], dtype=float)[:H_len], 'alpha': np.asarray(alpha_RU, dtype=float)[:H_len], 'direction': 1.0, 'cap_DA': np.asarray(as_ru_da_i, dtype=float)[:H_len], 'cap_RT': np.asarray(as_ru_rt_i, dtype=float)[:H_len]},
            'RD': {'q_DA': np.asarray(Solver_Outputs_target['c_RD_DA'], dtype=float)[:H_len], 'q_RT': np.asarray(Solver_Outputs_target['c_RD_RT'], dtype=float)[:H_len], 'alpha': np.asarray(alpha_RD, dtype=float)[:H_len], 'direction': -1.0, 'cap_DA': np.asarray(as_rd_da_i, dtype=float)[:H_len], 'cap_RT': np.asarray(as_rd_rt_i, dtype=float)[:H_len]},
            'SP': {'q_DA': np.asarray(Solver_Outputs_target['c_SP_DA'], dtype=float)[:H_len], 'q_RT': np.asarray(Solver_Outputs_target['c_SP_RT'], dtype=float)[:H_len], 'alpha': np.asarray(alpha_SP, dtype=float)[:H_len], 'direction': 1.0, 'cap_DA': np.asarray(as_sp_da_i, dtype=float)[:H_len], 'cap_RT': np.asarray(as_sp_rt_i, dtype=float)[:H_len]},
            'NSP': {'q_DA': np.asarray(Solver_Outputs_target['c_NSP_DA'], dtype=float)[:H_len], 'q_RT': np.asarray(Solver_Outputs_target['c_NSP_RT'], dtype=float)[:H_len], 'alpha': np.asarray(alpha_NSP, dtype=float)[:H_len], 'direction': 1.0, 'cap_DA': np.asarray(as_nsp_da_i, dtype=float)[:H_len], 'cap_RT': np.asarray(as_nsp_rt_i, dtype=float)[:H_len]},
        }
        _lmp_da = np.asarray(bid_pr_da_i, dtype=float)[:H_len]
        _lmp_rt = np.asarray(bid_pr_rt_i, dtype=float)[:H_len]
        _interval_audit_parts, _daily_audit_rows = [], []
        def _weighted_mean(values, weights):
            values = np.asarray(values, dtype=float); weights = np.asarray(weights, dtype=float)
            denom = float(np.sum(np.abs(weights)))
            return float(np.sum(values * np.abs(weights)) / denom) if denom > 1e-10 else 0.0
        for _product, _cfg in _product_inputs.items():
            _q_da = _cfg['q_DA']; _q_rt = _cfg['q_RT']; _q_actual = _q_da + _q_rt
            _alpha = _cfg['alpha']; _obligation = _cfg['direction'] * _alpha * _q_actual
            _da_energy_price = _lmp_da if _product == 'Energy' else _lmp_rt
            _rt_energy_price = _lmp_rt
            _da_energy_rev = dt_h * _da_energy_price * (_q_da if _product == 'Energy' else _cfg['direction'] * _alpha * _q_da)
            _rt_energy_rev = dt_h * _rt_energy_price * (_q_rt if _product == 'Energy' else _cfg['direction'] * _alpha * _q_rt)
            _da_cap_rev = dt_h * _cfg['cap_DA'] * _q_da
            _rt_cap_rev = dt_h * _cfg['cap_RT'] * _q_rt
            _total_rev = _da_energy_rev + _rt_energy_rev + _da_cap_rev + _rt_cap_rev
            _interval_audit_parts.append(pd.DataFrame({
                'Interval start': _audit_time, 'Product': _product,
                'DA quantity (kW)': _q_da, 'RT signed adjustment (kW)': _q_rt,
                'RT buy-back (kW)': np.maximum(-_q_rt, 0.0), 'RT add-on (kW)': np.maximum(_q_rt, 0.0),
                'Final physical position (kW)': _q_actual, 'Alpha': _alpha,
                'Signed energy obligation (kW)': _obligation,
                'DA LMP ($/kWh)': _lmp_da, 'RT LMP ($/kWh)': _lmp_rt,
                'DA AS capacity price ($/kWh)': _cfg['cap_DA'], 'RT AS capacity price ($/kWh)': _cfg['cap_RT'],
                'DA energy/activation revenue ($)': _da_energy_rev, 'RT energy/activation revenue ($)': _rt_energy_rev,
                'DA capacity revenue ($)': _da_cap_rev, 'RT capacity settlement ($)': _rt_cap_rev,
                'Total product revenue ($)': _total_rev,
            }))
            _daily_audit_rows.append({
                'Date': pd.Timestamp(TheDate_Day0).strftime('%Y-%m-%d'), 'Product': _product,
                'DA signed quantity (kWh or kW-h)': dt_h * float(np.sum(_q_da)),
                'RT signed adjustment (kWh or kW-h)': dt_h * float(np.sum(_q_rt)),
                'RT buy-back (kWh or kW-h)': dt_h * float(np.sum(np.maximum(-_q_rt, 0.0))),
                'RT add-on (kWh or kW-h)': dt_h * float(np.sum(np.maximum(_q_rt, 0.0))),
                'Final position (kWh or kW-h)': dt_h * float(np.sum(_q_actual)),
                'Activated energy obligation (kWh)': dt_h * float(np.sum(_obligation)),
                'Position-weighted alpha': _weighted_mean(_alpha, _q_actual),
                'DA energy LMP weighted ($/kWh; Energy only)': _weighted_mean(_lmp_da, _q_da) if _product == 'Energy' else np.nan,
                'RT energy/activation LMP weighted ($/kWh)': _weighted_mean(_lmp_rt, _q_rt if _product == 'Energy' else _obligation),
                'DA capacity price weighted by DA position ($/kWh)': _weighted_mean(_cfg['cap_DA'], _q_da),
                'RT capacity price weighted by RT position ($/kWh)': _weighted_mean(_cfg['cap_RT'], _q_rt),
                'DA energy/activation revenue ($)': float(np.sum(_da_energy_rev)),
                'RT energy/activation revenue ($)': float(np.sum(_rt_energy_rev)),
                'DA capacity revenue ($)': float(np.sum(_da_cap_rev)),
                'RT capacity settlement ($)': float(np.sum(_rt_cap_rev)),
                'Total WM revenue ($)': float(np.sum(_total_rev)),
                'Price interpretation': 'DA quantity x DA LMP + RT adjustment x RT LMP' if _product == 'Energy' else ('-alpha x final RD capacity x RT LMP + DA/RT capacity settlement' if _product == 'RD' else 'alpha x final capacity x RT LMP + DA/RT capacity settlement'),
            })
        _interval_market_audit = pd.concat(_interval_audit_parts, ignore_index=True).sort_values(['Interval start', 'Product'])
        _daily_market_audit = pd.DataFrame(_daily_audit_rows)
        _audit_revenue_residual = float(_interval_market_audit['Total product revenue ($)'].sum() - wm_profit_total_from_table)
        if abs(_audit_revenue_residual) > 1e-4:
            raise ValueError(f'Market audit revenue residual is {_audit_revenue_residual:.6g} dollars')
        _interval_market_audit.to_csv(profit_dir / f'{TARGET_SAVE}_market_interval_audit_with_prices.csv', index=False, float_format='%.8f')
        _daily_market_audit.to_csv(profit_dir / f'{TARGET_SAVE}_market_daily_summary_with_prices.csv', index=False, float_format='%.8f')
        _save_daily_6panel = _date_key_allowed(TheDate_Day0, globals().get('PLOT_DAILY_6PANEL_DATES', []))
        if _save_daily_6panel:
            daily_fig_files = plot_daily_solver_choice_figures(
                TheDate_Day0=TheDate_Day0,
                dt_h=dt_h,
                Solver_Outputs=Solver_Outputs_target,
                c_e_TOU_AL=c_e_TOU_AL,
                Bid_Pr_DA=Bid_Pr_DA,
                Bid_Pr_RT=Bid_Pr_RT,
                AS_Pr_RU_DA=AS_Pr_RU_DA,
                AS_Pr_RU_RT=AS_Pr_RU_RT,
                AS_Pr_RD_DA=AS_Pr_RD_DA,
                AS_Pr_RD_RT=AS_Pr_RD_RT,
                AS_Pr_SP_DA=AS_Pr_SP_DA,
                AS_Pr_SP_RT=AS_Pr_SP_RT,
                AS_Pr_NSP_DA=AS_Pr_NSP_DA,
                AS_Pr_NSP_RT=AS_Pr_NSP_RT,
                alpha_RU=alpha_RU,
                alpha_RD=alpha_RD,
                alpha_SP=alpha_SP,
                alpha_NSP=alpha_NSP,
                run_tag=run_tag_daily,
                M_Th_NCD = M_Th_NCD_prev,  # use previous day's threshold for fig(f)
                M_Th_PD = M_Th_PD_prev,
                P_BESS_max = P_BESS_max,
                P_EV_max = dynamic_P_EV_max,
                mpc_version_label = MPC_VERSION_LABEL,
                forecast_label = ('Perfect' if Fc_SessionkWh == 'PerfectSessionkWh' else 'Persistence'),
                threshold_case_idx = (DA_CASE_IDX if TARGET_SAVE == 'DA' else 0),
            )
        else:
            daily_fig_files = []
            print(f"Skipping 6-panel figure for {pd.Timestamp(TheDate_Day0).strftime('%Y%m%d')} (PLOT_DAILY_6PANEL_DATES={globals().get('PLOT_DAILY_6PANEL_DATES', [])})")
        '''
        cycle_den_rt = 2.0 * float(C_BESS) * float(SOC_BESS_max - SOC_BESS_min)
        cycle_base_rt = dt_h * float(np.sum(p_ch_BESS_value_RT[0, :] + p_dch_BESS_value_RT[0, :])) / cycle_den_rt if cycle_den_rt > 0 else np.nan
        cycle_case1_rt = dt_h * float(np.sum(p_ch_BESS_value_RT[1, :] + p_dch_BESS_value_RT[1, :])) / cycle_den_rt if cycle_den_rt > 0 else np.nan
        cycle_label = f"Base {cycle_base_rt:.3f}, Case1 {cycle_case1_rt:.3f}"
        stairplot_file = save_old_stairplot_beautified(
            H_Start_RT=H_Start_RT,
            TimeSeries_real=TimeSeries_real,
            dt_m_EV=dt_m_EV,
            dt_h=dt_h,
            EnergyDemand_Step_V0G_TheDates=EnergyDemand_Step_V0G_TheDates,
            EnergyDemand_Step_V1G_TheDates=EnergyDemand_Step_V1G_TheDates,
            p_GI_DA=p_GI_DA,
            dispatch_t0=dispatch_t0,
            Dispatch=Dispatch,
            p_BESS_value_RT=p_BESS_value_RT,
            Baseline_96=Baseline_96,
            EventHour_96=EventHour_96,
            D_Th_NCD_96=D_Th_NCD_96,
            D_Th_PD_96=D_Th_PD_96,
            Fc_SessionkWh=Fc_SessionkWh,
            Fc_NumbEV=Fc_NumbEV,
            Fc_AtArrival=Fc_AtArrival,
            WM_Mode=WM_Mode,
            cycle_label=cycle_label,
        )
        print(' -', stairplot_file)
        '''
        t5 = time.time()
        time_daily_save = t5 - t4
        '''
        print("\n" + "="*40)
        # print('RT MPC DPP compliant:', all(model['dpp_ok'] for model in rt_mpc_models))
        print("="*40)
        print(f"总耗时: {(time_prep_and_forecast + time_gurobi_solve + time_extract_results) + time_daily_save:.2f} 秒")
        print(f"   ├─ [非计算] 动态预测与参数赋值: {time_prep_and_forecast:.2f} 秒")
        print(f"   ├─ [纯计算] Gurobi 底层求解:  {time_gurobi_solve:.2f} 秒")
        print(f"   ├─ [非计算] 结果提取与保存:   {time_extract_results:.2f} 秒")
        print(f"   └─ [非计算] 每日结果保存:   {time_daily_save:.2f} 秒")
        print("="*40 + "\n")
        '''
    # end of daily loop
# end of monthly loop
end_time = time.time()
print("Main loop runtime: {:.2f} seconds".format(end_time - start_time))
PLOT_DAILY_6PANEL_DATES = globals().get('PLOT_DAILY_6PANEL_DATES', 'all')  # inherit runner/config selection
ANALYSIS_DAYS_CONFIG = None  # None uses RUN_DAYS_CONFIG for final financial plots
SAVE_FINAL_FINANCIAL_FIGURES = False  # main notebooks save CSVs; paper figures are made in paper_financial_comparison_plots.ipynb
def _date_key_allowed(date_like, selected_dates):
    """Return whether a daily expensive plot should be saved for date_like."""
    if selected_dates is None or selected_dates == 'all':
        return True
    if selected_dates == [] or selected_dates == () or selected_dates == set():
        return False
    target = pd.Timestamp(date_like).strftime('%Y%m%d')
    allowed = set()
    for item in selected_dates:
        s = str(item)
        if s.isdigit() and len(s) <= 2:
            allowed.add(f"{int(globals().get('year', 2025)):04d}{int(globals().get('month', 7)):02d}{int(s):02d}")
        else:
            allowed.add(pd.Timestamp(item).strftime('%Y%m%d'))
    return target in allowed


RUN_MAIN_LOOP_DIRECT=False: main loop block loaded without execution.


Main loop runtime: 0.01 seconds


In [ ]:
# Canonical site/resource/forecast runner for the SEGAN paper.
# It reuses the exact parameter, data, plotting, optimization, and financial
# cells above. No second copy of the MPC equations is maintained.
import hashlib
import json
import nbformat
import re
import time

CANONICAL_STUDY_ROOT = CANONICAL_RESULTS_ROOT
canonical_run_year = int(year)
canonical_run_month = 6 if CANONICAL_RUN_SCOPE == 'june' else int(CANONICAL_RUN_MONTH)
canonical_period_key = f'{canonical_run_year}-{canonical_run_month:02d}'
if CANONICAL_RUN_SCOPE == 'june':
    CANONICAL_PERIOD_ROOT = CANONICAL_STUDY_ROOT
elif CANONICAL_RUN_SCOPE == 'month':
    CANONICAL_PERIOD_ROOT = CANONICAL_STUDY_ROOT / 'monthly_runs' / canonical_period_key
else:
    CANONICAL_PERIOD_ROOT = CANONICAL_STUDY_ROOT / 'smoke_runs' / canonical_period_key
CANONICAL_MODEL_OUTPUT_ROOT = CANONICAL_PERIOD_ROOT / 'model_outputs'
CANONICAL_TABLE_DIR = CANONICAL_PERIOD_ROOT / 'tables'
CANONICAL_FIGURE_DIR = CANONICAL_PERIOD_ROOT / 'figures'
CANONICAL_BASELINE_DIR = CANONICAL_INPUT_ROOT / 'baseline_dispatch'
for directory in [CANONICAL_MODEL_OUTPUT_ROOT, CANONICAL_TABLE_DIR, CANONICAL_FIGURE_DIR, CANONICAL_BASELINE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if CANONICAL_SITE_CASE == 'Both':
    canonical_sites = list(CANONICAL_SITE_CONFIG)
else:
    canonical_sites = [CANONICAL_SITE_CASE]

all_forecasts = [
    ('PersistenceSessionkWh', 'PersistenceNumbEV', 'PerfectatArrival', 'Persistence'),
    ('PerfectSessionkWh', 'PerfectNumbEV', 'PerfectatArrival', 'Perfect'),
]
canonical_forecasts = [
    item for item in all_forecasts
    if CANONICAL_FORECAST_CASE == 'Both' or item[3] == CANONICAL_FORECAST_CASE
]

if CANONICAL_RESOURCE_CASE == 'Both':
    canonical_optimized_resources = ['EV_BESS', 'FULL_BTM']
    canonical_direct_resources = ['BUILDING_ONLY', 'BUILDING_PV']
elif CANONICAL_RESOURCE_CASE in CANONICAL_OPTIMIZED_RESOURCE_CASES:
    canonical_optimized_resources = [CANONICAL_RESOURCE_CASE]
    canonical_direct_resources = ['BUILDING_ONLY', 'BUILDING_PV'] if CANONICAL_CHAIN_START_MONTH else []
else:
    canonical_optimized_resources = []
    canonical_direct_resources = [CANONICAL_RESOURCE_CASE]

canonical_num_days = calendar.monthrange(canonical_run_year, canonical_run_month)[1]
if CANONICAL_RUN_SCOPE == 'smoke':
    _days_text = os.environ.get('CANONICAL_RUN_DAYS', '1')
    canonical_run_days = [int(day) for day in _days_text.split(',') if day.strip()]
    if not canonical_run_days or any(day < 1 or day > canonical_num_days for day in canonical_run_days):
        raise ValueError(f'CANONICAL_RUN_DAYS must contain valid day numbers for {canonical_period_key}')
else:
    canonical_run_days = list(range(1, canonical_num_days + 1))
canonical_plot_dates = [
    f'{canonical_run_year:04d}{canonical_run_month:02d}{day:02d}'
    for day in canonical_run_days if day in {1, 15}
]
if not canonical_plot_dates:
    canonical_plot_dates = [f'{canonical_run_year:04d}{canonical_run_month:02d}{canonical_run_days[0]:02d}']

def _canonical_slug(value):
    return ''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in str(value)).strip('_')

canonical_run_tag = '_'.join([
    canonical_period_key.replace('-', ''),
    _canonical_slug(CANONICAL_RUN_SCOPE),
    _canonical_slug(CANONICAL_SITE_CASE),
    _canonical_slug(CANONICAL_RESOURCE_CASE),
    _canonical_slug(CANONICAL_FORECAST_CASE),
])

def _canonical_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def _canonical_code_sources(notebook_path):
    notebook = nbformat.read(notebook_path, as_version=4)
    return [str(cell.source) for cell in notebook.cells if cell.cell_type == 'code']

def _canonical_first_source(sources, *markers):
    matches = [source for source in sources if all(marker in source for marker in markers)]
    if len(matches) != 1:
        raise ValueError(f'Expected one source cell for markers {markers}, found {len(matches)}')
    return matches[0]

def _canonical_source_starting(sources, prefix):
    matches = [source for source in sources if source.lstrip().startswith(prefix)]
    if len(matches) != 1:
        raise ValueError(f'Expected one source cell starting with {prefix!r}, found {len(matches)}')
    return matches[0]

rolling_sources = [
    source for source in _canonical_code_sources(Path('upscaledev_imp_rollingMPC.ipynb'))
    if not source.lstrip().startswith((
        '# Canonical site/resource/forecast runner',
        '# Review controls:',
    ))
]
rolling_parameter_source = _canonical_first_source(rolling_sources, '# MPC parameters', 'SDGE_DGR_PRIMARY_2024')
rolling_data_source = _canonical_first_source(
    rolling_sources, '# VERSION: 2025 EV data', 'BASELINE_DISPATCH_FILE',
    'net-meter identity max error:'
)
rolling_plot_source = _canonical_first_source(
    rolling_sources, 'def plot_daily_solver_choice_figures', 'ax1_tou = ax1.twinx()'
)
rolling_main_source = _canonical_source_starting(rolling_sources, '# clean all csv files presaved')
rolling_financial_source = _canonical_first_source(
    rolling_sources, 'Save WM profit breakdown summary CSV', 'c_pd_day'
)
rolling_summary_source = _canonical_source_starting(
    rolling_sources, '# ============================================================\n# Run-Period Financial Summary CSVs Only'
)
rolling_main_source = re.sub(
    r'RUN_MAIN_LOOP_DIRECT\s*=\s*False', 'RUN_MAIN_LOOP_DIRECT = True', rolling_main_source
)

model_source_hash = hashlib.sha256('\n'.join([
    rolling_parameter_source, rolling_data_source, rolling_plot_source,
    rolling_main_source, rolling_financial_source, rolling_summary_source,
]).encode()).hexdigest()

baseline_sources = _canonical_code_sources(Path('upscaledev_baseline.ipynb'))
baseline_import_source = _canonical_first_source(baseline_sources, 'import pandas as pd', 'import scs')
baseline_parameter_source = _canonical_first_source(baseline_sources, '# MPC parameters', 'BASELINE_OUTPUT_DIR')
baseline_data_source = _canonical_first_source(baseline_sources, '# VERSION: 2025 data', 'EV baseline input')
baseline_loop_source = _canonical_first_source(baseline_sources, 'BASELINE_START_MONTH', 'stop month is exclusive')

def _canonical_build_site_baselines(site, config):
    baseline_dir = CANONICAL_BASELINE_DIR / site
    baseline_dir.mkdir(parents=True, exist_ok=True)
    expected = {
        label: baseline_dir / f'2025_{fc_session}_{fc_count}_{fc_arrival}_baseline.csv'
        for fc_session, fc_count, fc_arrival, label in canonical_forecasts
    }
    expected_index = pd.date_range(
        f'{canonical_run_year}-02-01 00:00:00',
        f'{canonical_run_year}-05-31 23:45:00',
        freq='15min',
    )

    def _validate_baseline(path, label):
        history = pd.read_csv(path, low_memory=False)
        if 'Interval start' not in history.columns:
            raise ValueError(f'{site} {label} baseline lacks Interval start')
        history_time = pd.DatetimeIndex(pd.to_datetime(history['Interval start'], errors='raise'))
        duplicate_count = int(history_time.duplicated().sum())
        missing = expected_index.difference(history_time)
        extra = history_time.difference(expected_index)
        if duplicate_count or len(missing) or len(extra) or len(history_time) != len(expected_index):
            raise ValueError(
                f'{site} {label} baseline calendar is invalid: rows={len(history_time)}, '
                f'duplicates={duplicate_count}, missing={len(missing)}, extra={len(extra)}. '
                'Set REBUILD_CANONICAL_BASELINES=1 after fixing baseline generation.'
            )
        return history

    if not REBUILD_CANONICAL_BASELINES and all(path.exists() for path in expected.values()):
        for label, path in expected.items():
            _validate_baseline(path, label)
        return expected

    baseline_namespace = {'__name__': '__main__'}
    exec(baseline_import_source, baseline_namespace, baseline_namespace)
    baseline_namespace.update({
        'EV_DATA_FILE': Path(config['ev_file']),
        'BASELINE_OUTPUT_DIR': baseline_dir,
        'BASELINE_START_MONTH': 2,
        'BASELINE_STOP_MONTH_EXCLUSIVE': 6,
    })
    exec(baseline_parameter_source, baseline_namespace, baseline_namespace)
    for fc_session, fc_count, fc_arrival, label in canonical_forecasts:
        baseline_namespace.update({
            'Fc_SessionkWh': fc_session,
            'Fc_NumbEV': fc_count,
            'Fc_AtArrival': fc_arrival,
        })
        exec(baseline_data_source, baseline_namespace, baseline_namespace)
        exec(baseline_loop_source, baseline_namespace, baseline_namespace)
        path = expected[label]
        if not path.exists():
            raise FileNotFoundError(f'Baseline generation did not create {path}')
        _validate_baseline(path, label)
    return expected

def _canonical_chronological_history(site, resource_case, forecast_label, baseline_path):
    if not CANONICAL_CHAIN_START_MONTH:
        return baseline_path
    if not 2 < CANONICAL_CHAIN_START_MONTH <= canonical_run_month:
        raise ValueError('Invalid chronological start month')
    start = pd.Timestamp(canonical_run_year, CANONICAL_CHAIN_START_MONTH, 1)
    end = pd.Timestamp(canonical_run_year, canonical_run_month, 1)
    history = pd.read_csv(baseline_path, low_memory=False)
    history['Interval start'] = pd.to_datetime(history['Interval start'])
    history = history[history['Interval start'] < start].copy()
    fc_session, fc_count, fc_arrival, _ = next(x for x in all_forecasts if x[3] == forecast_label)
    sources = [str(baseline_path)]
    for previous_month in range(CANONICAL_CHAIN_START_MONTH, canonical_run_month):
        path = (CANONICAL_STUDY_ROOT / 'monthly_runs' / f'{canonical_run_year}-{previous_month:02d}' /
                'model_outputs' / site / resource_case / 'Dispatch' /
                f'{canonical_run_year}_{fc_session}_{fc_count}_{fc_arrival}_full_implementation.csv')
        if not path.exists():
            raise FileNotFoundError(f'Run preceding month first: {path}')
        executed = pd.read_csv(path, low_memory=False)
        executed['Interval start'] = pd.to_datetime(executed['Interval start'])
        executed = executed[executed['Interval start'].dt.month.eq(previous_month) & executed['Interval start'].dt.year.eq(canonical_run_year)]
        expected = pd.date_range(pd.Timestamp(canonical_run_year, previous_month, 1), periods=calendar.monthrange(canonical_run_year, previous_month)[1]*96, freq='15min')
        if not pd.DatetimeIndex(executed['Interval start']).equals(expected):
            raise ValueError(f'Incomplete prior executed month: {path}')
        history = pd.concat([history, executed], ignore_index=True)
        sources.append(str(path))
    history = history.sort_values('Interval start').reset_index(drop=True)
    expected = pd.date_range(pd.Timestamp(canonical_run_year, 2, 1), end-pd.Timedelta(minutes=15), freq='15min')
    if not pd.DatetimeIndex(history['Interval start']).equals(expected):
        raise ValueError('History must be contiguous February plus prior executed months')
    destination = CANONICAL_PERIOD_ROOT / 'inputs' / site / resource_case
    destination.mkdir(parents=True, exist_ok=True)
    path = destination / f'{forecast_label}_preperiod_dispatch.csv'
    history.to_csv(path, index=False)
    path.with_suffix('.json').write_text(json.dumps({'sources': sources, 'baseline_cutoff_exclusive': str(start),
        'history_end_exclusive': str(end), 'source_sha256': {source: _canonical_sha256(source) for source in sources}}, indent=2))
    return path

def _canonical_direct_billing(site, config):
    btm = pd.read_csv(config['btm_file'])
    btm['Interval start'] = pd.to_datetime(btm['Interval start'], errors='raise')
    btm = btm[
        btm['Interval start'].dt.year.eq(canonical_run_year)
        & btm['Interval start'].dt.month.eq(canonical_run_month)
        & btm['Interval start'].dt.day.isin(canonical_run_days)
    ].copy()
    if btm.empty:
        raise ValueError(f'No passive-meter data for {site} in {canonical_period_key}')
    rows = []
    for resource_case in canonical_direct_resources:
        p_gi = btm['p_load_kW'].to_numpy(float)
        if resource_case == 'BUILDING_PV':
            p_gi = p_gi - btm['p_PV_kW'].to_numpy(float)
        rates, _, _ = get_sdge_dgr_tariff_profile(btm['Interval start'])
        on_peak = get_sdge_on_peak_mask(btm['Interval start']).astype(bool)
        tou_cost = -float(np.sum(rates * p_gi * dt_h))
        pd_peak = float(np.max(p_gi[on_peak])) if on_peak.any() else 0.0
        ncd_peak = float(np.max(p_gi))
        pd_cost = -get_sdge_pd_rate(btm['Interval start'].iloc[0]) * pd_peak
        ncd_cost = -c_NCD * ncd_peak
        rows.append({
            'site': site,
            'resource_case': resource_case,
            'days': len(canonical_run_days),
            'WM Revenue': 0.0,
            'TOU Cost': tou_cost,
            'PD Cost': pd_cost,
            'NCD Cost': ncd_cost,
            'EV Revenue': 0.0,
            'Total Revenue': tou_cost + pd_cost + ncd_cost,
            'definition': 'direct passive PCC billing; no optimization',
        })
    return rows

canonical_run_records = []
canonical_direct_rows = []
if not RUN_CANONICAL_STUDY:
    print(
        'Canonical paper runner is OFF. Set RUN_CANONICAL_STUDY=True (or environment variable 1) '
        'after reviewing CANONICAL_SITE_CASE, CANONICAL_RESOURCE_CASE, CANONICAL_FORECAST_CASE, '
        'and CANONICAL_RUN_SCOPE.'
    )
else:
    selected_configs = {site: CANONICAL_SITE_CONFIG[site] for site in canonical_sites}
    for site, config in selected_configs.items():
        for required in [config['ev_file'], config['btm_file']]:
            if not Path(required).exists():
                raise FileNotFoundError(
                    f'Missing canonical site input {required}; run upscaledev_site_data_postprocessing.ipynb'
                )
        canonical_direct_rows.extend(_canonical_direct_billing(site, config))

    baseline_by_site = {
        site: _canonical_build_site_baselines(site, config)
        for site, config in selected_configs.items()
    } if canonical_optimized_resources else {}

    exec(rolling_parameter_source, globals(), globals())
    SOLVER_THREADS = int(os.environ.get('CANONICAL_SOLVER_THREADS', '4'))
    exec(rolling_plot_source, globals(), globals())

    for site, config in selected_configs.items():
        for resource_case in canonical_optimized_resources:
            passive_enabled = resource_case == 'FULL_BTM'
            VERSION_DIR = str(CANONICAL_MODEL_OUTPUT_ROOT / site / resource_case)
            Path(VERSION_DIR).mkdir(parents=True, exist_ok=True)
            for fc_session, fc_count, fc_arrival, forecast_label in canonical_forecasts:
                EV_DATA_FILE = Path(config['ev_file'])
                BASELINE_DISPATCH_FILE = _canonical_chronological_history(
                    site, resource_case, forecast_label, baseline_by_site[site][forecast_label])
                PASSIVE_DER_ENABLED = passive_enabled
                BTM_BUILDING_FILE = Path(config['btm_file'])
                BTM_PV_FILE = Path(config['btm_file'])
                BTM_LOAD_SCALE = 1.0
                BTM_PV_SCALE = 1.0
                AS_ACTIVATION_MODE = 'caiso_regulation'
                AS_ACTIVATION_CONSTANTS = dict(AS_ACTIVATION_DEFAULTS)
                AS_ACTIVATION_SCALE = dict(AS_ACTIVATION_SCALE_DEFAULTS)
                AS_CONTINGENCY_ACTIVATION_FILE = None
                RUN_MONTHS_CONFIG = [canonical_run_month]
                RUN_DAYS_CONFIG = list(canonical_run_days)
                RUN_MODES = ['full']
                CLEAR_IMPLEMENTATION_AT_RUN_START = True
                PLOT_DAILY_6PANEL_DATES = list(canonical_plot_dates)
                ANALYSIS_DAYS_CONFIG = [
                    f'{canonical_run_year:04d}{canonical_run_month:02d}{day:02d}'
                    for day in canonical_run_days
                ]
                SENSITIVITY_ENABLED = False
                SENSITIVITY_CASE_NAME = f'canonical_{site}_{resource_case}'
                Fc_SessionkWh = fc_session
                Fc_NumbEV = fc_count
                Fc_AtArrival = fc_arrival
                WM_Mode = 'full'
                Enable_WM = True
                RT_SOLVER_STATUS_COUNTS = {}
                RT_SOLVER_LIMIT_EVENTS = []

                exec(rolling_data_source, globals(), globals())
                started = time.time()
                exec(rolling_main_source, globals(), globals())
                exec(rolling_financial_source, globals(), globals())
                exec(rolling_summary_source, globals(), globals())
                canonical_run_records.append({
                    'site': site,
                    'resource_case': resource_case,
                    'forecast': forecast_label,
                    'forecast_key': fc_session,
                    'elapsed_s': time.time() - started,
                    'optimal_steps': int(RT_SOLVER_STATUS_COUNTS.get('optimal', 0)),
                    'time_limit_events': len(RT_SOLVER_LIMIT_EVENTS),
                    'btm_net_identity_max_error_kW': (
                        float(BTM_NET_IDENTITY_MAX_ERROR_KW) if passive_enabled else 0.0
                    ),
                })

    if canonical_direct_rows:
        pd.DataFrame(canonical_direct_rows).to_csv(
            CANONICAL_TABLE_DIR / f'{canonical_run_tag}_passive_counterfactuals.csv',
            index=False, float_format='%.6f'
        )

    manifest = {
        'study': 'Canonical site-specific Rolling 24-h MPC',
        'scope': CANONICAL_RUN_SCOPE,
        'chain_start_month': CANONICAL_CHAIN_START_MONTH,
        'dates': [
            f'{canonical_run_year:04d}-{canonical_run_month:02d}-{day:02d}'
            for day in canonical_run_days
        ],
        'sites': canonical_sites,
        'optimized_resource_cases': canonical_optimized_resources,
        'direct_resource_cases': canonical_direct_resources,
        'forecast_cases': [item[3] for item in canonical_forecasts],
        'bess': {'power_kW': P_BESS_max, 'energy_kWh': C_BESS},
        'passive_der_scope': 'retail PCC meter only; no WM energy or AS capacity from PV/building',
        'model_source_sha256': model_source_hash,
        'input_sha256': {
            site: {
                'ev': _canonical_sha256(config['ev_file']),
                'btm': _canonical_sha256(config['btm_file']),
            }
            for site, config in selected_configs.items()
        },
        'runs': canonical_run_records,
    }
    (CANONICAL_PERIOD_ROOT / f'{canonical_run_tag}_manifest.json').write_text(
        json.dumps(manifest, indent=2)
    )

def collect_canonical_site_results():
    financial_rows = []
    validation_rows = []
    for record in canonical_run_records:
        site = record['site']
        resource_case = record['resource_case']
        forecast_label = record['forecast']
        fc_session, fc_count, fc_arrival, _ = next(
            item for item in canonical_forecasts if item[3] == forecast_label
        )
        result_root = CANONICAL_MODEL_OUTPUT_ROOT / site / resource_case
        forecast_key = f'2025_{fc_session}_{fc_count}_{fc_arrival}'
        daily_path = result_root / 'Plots/Cost' / forecast_key / 'daily_financial_detail.csv'
        daily = pd.read_csv(daily_path)
        daily['Date'] = pd.to_datetime(daily['Date'])
        daily = daily[
            daily['Date'].dt.year.eq(canonical_run_year)
            & daily['Date'].dt.month.eq(canonical_run_month)
            & daily['Date'].dt.day.isin(canonical_run_days)
            & daily['Case'].eq('Both')
        ]
        sums = daily[['Total Revenue', 'WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost', 'EV Revenue']].sum()
        financial_rows.append({
            'site': site, 'resource_case': resource_case, 'forecast': forecast_label,
            'days': len(daily), **sums.to_dict()
        })

        trace_dir = result_root / 'Validation_Traces' / f'{forecast_key}_full'
        trace_paths = [
            trace_dir / (
                f'rolling_validation_trace_{canonical_run_year:04d}'
                f'{canonical_run_month:02d}{day:02d}.csv'
            )
            for day in canonical_run_days
        ]
        missing = [str(path) for path in trace_paths if not path.exists()]
        if missing:
            raise FileNotFoundError(f'Missing canonical validation traces: {missing[:3]}')
        traces = pd.concat([pd.read_csv(path) for path in trace_paths], ignore_index=True)
        financial_identity = (
            daily['Total Revenue']
            - daily[['WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost', 'EV Revenue']].sum(axis=1)
        ).abs().max()
        implementation_path = result_root / 'Dispatch' / f'{forecast_key}_full_implementation.csv'
        implementation = pd.read_csv(implementation_path, low_memory=False)
        implementation['Interval start'] = pd.to_datetime(implementation['Interval start'])
        implementation = implementation[
            implementation['Interval start'].dt.year.eq(canonical_run_year)
            & implementation['Interval start'].dt.month.eq(canonical_run_month)
            & implementation['Interval start'].dt.day.isin(canonical_run_days)
        ]
        numeric_nan_count = int(implementation.select_dtypes(include=[np.number]).isna().sum().sum())
        validation = {
            'site': site,
            'resource_case': resource_case,
            'forecast': forecast_label,
            'trace_rows': len(traces),
            'expected_rows': 96 * len(canonical_run_days),
            'optimal_steps': record['optimal_steps'],
            'time_limit_events': record['time_limit_events'],
            'meter_residual_max_kW': float(traces['meter_balance_residual_kW'].abs().max()),
            'passive_WM_position_max_kW': float(traces['passive_WM_position_kW'].abs().max()),
            'AS_actual_min_kW': float(traces[[
                'c_RU_actual_kW', 'c_RD_actual_kW', 'c_SP_actual_kW', 'c_NSP_actual_kW'
            ]].min().min()),
            'SOC_min': float(traces['SOC'].min()),
            'SOC_max': float(traces['SOC'].max()),
            'EV_energy_kWh': float(traces['p_EV_kW'].sum() * dt_h),
            'financial_identity_max_dollar': float(financial_identity),
            'implementation_numeric_nan_count': numeric_nan_count,
            'btm_net_identity_max_error_kW': record['btm_net_identity_max_error_kW'],
        }
        validation['PASS'] = bool(
            validation['trace_rows'] == validation['expected_rows']
            and validation['optimal_steps'] == validation['expected_rows']
            and validation['time_limit_events'] == 0
            and validation['meter_residual_max_kW'] <= 1e-5
            and validation['passive_WM_position_max_kW'] <= 1e-9
            and validation['AS_actual_min_kW'] >= -1e-5
            and validation['SOC_min'] >= SOC_BESS_min - 1e-5
            and validation['SOC_max'] <= SOC_BESS_max + 1e-5
            and validation['financial_identity_max_dollar'] <= 0.02
            and validation['implementation_numeric_nan_count'] == 0
            and validation['btm_net_identity_max_error_kW'] <= 2e-5
        )
        validation_rows.append(validation)
    return pd.DataFrame(financial_rows), pd.DataFrame(validation_rows)

if RUN_CANONICAL_STUDY and canonical_run_records:
    canonical_financial, canonical_validation = collect_canonical_site_results()
    canonical_financial.to_csv(
        CANONICAL_TABLE_DIR / f'{canonical_run_tag}_financial_comparison.csv',
        index=False, float_format='%.6f'
    )
    canonical_validation.to_csv(
        CANONICAL_TABLE_DIR / f'{canonical_run_tag}_validation_summary.csv',
        index=False, float_format='%.9f'
    )
    display(canonical_financial.round(2))
    display(canonical_validation)
    if not canonical_validation['PASS'].all():
        raise AssertionError('Canonical site-study validation gate failed.')
    energy_spread = canonical_validation.groupby(['site', 'resource_case'])['EV_energy_kWh'].agg(
        lambda values: float(values.max() - values.min())
    )
    if (energy_spread > 0.02).any():
        raise AssertionError(f'Forecast cases did not serve equal EV energy: {energy_spread.to_dict()}')
    print('CANONICAL SITE STUDY VALIDATION: PASS')


In [7]:
# Save WM profit breakdown summary CSV for each day (full / retail_only)
# And build a daily table without rerunning optimization:
# rows: Retail only, Both
# cols: Total Cost, WM Revenue, TOU Cost, Demand Cost (PD+NCD), EV Revenue
import calendar
# 🛡️ 【新增健壮兜底】：如果代码里没有定义全局 RUN_MODES，默认使用原逻辑的两个
if 'RUN_MODES' not in globals():
    RUN_MODES = ['full', 'retail_only']
# 根据当前的 RUN_MODES 动态确定参与结算的模式
modes = [m for m in ['full', 'retail_only'] if m in RUN_MODES]
base_dir = Path(VERSION_DIR) / f'Plots/Solver_{TARGET_SAVE}_Choices'
financial_dir = base_dir / f'{TARGET_SAVE}_financial_tables'
financial_dir.mkdir(parents=True, exist_ok=True)
dispatch_dir = Path(VERSION_DIR) / 'Dispatch'
os.makedirs(financial_dir, exist_ok=True)
os.makedirs(dispatch_dir, exist_ok=True)
mode_label = {
    'retail_only': 'Retail only',
    'full': 'Both',
}
# Collect daily WM & TOU breakdown files by mode: {mode: {YYYYMMDD: file_path}}
daily_wm_files = {m: {} for m in modes}
for mode in modes:
    scenario_dir = base_dir / f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{mode}'
    for fp in sorted(scenario_dir.glob(f'*/{TARGET_SAVE}_WM_profit_breakdown_{mode}.csv')):
        day_key = fp.parent.name
        daily_wm_files[mode][day_key] = fp
daily_tou_files = {m: {} for m in modes}
for mode in modes:
    scenario_dir = base_dir / f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{mode}'
    for fp in sorted(scenario_dir.glob(f'*/{TARGET_SAVE}_WM_TOU_breakdown_{mode}.csv')):
        day_key = fp.parent.name
        daily_tou_files[mode][day_key] = fp
# Collect implementation and daily summary files by mode
impl_file_by_mode = {}
daily_summary_file_by_mode = {}
for mode in modes:
    impl_file = dispatch_dir / f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{mode}_implementation.csv'
    if impl_file.exists():
        impl_file_by_mode[mode] = impl_file
    summary_file = dispatch_dir / f'2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}_{mode}_daily_summary.csv'
    if summary_file.exists():
        daily_summary_file_by_mode[mode] = summary_file
# 🛡️ 【核心修改：健壮交集】：只对自己激活的 modes 进行天数交集匹配，防止因为缺失另一个 mode 而崩溃
common_days = None
for mode in modes:
    mode_days = set(daily_wm_files[mode].keys()).intersection(set(daily_tou_files[mode].keys()))
    if common_days is None:
        common_days = mode_days
    else:
        common_days = common_days.intersection(mode_days)
if not common_days:
    raise FileNotFoundError(f'No complete daily set found for the active modes: {modes}.')
# Cache dispatch csvs once
impl_cache = {m: pd.read_csv(p) for m, p in impl_file_by_mode.items()}
for m in impl_cache:
    impl_cache[m]['Interval start'] = pd.to_datetime(impl_cache[m]['Interval start'])
summary_cache = {m: pd.read_csv(p) for m, p in daily_summary_file_by_mode.items()}
# Tariff profiles are rebuilt for each calendar day below; no static 96-step fallback is used.

c_ncd_local = float(globals().get('c_NCD', 15.38))
ev_service_rate = 0.4
saved_paths = []
# 🛡️ 用字典来动态维护每个 mode 的最大功率追溯，防止多天循环时索引混乱
max_p_ncd_so_far = {m: 0.0 for m in modes}
max_p_pd_so_far = {m: 0.0 for m in modes}
current_billing_month = None
for day_key in sorted(common_days):
    day_ts_for_reset = pd.to_datetime(day_key, format='%Y%m%d')
    billing_month = day_ts_for_reset.to_period('M')
    if current_billing_month != billing_month:
        max_p_ncd_so_far = {m: 0.0 for m in modes}
        max_p_pd_so_far = {m: 0.0 for m in modes}
        current_billing_month = billing_month
    # 🛡️ 【健壮拆分表生成】：只对当前存在于 modes 里的数据进行汇总提取
    profit_rows = []
    profit_idx = []
    for mode in modes:
        mode_df = pd.read_csv(daily_wm_files[mode][day_key])
        breakdown_cols = [c for c in mode_df.columns if c not in {'Interval start', 'profit_sum_all_products'}]
        profit_rows.append(mode_df[breakdown_cols].sum(numeric_only=True))
        profit_idx.append(mode)
    profit_sum_table = pd.DataFrame(profit_rows, index=profit_idx)
    profit_sum_table.columns = [col.replace('_profit', '') for col in profit_sum_table.columns]
    profit_sum_table = profit_sum_table.round(2)
    profit_sum_table['wm_total'] = profit_sum_table.sum(axis=1).round(2)
    tou_rows = []
    tou_idx = []
    for mode in modes:
        mode_tou_df = pd.read_csv(daily_tou_files[mode][day_key])
        breakdown_cols = [c for c in mode_tou_df.columns if c not in {'Interval start', 'cost_sum_all_products'}]
        tou_rows.append(mode_tou_df[breakdown_cols].sum(numeric_only=True))
        tou_idx.append(mode)
    tou_sum_table = pd.DataFrame(tou_rows, index=tou_idx)
    tou_sum_table.columns = [col.replace('_cost', '') for col in tou_sum_table.columns]
    tou_sum_table = tou_sum_table.round(2)
    tou_sum_table['tou_total'] = tou_sum_table.sum(axis=1).round(2)
    for mode in modes:
        out_dir = daily_wm_files[mode][day_key].parent
        out_file = out_dir / f'{TARGET_SAVE}_WM_profit_breakdown_summary_{day_key}.csv'
        profit_sum_table.to_csv(out_file, float_format='%.2f')
        saved_paths.append(out_file)
        out_file = out_dir / f'{TARGET_SAVE}_WM_TOU_breakdown_summary_{day_key}.csv'
        tou_sum_table.to_csv(out_file, float_format='%.2f')
        saved_paths.append(out_file)
    # 2) build daily financial table
    day_rows = []
    # 🛡️ 仅遍历当前被激活的模式
    for mode in modes:
        if mode not in impl_cache or mode not in summary_cache:
            continue
        day_ts = pd.to_datetime(day_key, format='%Y%m%d')
        tou_rates_96, _, _ = get_sdge_dgr_tariff_profile(pd.date_range(day_ts, periods=96, freq='15min'))
        c_pd_day = get_sdge_pd_rate(day_ts)
        day_start = day_ts
        day_end = day_ts + pd.Timedelta(days=1)
        day_impl = impl_cache[mode][
            (impl_cache[mode]['Interval start'] >= day_start)
            & (impl_cache[mode]['Interval start'] < day_end)
        ].copy()
        if day_impl.empty:
            continue
        wm_df = pd.read_csv(daily_wm_files[mode][day_key])
        wm_revenue = float(wm_df['profit_sum_all_products'].sum())
        wm_bess_energy = float(wm_df['BESS_Energy_Profit'].sum()) if 'BESS_Energy_Profit' in wm_df.columns else 0.0
        wm_ev_energy   = float(wm_df['EV_Energy_Profit'].sum()) if 'EV_Energy_Profit' in wm_df.columns else 0.0
        wm_bess_cap    = float(wm_df['BESS_Capacity_Profit'].sum()) if 'BESS_Capacity_Profit' in wm_df.columns else 0.0
        wm_ev_cap      = float(wm_df['EV_Capacity_Profit'].sum()) if 'EV_Capacity_Profit' in wm_df.columns else 0.0
        wm_bess_total  = float(wm_df['BESS_Total_WM_Profit'].sum()) if 'BESS_Total_WM_Profit' in wm_df.columns else 0.0
        wm_ev_total    = float(wm_df['EV_Total_WM_Profit'].sum()) if 'EV_Total_WM_Profit' in wm_df.columns else 0.0
        # Reconcile interval-CSV rounding so controllable EV+BESS components sum exactly
        # to the settled WM revenue used in the financial identity.
        wm_ev_energy = wm_revenue - wm_bess_energy - wm_bess_cap - wm_ev_cap
        wm_bess_total = wm_bess_energy + wm_bess_cap
        wm_ev_total = wm_ev_energy + wm_ev_cap
        tou_energy_col = 'Base_meter [kWh]' if 'Base_meter [kWh]' in day_impl.columns else ('Base_withBESS [kWh]' if 'Base_withBESS [kWh]' in day_impl.columns else 'Base [kWh]')
        if tou_energy_col not in day_impl.columns:
            raise KeyError(f"{tou_energy_col} column missing in implementation file for mode={mode}.")
        base_kwh = pd.to_numeric(day_impl[tou_energy_col], errors='coerce').fillna(0.0).to_numpy()
        if len(base_kwh) != 96:
            n = min(len(base_kwh), len(tou_rates_96))
            tou_cost_abs = float(np.sum(base_kwh[:n] * tou_rates_96[:n]))
        else:
            tou_cost_abs = float(np.sum(base_kwh * tou_rates_96))
        tou_cost = -tou_cost_abs
        # NWM energy attribution is exactly additive. Passive building/PV
        # meter energy is kept separate instead of being mislabeled as BESS.
        ev_profile_col = f'{Cases[0]} [kWh]' if f'{Cases[0]} [kWh]' in day_impl.columns else 'Base [kWh]'
        ev_kwh_profile_all = pd.to_numeric(day_impl[ev_profile_col], errors='coerce').fillna(0.0).to_numpy()
        n_profile = min(len(base_kwh), len(ev_kwh_profile_all), len(tou_rates_96))
        grid_kwh_profile = np.asarray(base_kwh[:n_profile], dtype=float)
        ev_kwh_profile = np.asarray(ev_kwh_profile_all[:n_profile], dtype=float)
        if {'p_load [kW]', 'p_PV [kW]'}.issubset(day_impl.columns):
            native_kwh_profile = (
                pd.to_numeric(day_impl['p_load [kW]'], errors='coerce').fillna(0.0).to_numpy()[:n_profile]
                - pd.to_numeric(day_impl['p_PV [kW]'], errors='coerce').fillna(0.0).to_numpy()[:n_profile]
            ) * dt_h
        else:
            native_kwh_profile = np.zeros(n_profile, dtype=float)
        bess_kwh_profile = grid_kwh_profile - ev_kwh_profile - native_kwh_profile
        tou_rate_profile = np.asarray(tou_rates_96[:n_profile], dtype=float)
        nwm_ev_tou = -float(np.sum(ev_kwh_profile * tou_rate_profile))
        nwm_bess_energy = -float(np.sum(bess_kwh_profile * tou_rate_profile))
        nwm_passive_der_energy = -float(np.sum(native_kwh_profile * tou_rate_profile))
        ncd_col = 'NCD_Base' if 'NCD_Base' in day_impl.columns else None
        pd_col = 'PD_Base' if 'PD_Base' in day_impl.columns else None
        if ncd_col is None or pd_col is None:
            raise KeyError(f"NCD_Base/PD_Base missing in implementation file for mode={mode}.")
        ncd_peak_kw = float(pd.to_numeric(day_impl[ncd_col], errors='coerce').fillna(0.0).max())
        pd_peak_kw = float(pd.to_numeric(day_impl[pd_col], errors='coerce').fillna(0.0).max())
        # 🛡️ 使用字典动态检索更新，安全隔离各模式各自的最大追溯功率
        ncd_peak_diff = max(ncd_peak_kw - max_p_ncd_so_far[mode], 0)
        pd_peak_diff = max(pd_peak_kw - max_p_pd_so_far[mode], 0)
        pd_cost = -float(c_pd_day * pd_peak_diff)
        ncd_cost = -float(c_ncd_local * ncd_peak_diff)
        demand_cost = pd_cost + ncd_cost
        # Attribute each incremental demand charge at its binding realized peak.
        grid_kw_profile = grid_kwh_profile / dt_h
        ev_kw_profile = ev_kwh_profile / dt_h
        native_kw_profile = native_kwh_profile / dt_h
        interval_profile = pd.to_datetime(day_impl['Interval start']).iloc[:n_profile]
        def _split_incremental_demand_cost(cost_value, target_peak_kw, candidate_mask):
            if abs(cost_value) <= 1e-12 or n_profile == 0:
                return 0.0, 0.0, 0.0
            candidates = np.flatnonzero(np.asarray(candidate_mask, dtype=bool))
            if len(candidates) == 0:
                candidates = np.arange(n_profile)
            idx = int(candidates[np.argmin(np.abs(grid_kw_profile[candidates] - target_peak_kw))])
            denom = float(grid_kw_profile[idx])
            if abs(denom) <= 1e-9:
                return float(cost_value), 0.0, 0.0
            ev_part = float(cost_value) * float(ev_kw_profile[idx]) / denom
            native_part = float(cost_value) * float(native_kw_profile[idx]) / denom
            return ev_part, float(cost_value) - ev_part - native_part, native_part
        all_mask = np.ones(n_profile, dtype=bool)
        pd_mask_profile = get_sdge_on_peak_mask(interval_profile) > 0.5
        ncd_ev_cap, ncd_bess_cap, ncd_native_cap = _split_incremental_demand_cost(ncd_cost, ncd_peak_kw, all_mask)
        pd_ev_cap, pd_bess_cap, pd_native_cap = _split_incremental_demand_cost(pd_cost, pd_peak_kw, pd_mask_profile)
        nwm_ev_capacity = ncd_ev_cap + pd_ev_cap
        nwm_bess_capacity = ncd_bess_cap + pd_bess_cap
        nwm_passive_der_capacity = ncd_native_cap + pd_native_cap
        max_p_ncd_so_far[mode] = max(max_p_ncd_so_far[mode], ncd_peak_kw)
        max_p_pd_so_far[mode]  = max(max_p_pd_so_far[mode], pd_peak_kw)
        # EV revenue from physical EV-only executed energy. Use implementation CSV,
        # not daily_summary validation rows, because daily_summary is row-per-session and
        # can contain preserved historical rows across reruns.
        ev_energy_col = f'{Cases[0]} [kWh]' if f'{Cases[0]} [kWh]' in day_impl.columns else 'Base [kWh]'
        if ev_energy_col not in day_impl.columns:
            raise KeyError(f"{ev_energy_col} column missing in implementation file for mode={mode}.")
        ev_energy_kwh = float(pd.to_numeric(day_impl[ev_energy_col], errors='coerce').fillna(0.0).sum())
        ev_revenue = ev_service_rate * ev_energy_kwh
        nwm_ev_energy = ev_revenue + nwm_ev_tou
        total_revenue = wm_revenue + tou_cost + demand_cost + ev_revenue
        day_rows.append({
            'Case / US$': mode_label[mode],
            'Total Revenue': round(total_revenue, 2),
            'WM Revenue': round(wm_revenue, 2),
            'WM BESS Energy': round(wm_bess_energy, 2),
            'WM EV Energy': round(wm_ev_energy, 2),
            'WM BESS Capacity': round(wm_bess_cap, 2),
            'WM EV Capacity': round(wm_ev_cap, 2),
            'WM BESS Total': round(wm_bess_total, 2),
            'WM EV Total': round(wm_ev_total, 2),
            'NWM BESS Energy': round(nwm_bess_energy, 2),
            'NWM EV Energy': round(nwm_ev_energy, 2),
            'NWM BESS Capacity': round(nwm_bess_capacity, 2),
            'NWM EV Capacity': round(nwm_ev_capacity, 2),
            'NWM Passive DER Energy': round(nwm_passive_der_energy, 2),
            'NWM Passive DER Capacity': round(nwm_passive_der_capacity, 2),
            'TOU Cost': round(tou_cost, 2),
            'PD Cost': round(pd_cost, 2),
            'NCD Cost': round(ncd_cost, 2),
            'EV Revenue': round(ev_revenue, 2),
        })
    # 🛡️ 【核心修改：健壮表格生成】：只要有活跃行数据就保存，不再卡死“必须等于2行”
    if len(day_rows) > 0:
        table_2x5 = pd.DataFrame(day_rows)
        # 依据定义好的范畴进行排序排列
        order = [mode_label[m] for m in ['retail_only', 'full'] if m in modes]
        table_2x5['Case / US$'] = pd.Categorical(table_2x5['Case / US$'], categories=order, ordered=True)
        table_2x5 = table_2x5.sort_values('Case / US$').reset_index(drop=True)
        out_main = financial_dir / f'{TARGET_SAVE}_financial_table_{day_key}.csv'
        table_2x5.to_csv(out_main, index=False, float_format='%.2f')
        saved_paths.append(out_main)
print(f'Saved {len(saved_paths)} files across {len(common_days)} days.')

Saved 93 files across 31 days.


In [8]:
# ============================================================
# Run-Period Financial Summary CSVs Only
# ============================================================
# Main optimization notebooks intentionally do not generate paper figures.
# Use paper_financial_comparison_plots.ipynb for all publication-quality plots.
from pathlib import Path
import numpy as np
import pandas as pd
_version_dir = Path(globals().get('VERSION_DIR', 'Results_Rolling'))
base_dir_fin = _version_dir / f"Plots/Solver_{TARGET_SAVE}_Choices/{TARGET_SAVE}_financial_tables"
cost_plot_dir = _version_dir / "Plots/Cost" / f"2025_{Fc_SessionkWh}_{Fc_NumbEV}_{Fc_AtArrival}"
cost_plot_dir.mkdir(parents=True, exist_ok=True)
modes_sum = ['retail_only', 'full']
mode_label_sum = {'retail_only': 'Retail only', 'full': 'Both'}
metric_cols = ['Total Revenue', 'WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost', 'EV Revenue']
extra_cols = [
    'WM BESS Energy', 'WM EV Energy',
    'WM BESS Capacity', 'WM EV Capacity',
    'WM BESS Total', 'WM EV Total',
    'NWM BESS Energy', 'NWM EV Energy', 'NWM BESS Capacity', 'NWM EV Capacity',
    'NWM Passive DER Energy', 'NWM Passive DER Capacity'
]
fin_files = sorted(base_dir_fin.glob(f'{TARGET_SAVE}_financial_table_????????.csv'))
_analysis_days_config = globals().get('ANALYSIS_DAYS_CONFIG', None)
if _analysis_days_config is None:
    _analysis_days_config = globals().get('RUN_DAYS_CONFIG', None)
if _analysis_days_config:
    _analysis_day_keys = set()
    for _d in _analysis_days_config:
        _s = str(_d)
        if _s.isdigit() and len(_s) <= 2:
            _analysis_day_keys.add(f"{int(globals().get('year', 2025)):04d}{int(globals().get('month', 7)):02d}{int(_s):02d}")
        else:
            _analysis_day_keys.add(pd.Timestamp(_d).strftime('%Y%m%d'))
    fin_files = [fp for fp in fin_files if fp.stem.replace(f'{TARGET_SAVE}_financial_table_', '') in _analysis_day_keys]
    print(f'Financial summary restricted to run dates: {sorted(_analysis_day_keys)}')
if not fin_files:
    raise FileNotFoundError(f'No daily financial tables found in {base_dir_fin} for the configured run dates.')
daily_records = []
for fp in fin_files:
    day_key = fp.stem.replace(f'{TARGET_SAVE}_financial_table_', '')
    day_ts = pd.to_datetime(day_key, format='%Y%m%d')
    df = pd.read_csv(fp)
    for mode in modes_sum:
        label = mode_label_sum[mode]
        row_df = df[df['Case / US$'] == label]
        if row_df.empty:
            continue
        row = row_df.iloc[0]
        rec = {'Date': day_ts, 'Case': label}
        for col in metric_cols + extra_cols:
            rec[col] = float(row.get(col, 0.0))
        rec['WM Energy Revenue'] = rec['WM BESS Energy'] + rec['WM EV Energy']
        rec['WM Capacity Revenue'] = rec['WM BESS Capacity'] + rec['WM EV Capacity']
        daily_records.append(rec)
daily_df = pd.DataFrame(daily_records).sort_values(['Date', 'Case']).reset_index(drop=True)
# Keep an immutable view of the configured run period for the printed/summary
# totals. The archival daily CSV may contain other dates from earlier runs.
run_period_daily_df = daily_df.copy()
summary_csv = cost_plot_dir / 'monthly_financial_summary.csv'
daily_csv = cost_plot_dir / 'daily_financial_detail.csv'
# Preserve previously saved dates/cases that were not part of this run.
# The current run replaces only matching (Date, Case) rows, so a Retail-only
# rerun cannot erase already validated Full/Both financial results.
if daily_csv.exists():
    previous_daily = pd.read_csv(daily_csv)
    if {'Date', 'Case'}.issubset(previous_daily.columns):
        previous_daily['Date'] = pd.to_datetime(previous_daily['Date'])
        daily_df = pd.concat([previous_daily, daily_df], ignore_index=True, sort=False)
daily_df = daily_df.drop_duplicates(['Date', 'Case'], keep='last').sort_values(['Date', 'Case']).reset_index(drop=True)
monthly_summary = (
    run_period_daily_df.groupby('Case')[metric_cols]
    .sum(numeric_only=True)
    .round(2)
    .reindex([label for label in ['Retail only', 'Both'] if label in set(daily_df['Case'])])
)
monthly_summary.index.name = 'Case / US$'
monthly_summary.to_csv(summary_csv, float_format='%.2f')
daily_df.to_csv(daily_csv, index=False, float_format='%.2f')
print('\n' + '='*70)
print('  Run-Period Financial Summary  (US$)')
print('='*70)
print(monthly_summary.to_string())
print('='*70)
print(f'Daily detail CSV saved to: {daily_csv}')
print(f'Summary CSV saved to: {summary_csv}')
print('Paper figures are generated by paper_financial_comparison_plots.ipynb')


Financial summary restricted to run dates: ['20250701', '20250702', '20250703', '20250704', '20250705', '20250706', '20250707', '20250708', '20250709', '20250710', '20250711', '20250712', '20250713', '20250714', '20250715', '20250716', '20250717', '20250718', '20250719', '20250720', '20250721', '20250722', '20250723', '20250724', '20250725', '20250726', '20250727', '20250728', '20250729', '20250730', '20250731']

  Run-Period Financial Summary  (US$)
            Total Revenue  WM Revenue   TOU Cost  PD Cost  NCD Cost  EV Revenue
Case / US$                                                                     
Both           -252842.03     3951.48 -253808.03  -3943.5 -20989.77     21947.8
Daily detail CSV saved to: Sensitivity_Results/Rolling_24h/july2025_retail_btm_caiso_alpha/ev_bess_pv_building_retail/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv
Summary CSV saved to: Sensitivity_Results/Rolling_24h/july2025_retail_btm_caiso_alpha/e

## Isolated 24 h Rolling sensitivity runner

This optional runner keeps the validated reference outputs immutable. Set `RUN_ROLLING_SENSITIVITY = True` only after reviewing the case list. Each enabled case writes to `Sensitivity_Results/Rolling_24h/<case>/<resource_tag>`, records a manifest, and uses the same 24 h Rolling model. `passive_der_enabled=True` reads independently QCed building and PV meter data, but these passive resources enter only retail TOU and demand charges. They have no DA/RT energy position, AS capacity, wholesale baseline, or WM revenue. CAISO regulation attenuation factors are produced by `upscaledev_AS_callrate_postprocessing.ipynb`; constant call-rate cases provide controlled counterfactuals.

In [9]:
# Review controls: this cell does nothing unless explicitly enabled.
RUN_ROLLING_SENSITIVITY = os.environ.get('RUN_ROLLING_SENSITIVITY', '0') == '1'
ROLLING_SENSITIVITY_BATCH = os.environ.get('ROLLING_SENSITIVITY_BATCH', '')
ROLLING_SENSITIVITY_CASE_FILTER = os.environ.get('ROLLING_SENSITIVITY_CASE_FILTER', '')
ROLLING_SENSITIVITY_SKIP_SOLVE = os.environ.get('ROLLING_SENSITIVITY_SKIP_SOLVE', '0') == '1'
ROLLING_SENSITIVITY_REUSE_EXISTING = os.environ.get('ROLLING_SENSITIVITY_REUSE_EXISTING', '0') == '1'
_COMMON_SENSITIVITY_INPUTS = {
    'modes': ['full'],
    'passive_der_enabled': True,
    'alpha_scale': {'RU': 1.0, 'RD': 1.0, 'SP': 1.0, 'NSP': 1.0},
    'contingency_activation_file': None,
    'ev_data_file': '2025Data/EV_data/UCSD_AllSites_Merge_PostProcessedSession_QC.csv',
    'building_input_file': '2025Data/Bldg_data/Bldg_load_2025_15min_QC.csv',
    'pv_input_file': '2025Data/PV_data/PV_generation_2025_15min_QC.csv',
    'building_scale': 1.0, 'pv_scale': 1.0,
}
_PERSISTENCE_FORECAST = [('PersistenceSessionkWh', 'PersistenceNumbEV', 'PerfectatArrival')]
_TWO_FORECASTS = _PERSISTENCE_FORECAST + [('PerfectSessionkWh', 'PerfectNumbEV', 'PerfectatArrival')]
ROLLING_SENSITIVITY_CASES = []
# One-day controlled regulation-call screen. SP/NSP stay at 0.2 so only RU/RD changes.
for _reg_alpha in [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]:
    ROLLING_SENSITIVITY_CASES.append({
        **_COMMON_SENSITIVITY_INPUTS, 'enabled': False, 'run_kind': 'july01_callrate_screen',
        'name': f'july01_retail_btm_reg_alpha_{int(round(100 * _reg_alpha)):03d}',
        'months': [7], 'days': [1], 'forecasts': _PERSISTENCE_FORECAST,
        'alpha_mode': 'constant',
        'alpha_constants': {'RU': _reg_alpha, 'RD': _reg_alpha, 'SP': 0.2, 'NSP': 0.2},
    })
ROLLING_SENSITIVITY_CASES += [
    {**_COMMON_SENSITIVITY_INPUTS, 'enabled': False, 'run_kind': 'july01_callrate_screen',
     'name': 'july01_retail_btm_caiso_alpha', 'months': [7], 'days': [1],
     'forecasts': _PERSISTENCE_FORECAST, 'alpha_mode': 'caiso_regulation'},
    {**_COMMON_SENSITIVITY_INPUTS, 'enabled': False, 'run_kind': 'monthly_forecast_comparison',
     'name': 'july2025_retail_btm_caiso_alpha', 'months': [7], 'days': list(range(1, 32)),
     'forecasts': _TWO_FORECASTS, 'alpha_mode': 'caiso_regulation'},
]
# Formal July forecast x passive-BTM comparison.  An optional day list is used only for smoke tests.
_JULY_BTM_2X2_DAY_TEXT = os.environ.get('ROLLING_SENSITIVITY_DAYS', '').strip()
_JULY_BTM_2X2_DAYS = ([int(v) for v in _JULY_BTM_2X2_DAY_TEXT.split(',') if v.strip()]
                        if _JULY_BTM_2X2_DAY_TEXT else list(range(1, 32)))
for _passive_der_enabled in [False, True]:
    ROLLING_SENSITIVITY_CASES.append({
        **_COMMON_SENSITIVITY_INPUTS,
        'enabled': ROLLING_SENSITIVITY_BATCH == 'july_btm_2x2',
        'run_kind': 'july_btm_2x2',
        'name': 'july2025_forecast_btm_2x2',
        'months': [7], 'days': _JULY_BTM_2X2_DAYS,
        'plot_dates': ['20250701'], 'forecasts': _TWO_FORECASTS,
        'passive_der_enabled': _passive_der_enabled, 'alpha_mode': 'caiso_regulation',
    })
# Unit-consistent multi-date screen. These are scenario activation factors,
# while the CAISO case uses hourly SOC attenuation factors (not realized AGC).
_CALLRATE_UNITFIX_DAYS = [1, 2, 3]
for _reg_alpha in [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]:
    ROLLING_SENSITIVITY_CASES.append({
        **_COMMON_SENSITIVITY_INPUTS,
        'enabled': ROLLING_SENSITIVITY_BATCH == 'callrate_unitfix_multidate',
        'run_kind': 'callrate_unitfix_multidate',
        'name': f'callrate_unitfix_20250701_03_reg_alpha_{int(round(100 * _reg_alpha)):03d}',
        'months': [7], 'days': _CALLRATE_UNITFIX_DAYS,
        'plot_dates': [f'202507{d:02d}' for d in _CALLRATE_UNITFIX_DAYS],
        'forecasts': _PERSISTENCE_FORECAST, 'alpha_mode': 'constant',
        'alpha_constants': {'RU': _reg_alpha, 'RD': _reg_alpha, 'SP': 0.2, 'NSP': 0.2},
    })
ROLLING_SENSITIVITY_CASES.append({
    **_COMMON_SENSITIVITY_INPUTS,
    'enabled': ROLLING_SENSITIVITY_BATCH == 'callrate_unitfix_multidate',
    'run_kind': 'callrate_unitfix_multidate',
    'name': 'callrate_unitfix_20250701_03_caiso_attenuation',
    'months': [7], 'days': _CALLRATE_UNITFIX_DAYS,
    'plot_dates': [f'202507{d:02d}' for d in _CALLRATE_UNITFIX_DAYS],
    'forecasts': _PERSISTENCE_FORECAST, 'alpha_mode': 'caiso_regulation',
})

if RUN_ROLLING_SENSITIVITY:
    import hashlib, json, nbformat, re, subprocess
    notebook_path = Path('upscaledev_imp_rollingMPC.ipynb')
    nb_self = nbformat.read(str(notebook_path), as_version=4)
    data_source = main_loop_source = financial_table_source = financial_summary_source = None
    for cell in nb_self.cells:
        if cell.get('cell_type') != 'code':
            continue
        src = str(cell.get('source', ''))
        is_sensitivity_runner = ('RUN_ROLLING_SENSITIVITY' in src and 'ROLLING_SENSITIVITY_CASES' in src)
        if is_sensitivity_runner:
            continue
        if '# VERSION: 2025 EV data' in src and 'Read baseline data' in src:
            data_source = src
        if 'Main loop runtime' in src and 'RUN_MAIN_LOOP_DIRECT' in src and 'Run July-August 2x2' not in src:
            main_loop_source = src
        if 'Save WM profit breakdown summary CSV' in src and 'Run July-August 2x2' not in src:
            financial_table_source = src
        if 'Run-Period Financial Summary CSVs Only' in src and 'Run July-August 2x2' not in src:
            financial_summary_source = src
    if any(src is None for src in [data_source, main_loop_source, financial_table_source, financial_summary_source]):
        raise ValueError('Could not locate all source cells required by the sensitivity runner')
    main_loop_source = re.sub(r'RUN_MAIN_LOOP_DIRECT\s*=\s*False', 'RUN_MAIN_LOOP_DIRECT = True', main_loop_source)
    model_source_sha256 = hashlib.sha256(('\n'.join([data_source, main_loop_source, financial_table_source, financial_summary_source])).encode('utf-8')).hexdigest()

    def _sha256(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(chunk)
        return digest.hexdigest()

    _enabled_sensitivity_cases = [c for c in ROLLING_SENSITIVITY_CASES if c.get('enabled', False)]
    if ROLLING_SENSITIVITY_CASE_FILTER:
        _enabled_sensitivity_cases = [c for c in _enabled_sensitivity_cases if ROLLING_SENSITIVITY_CASE_FILTER in c['name']]
    for sensitivity_case in _enabled_sensitivity_cases:
        if ROLLING_SENSITIVITY_SKIP_SOLVE:
            print('Reusing existing sensitivity results:', sensitivity_case['name'])
            continue
        _candidate_resource_tag = 'ev_bess_pv_building_retail' if sensitivity_case.get('passive_der_enabled', False) else 'ev_bess'
        _candidate_dir = SENSITIVITY_OUTPUT_ROOT / _safe_sensitivity_slug(sensitivity_case['name']) / _candidate_resource_tag
        if ROLLING_SENSITIVITY_REUSE_EXISTING and (_candidate_dir / 'sensitivity_manifest.json').exists():
            print('Validated result directory already exists; reusing:', _candidate_dir)
            continue
        SENSITIVITY_ENABLED = True
        SENSITIVITY_CASE_NAME = _safe_sensitivity_slug(sensitivity_case['name'])
        PASSIVE_DER_ENABLED = bool(sensitivity_case.get('passive_der_enabled', False))
        SENSITIVITY_RESOURCE_TAG = 'ev_bess_pv_building_retail' if PASSIVE_DER_ENABLED else 'ev_bess'
        VERSION_DIR = str(SENSITIVITY_OUTPUT_ROOT / SENSITIVITY_CASE_NAME / SENSITIVITY_RESOURCE_TAG)
        if Path(VERSION_DIR).resolve() == BASE_VERSION_DIR.resolve():
            raise ValueError('Sensitivity output may not equal the validated reference directory')
        EV_DATA_FILE = Path(sensitivity_case['ev_data_file'])
        BASELINE_DISPATCH_FILE = sensitivity_case.get('baseline_dispatch_file')
        BTM_BUILDING_FILE = Path(sensitivity_case.get('building_input_file', BTM_BUILDING_FILE))
        BTM_PV_FILE = Path(sensitivity_case.get('pv_input_file', BTM_PV_FILE))
        BTM_LOAD_SCALE = float(sensitivity_case.get('building_scale', 1.0))
        BTM_PV_SCALE = float(sensitivity_case.get('pv_scale', 1.0))
        AS_ACTIVATION_MODE = sensitivity_case.get('alpha_mode', 'constant')
        AS_ACTIVATION_CONSTANTS = dict(sensitivity_case.get('alpha_constants', AS_ACTIVATION_DEFAULTS))
        AS_ACTIVATION_SCALE = dict(sensitivity_case.get('alpha_scale', AS_ACTIVATION_SCALE_DEFAULTS))
        AS_CONTINGENCY_ACTIVATION_FILE = sensitivity_case.get('contingency_activation_file')
        RUN_MONTHS_CONFIG = list(sensitivity_case['months'])
        RUN_DAYS_CONFIG = list(sensitivity_case['days'])
        RUN_MODES = list(sensitivity_case.get('modes', ['full']))
        CLEAR_IMPLEMENTATION_AT_RUN_START = True
        PLOT_DAILY_6PANEL_DATES = list(sensitivity_case.get('plot_dates', ['20250701']))
        Path(VERSION_DIR).mkdir(parents=True, exist_ok=True)
        case_solver_audit = {}
        for Fc_SessionkWh, Fc_NumbEV, Fc_AtArrival in sensitivity_case['forecasts']:
            exec(data_source, globals(), globals())
            for WM_Mode in RUN_MODES:
                Enable_WM = WM_Mode == 'full'
                RT_SOLVER_STATUS_COUNTS = {}
                RT_SOLVER_LIMIT_EVENTS = []
                exec(main_loop_source, globals(), globals())
                audit_key = f'{Fc_SessionkWh}|{Fc_NumbEV}|{Fc_AtArrival}|{WM_Mode}'
                case_solver_audit[audit_key] = {
                    'status_counts': dict(RT_SOLVER_STATUS_COUNTS),
                    'time_limit_events': list(RT_SOLVER_LIMIT_EVENTS),
                }
            ANALYSIS_DAYS_CONFIG = [f'{year}{m:02d}{d:02d}' for m in RUN_MONTHS_CONFIG for d in RUN_DAYS_CONFIG]
            exec(financial_table_source, globals(), globals())
            exec(financial_summary_source, globals(), globals())
        try:
            git_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
        except Exception:
            git_commit = 'unavailable'
        manifest = {
            'case': SENSITIVITY_CASE_NAME, 'controller': 'Rolling 24 h',
            'months': RUN_MONTHS_CONFIG, 'days': RUN_DAYS_CONFIG, 'modes': RUN_MODES,
            'forecasts': sensitivity_case['forecasts'], 'alpha_mode': AS_ACTIVATION_MODE,
            'passive_der_enabled': PASSIVE_DER_ENABLED, 'resource_tag': SENSITIVITY_RESOURCE_TAG,
            'building_input_file': str(BTM_BUILDING_FILE) if PASSIVE_DER_ENABLED else None,
            'pv_input_file': str(BTM_PV_FILE) if PASSIVE_DER_ENABLED else None,
            'run_kind': sensitivity_case.get('run_kind'),
            'btm_market_scope': 'retail meter only; excluded from DA/RT energy, AS capacity, and all WM revenue',
            'building_scale': BTM_LOAD_SCALE, 'pv_scale': BTM_PV_SCALE,
            'alpha_constants': AS_ACTIVATION_CONSTANTS, 'alpha_scale': AS_ACTIVATION_SCALE,
            'regulation_attenuation_file': str(AS_REGULATION_ATTENUATION_FILE),
            'regulation_attenuation_sha256': _sha256(AS_REGULATION_ATTENUATION_FILE),
            'building_input_sha256': _sha256(BTM_BUILDING_FILE) if PASSIVE_DER_ENABLED else None,
            'pv_input_sha256': _sha256(BTM_PV_FILE) if PASSIVE_DER_ENABLED else None,
            'contingency_activation_file': str(AS_CONTINGENCY_ACTIVATION_FILE) if AS_CONTINGENCY_ACTIVATION_FILE else None,
            'sp_nsp_interpretation': 'explicit event file if supplied; otherwise fixed documented assumptions',
            'price_unit_in_model_plots_and_tables': '$/kWh',
            'da_asmp_normalization': 'raw hourly $/MW divided by 1000; repeated across four 15-min steps and integrated with dt_h',
            'rt_asmp_normalization': 'raw 15-min binding $/MW divided by (1000*dt_h); integrated with dt_h',
            'da_award_granularity': 'hourly block; p_DA and all c_*_DA fixed within each four-step hour',
            'solver_audit': case_solver_audit,
            'ev_data_file': str(EV_DATA_FILE), 'ev_data_sha256': _sha256(EV_DATA_FILE),
            'baseline_dispatch_file': str(BASELINE_DISPATCH_FILE) if BASELINE_DISPATCH_FILE else None,
            'baseline_dispatch_sha256': _sha256(BASELINE_DISPATCH_FILE) if BASELINE_DISPATCH_FILE else None,
            'notebook': str(notebook_path), 'notebook_sha256': _sha256(notebook_path),
            'model_source_sha256': model_source_sha256,
            'git_commit': git_commit, 'output_directory': VERSION_DIR,
        }
        (Path(VERSION_DIR) / 'sensitivity_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
        print('Completed isolated sensitivity case:', VERSION_DIR)

    _all_enabled_cases = list(_enabled_sensitivity_cases)
    _callrate_cases = [c for c in _all_enabled_cases if c.get('run_kind') == 'july01_callrate_screen']
    if _callrate_cases:
        import textwrap
        exec(textwrap.dedent(r'''
    # Consolidated, reproducible July-01 comparison and validation gate.
    import matplotlib.pyplot as plt
    comparison_rows, audit_rows, product_rows = [], [], []
    reference_baseline = None
    enabled_cases = _callrate_cases
    for case in enabled_cases:
        case_name = _safe_sensitivity_slug(case['name'])
        resource_tag = 'ev_bess_pv_building_retail' if case.get('passive_der_enabled', False) else 'ev_bess'
        case_dir = SENSITIVITY_OUTPUT_ROOT / case_name / resource_tag
        manifest = json.loads((case_dir / 'sensitivity_manifest.json').read_text())
        trace_path = next(case_dir.glob('Validation_Traces/*_full/rolling_validation_trace_20250701.csv'))
        wm_path = next(case_dir.glob('Plots/Solver_RT_Choices/*_full/20250701/RT_WM_profit_breakdown_full.csv'))
        market_summary_path = next(case_dir.glob('Plots/Solver_RT_Choices/*_full/20250701/RT_market_daily_summary_with_prices.csv'))
        financial_path = next(case_dir.glob('Plots/Cost/*/daily_financial_detail.csv'))
        trace = pd.read_csv(trace_path)
        if reference_baseline is None:
            reference_baseline = trace['baseline_kW'].to_numpy(float)
        baseline_difference = float(np.max(np.abs(trace['baseline_kW'].to_numpy(float) - reference_baseline)))
        wm = pd.read_csv(wm_path)
        alpha_label = 'CAISO hourly' if manifest['alpha_mode'] == 'caiso_regulation' else f"Constant RU/RD={manifest['alpha_constants']['RU']:.1f}"
        market_summary = pd.read_csv(market_summary_path)
        market_summary.insert(0, 'Alpha', alpha_label)
        market_summary.insert(0, 'Case', case_name)
        product_rows.append(market_summary)
        financial = pd.read_csv(financial_path)
        financial = financial[(financial['Date'].astype(str) == '2025-07-01') & (financial['Case'] == 'Both')].iloc[-1]
        solver = manifest['solver_audit'][next(iter(manifest['solver_audit']))]
        wm_sums = {col: float(wm[col].sum()) for col in [
            'BESS_Energy_Profit', 'EV_Energy_Profit',
            'BESS_Capacity_Profit', 'EV_Capacity_Profit', 'profit_sum_all_products']}
        comparison_rows.append({
            'Case': case_name, 'Portfolio': 'EV+BESS with passive PV/building retail meter' if manifest['passive_der_enabled'] else 'EV+BESS',
            'Alpha': alpha_label,
            'Total Revenue': float(financial['Total Revenue']), 'WM Revenue': wm_sums['profit_sum_all_products'],
            'TOU Cost': float(financial['TOU Cost']), 'PD Cost': float(financial['PD Cost']),
            'NCD Cost': float(financial['NCD Cost']), 'EV Revenue': float(financial['EV Revenue']),
            'WM BESS Energy': wm_sums['BESS_Energy_Profit'], 'WM EV Energy': wm_sums['EV_Energy_Profit'],
            'WM BESS Capacity': wm_sums['BESS_Capacity_Profit'], 'WM EV Capacity': wm_sums['EV_Capacity_Profit'],
            'EV energy (kWh)': float(trace['p_EV_kW'].sum() * dt_h), 'Terminal SOC': float(trace['SOC'].iloc[-1]),
        })
        financial_identity = abs(float(financial['Total Revenue']) - sum(float(financial[c]) for c in ['WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost', 'EV Revenue']))
        audit = {
            'Case': case_name, 'Trace rows checked': len(trace), 'Expected optimal steps': 96 * len(case['days']),
            'Optimal steps': int(solver['status_counts'].get('optimal', 0)),
            'Time-limit events': len(solver['time_limit_events']),
            'Meter residual max (kW)': float(trace['meter_balance_residual_kW'].abs().max()),
            'Passive WM position max (kW)': float(trace['passive_WM_position_kW'].abs().max()),
            'Controlled identity max (kW)': float((trace['p_actual_kW'] - trace['p_controlled_residual_kW']).abs().max()),
            'Up slack min (kW)': float(trace['actual_up_capability_slack_kW'].min()),
            'Down slack min (kW)': float(trace['actual_down_capability_slack_kW'].min()),
            'AS actual min (kW)': float(trace[['c_RU_actual_kW','c_RD_actual_kW','c_SP_actual_kW','c_NSP_actual_kW']].min().min()),
            'SOC min': float(trace['SOC'].min()), 'SOC max': float(trace['SOC'].max()),
            'Terminal SOC': float(trace['SOC'].iloc[-1]), 'Terminal target required': len(case['days']) == 1,
            'EV energy (kWh)': float(trace['p_EV_kW'].sum() * dt_h),
            'Financial identity residual ($)': financial_identity,
            'Baseline difference max (kW)': baseline_difference,
            'WM attribution residual ($)': abs(wm_sums['profit_sum_all_products'] - wm_sums['BESS_Energy_Profit'] - wm_sums['EV_Energy_Profit'] - wm_sums['BESS_Capacity_Profit'] - wm_sums['EV_Capacity_Profit']),
        }
        audit['PASS'] = bool(audit['Trace rows checked'] == 96 and audit['Optimal steps'] == audit['Expected optimal steps'] and audit['Time-limit events'] == 0
            and audit['Meter residual max (kW)'] <= 1e-6 and audit['Passive WM position max (kW)'] <= 1e-9
            and audit['Controlled identity max (kW)'] <= 1e-6 and audit['Up slack min (kW)'] >= -1e-6
            and audit['Down slack min (kW)'] >= -1e-6 and audit['AS actual min (kW)'] >= -1e-6
            and audit['SOC min'] >= SOC_BESS_min - 1e-6 and audit['SOC max'] <= SOC_BESS_max + 1e-6
            and ((not audit['Terminal target required']) or abs(audit['Terminal SOC'] - 0.5) <= 1e-6) and financial_identity <= 0.02
            and audit['Baseline difference max (kW)'] <= 1e-9 and audit['WM attribution residual ($)'] <= 1e-3)
        audit_rows.append(audit)

    comparison_df = pd.DataFrame(comparison_rows)
    audit_df = pd.DataFrame(audit_rows)
    comparison_dir = SENSITIVITY_OUTPUT_ROOT / 'july01_callrate_comparison'
    comparison_dir.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(comparison_dir / 'july01_callrate_financial_comparison.csv', index=False, float_format='%.6f')
    audit_df.to_csv(comparison_dir / 'july01_callrate_validation_audit.csv', index=False, float_format='%.9f')
    pd.concat(product_rows, ignore_index=True).to_csv(comparison_dir / 'july01_callrate_product_revenue_with_prices.csv', index=False, float_format='%.9f')
    if not audit_df['PASS'].all():
        raise AssertionError('One or more July-01 sensitivity validation gates failed; inspect the audit CSV.')
    ev_energy_span = comparison_df['EV energy (kWh)'].max() - comparison_df['EV energy (kWh)'].min()
    if ev_energy_span > 1e-6:
        raise AssertionError(f'EV daily energy differs across matched cases by {ev_energy_span:.3e} kWh')

    ordered_names = [_safe_sensitivity_slug(c['name']) for c in enabled_cases]
    plot_df = comparison_df.set_index('Case').loc[ordered_names]
    labels = [label.replace('Constant RU/RD=', '') if label.startswith('Constant') else 'CAISO\nhourly' for label in plot_df['Alpha']]
    x = np.arange(len(plot_df))
    fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
    components = [('WM BESS Energy','#377eb8'), ('WM EV Energy','#4daf4a'), ('WM BESS Capacity','#984ea3'),
                  ('WM EV Capacity','#ff7f00')]
    pos = np.zeros(len(plot_df)); neg = np.zeros(len(plot_df))
    for col, color in components:
        vals = plot_df[col].to_numpy(float); bottom = np.where(vals >= 0, pos, neg)
        axes[0,0].bar(x, vals, bottom=bottom, label=col.replace('WM ', ''), color=color)
        pos += np.where(vals >= 0, vals, 0); neg += np.where(vals < 0, vals, 0)
    axes[0,0].axhline(0, color='black', lw=0.8); axes[0,0].set_title('(a) Wholesale-market value decomposition')
    axes[0,0].set_ylabel('US$/day'); axes[0,0].legend(ncol=3, fontsize=8, loc='upper center')
    retail_cols = [('TOU Cost','#1f78b4'), ('PD Cost','#33a02c'), ('NCD Cost','#e31a1c')]
    bottom = np.zeros(len(plot_df))
    for col, color in retail_cols:
        vals = plot_df[col].to_numpy(float); axes[0,1].bar(x, vals, bottom=bottom, label=col, color=color); bottom += vals
    axes[0,1].axhline(0, color='black', lw=0.8); axes[0,1].set_title('(b) Retail-cost decomposition')
    axes[0,1].set_ylabel('US$/day'); axes[0,1].legend(ncol=3, fontsize=8, loc='lower center')
    total_vals = plot_df['Total Revenue'].to_numpy(float)
    bars = axes[1,0].bar(x, total_vals, color=plt.cm.viridis(np.linspace(0.15, 0.85, len(plot_df))))
    axes[1,0].axhline(0, color='black', lw=0.8); axes[1,0].set_title('(c) Net system revenue')
    axes[1,0].set_ylabel('US$/day'); axes[1,0].bar_label(bars, fmt='$%.0f', padding=3, fontsize=9)
    axes[1,1].plot(x, plot_df['WM Revenue'].to_numpy(float), marker='o', linewidth=2, color='#31a354')
    axes[1,1].axhline(0, color='black', lw=0.8)
    axes[1,1].set_title('(d) Net wholesale revenue across call-rate cases')
    axes[1,1].set_ylabel('US$/day'); axes[1,1].grid(alpha=0.25)
    for ax in axes[0, :]: ax.set_xticks(x); ax.set_xticklabels([])
    for ax in axes[1, :]:
        ax.set_xticks(x, labels)
        ax.set_xlabel(r'Fixed $\alpha_{RU}=\alpha_{RD}$ (except CAISO)')
    fig.suptitle('Rolling 24-h MPC regulation-activation sensitivity | passive PV/building retail meter | 2025-07-01', fontsize=16, fontweight='bold')
    fig.text(0.5, -0.015, 'PV/building affect retail cost only; WM value is EV+BESS. CAISO uses published Q3 hourly SOC attenuation factors, not realized AGC dispatch.', ha='center', fontsize=9)
    comparison_png = comparison_dir / 'july01_callrate_financial_comparison.png'
    fig.savefig(comparison_png, dpi=300, bbox_inches='tight'); plt.close(fig)
    print('Sensitivity comparison saved to:', comparison_dir)
    print(audit_df[['Case','Optimal steps','Time-limit events','PASS']].to_string(index=False))
        '''))

    # Consolidate the unit-consistent three-day call-rate/attenuation screen.
    _unitfix_cases = [c for c in _all_enabled_cases if c.get('run_kind') == 'callrate_unitfix_multidate']
    if _unitfix_cases:
        _cmp_dir = SENSITIVITY_OUTPUT_ROOT / 'callrate_unitfix_multidate_comparison'
        _cmp_dir.mkdir(parents=True, exist_ok=True)
        _financial_rows, _case_audit_rows = [], []
        _daily_audit_rows, _product_rows, _interval_rows = [], [], []
        for _case in _unitfix_cases:
            _case_name = _safe_sensitivity_slug(_case['name'])
            _case_dir = SENSITIVITY_OUTPUT_ROOT / _case_name / 'ev_bess_pv_building_retail'
            _manifest = json.loads((_case_dir / 'sensitivity_manifest.json').read_text())
            _alpha_label = ('CAISO attenuation' if _manifest['alpha_mode'] == 'caiso_regulation'
                            else f"Fixed {float(_manifest['alpha_constants']['RU']):.1f}")
            _audit_key = next(iter(_manifest['solver_audit']))
            _solver = _manifest['solver_audit'][_audit_key]
            _expected_steps = 96 * len(_case['days'])
            _case_pass = (int(_solver['status_counts'].get('optimal', 0)) == _expected_steps
                          and len(_solver['time_limit_events']) == 0)
            _case_audit_rows.append({
                'Case': _case_name, 'Alpha': _alpha_label, 'Expected optimal steps': _expected_steps,
                'Optimal steps': int(_solver['status_counts'].get('optimal', 0)),
                'Time-limit events': len(_solver['time_limit_events']), 'PASS': _case_pass,
            })
            _forecast_tag = '2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival'
            _financial_path = _case_dir / 'Plots/Cost' / _forecast_tag / 'daily_financial_detail.csv'
            _financial = pd.read_csv(_financial_path)
            for _day in _case['days']:
                _date_key = f'202507{int(_day):02d}'
                _date_iso = f'2025-07-{int(_day):02d}'
                _plot_dir = _case_dir / 'Plots/Solver_RT_Choices' / f'{_forecast_tag}_full' / _date_key
                _trace_path = _case_dir / 'Validation_Traces' / f'{_forecast_tag}_full' / f'rolling_validation_trace_{_date_key}.csv'
                _wm = pd.read_csv(_plot_dir / 'RT_WM_profit_breakdown_full.csv')
                _product = pd.read_csv(_plot_dir / 'RT_market_daily_summary_with_prices.csv')
                _interval = pd.read_csv(_plot_dir / 'RT_market_interval_audit_with_prices.csv')
                _trace = pd.read_csv(_trace_path)
                _product.insert(0, 'Alpha scenario', _alpha_label); _product.insert(0, 'Case', _case_name)
                _interval = _interval.rename(columns={'Alpha': 'Interval activation factor'})
                _interval.insert(0, 'Alpha scenario', _alpha_label); _interval.insert(0, 'Case', _case_name)
                _product_rows.append(_product); _interval_rows.append(_interval)
                _fin = _financial[(_financial['Date'].astype(str) == _date_iso) & (_financial['Case'] == 'Both')].iloc[-1]
                _wm_sums = {c: float(_wm[c].sum()) for c in [
                    'BESS_Energy_Profit', 'EV_Energy_Profit',
                    'BESS_Capacity_Profit', 'EV_Capacity_Profit', 'profit_sum_all_products']}
                _financial_rows.append({
                    'Date': _date_iso, 'Case': _case_name, 'Alpha': _alpha_label,
                    'Total Revenue': float(_fin['Total Revenue']), 'WM Revenue': _wm_sums['profit_sum_all_products'],
                    'TOU Cost': float(_fin['TOU Cost']), 'PD Cost': float(_fin['PD Cost']),
                    'NCD Cost': float(_fin['NCD Cost']), 'EV Revenue': float(_fin['EV Revenue']),
                    'WM BESS Energy': _wm_sums['BESS_Energy_Profit'], 'WM EV Energy': _wm_sums['EV_Energy_Profit'],
                    'WM BESS Capacity': _wm_sums['BESS_Capacity_Profit'], 'WM EV Capacity': _wm_sums['EV_Capacity_Profit'],
                })
                _identity = abs(float(_fin['Total Revenue']) - sum(float(_fin[c]) for c in [
                    'WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost', 'EV Revenue']))
                _daily_pass = bool(len(_trace) == 96
                    and float(_trace['meter_balance_residual_kW'].abs().max()) <= 1e-6
                    and float(_trace['passive_WM_position_kW'].abs().max()) <= 1e-9
                    and float(_trace['actual_up_capability_slack_kW'].min()) >= -1e-6
                    and float(_trace['actual_down_capability_slack_kW'].min()) >= -1e-6
                    and float(_trace[['c_RU_actual_kW','c_RD_actual_kW','c_SP_actual_kW','c_NSP_actual_kW']].min().min()) >= -1e-6
                    and _identity <= 0.02)
                _daily_audit_rows.append({
                    'Date': _date_iso, 'Case': _case_name, 'Alpha': _alpha_label, 'Trace rows': len(_trace),
                    'Meter residual max (kW)': float(_trace['meter_balance_residual_kW'].abs().max()),
                    'Passive WM position max (kW)': float(_trace['passive_WM_position_kW'].abs().max()),
                    'Up slack min (kW)': float(_trace['actual_up_capability_slack_kW'].min()),
                    'Down slack min (kW)': float(_trace['actual_down_capability_slack_kW'].min()),
                    'AS actual min (kW)': float(_trace[['c_RU_actual_kW','c_RD_actual_kW','c_SP_actual_kW','c_NSP_actual_kW']].min().min()),
                    'Financial identity residual ($)': _identity, 'PASS': _daily_pass,
                })
        _financial_df = pd.DataFrame(_financial_rows)
        _case_audit_df = pd.DataFrame(_case_audit_rows)
        _daily_audit_df = pd.DataFrame(_daily_audit_rows)
        _financial_df.to_csv(_cmp_dir / 'callrate_multidate_financial_comparison.csv', index=False, float_format='%.8f')
        _case_audit_df.to_csv(_cmp_dir / 'callrate_multidate_solver_audit.csv', index=False)
        _daily_audit_df.to_csv(_cmp_dir / 'callrate_multidate_daily_validation_audit.csv', index=False, float_format='%.9f')
        pd.concat(_product_rows, ignore_index=True).to_csv(_cmp_dir / 'callrate_multidate_daily_product_revenue_with_prices.csv', index=False, float_format='%.9f')
        pd.concat(_interval_rows, ignore_index=True).to_csv(_cmp_dir / 'callrate_multidate_interval_market_audit_with_prices.csv', index=False, float_format='%.9f')
        if not (_case_audit_df['PASS'].all() and _daily_audit_df['PASS'].all()):
            raise AssertionError('Unit-consistent multi-date sensitivity validation failed')
        _alpha_order_all = [f'Fixed {v:.1f}' for v in [0.0,0.1,0.3,0.5,0.7,1.0]] + ['CAISO attenuation']
        _alpha_order = [s for s in _alpha_order_all if s in set(_financial_df['Alpha'])]
        _agg = _financial_df.groupby('Alpha', as_index=False)[[
            'Total Revenue','WM Revenue','WM BESS Energy','WM EV Energy','WM BESS Capacity','WM EV Capacity']].sum().set_index('Alpha').loc[_alpha_order]
        _x = np.arange(len(_agg)); _labels = [s.replace('Fixed ','') if s.startswith('Fixed ') else 'CAISO\nattenuation' for s in _agg.index]
        _fig, _axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
        for _date, _g in _financial_df.groupby('Date'):
            _g = _g.set_index('Alpha').loc[_alpha_order]
            _axes[0,0].plot(_x, _g['WM Revenue'], marker='o', linewidth=2, label=_date)
            _axes[0,1].plot(_x, _g['Total Revenue'], marker='o', linewidth=2, label=_date)
        _axes[0,0].set_title('(a) Daily wholesale-market revenue'); _axes[0,0].set_ylabel('US$/day')
        _axes[0,1].set_title('(b) Daily total net revenue'); _axes[0,1].set_ylabel('US$/day')
        _axes[0,0].legend(ncol=3, fontsize=8); _axes[0,1].legend(ncol=3, fontsize=8)
        _bottom = np.zeros(len(_agg))
        for _col, _color in [('WM BESS Energy','#377eb8'),('WM EV Energy','#4daf4a'),('WM BESS Capacity','#984ea3'),('WM EV Capacity','#ff7f00')]:
            _vals = _agg[_col].to_numpy(float); _axes[1,0].bar(_x, _vals, bottom=_bottom, label=_col.replace('WM ',''), color=_color); _bottom += _vals
        _axes[1,0].set_title('(c) Three-day WM decomposition'); _axes[1,0].set_ylabel('US$')
        _axes[1,0].legend(ncol=2, fontsize=8)
        _ref_label = 'CAISO attenuation' if 'CAISO attenuation' in _agg.index else _agg.index[0]
        _ref = float(_agg.loc[_ref_label,'WM Revenue'])
        _axes[1,1].bar(_x, _agg['WM Revenue'].to_numpy(float) - _ref, color='#31a354')
        _axes[1,1].axhline(0, color='black', linewidth=0.8); _axes[1,1].set_title('(d) WM revenue relative to CAISO attenuation')
        _axes[1,1].set_ylabel('US$ over three days')
        for _ax in _axes.flat:
            _ax.set_xticks(_x, _labels); _ax.grid(axis='y', alpha=0.25)
        _fig.suptitle('Rolling 24-h MPC activation-factor sensitivity | passive PV/building retail meter | 2025-07-01 to 2025-07-03', fontsize=15, fontweight='bold')
        _fig.savefig(_cmp_dir / 'callrate_multidate_financial_comparison.png', dpi=300, bbox_inches='tight'); plt.close(_fig)
        print('Unit-consistent multi-date comparison saved to:', _cmp_dir)

    # Consolidate the two full-July forecast runs without rerunning optimization.
    for _monthly_case in [c for c in _all_enabled_cases if c.get('run_kind') == 'monthly_forecast_comparison']:
        _monthly_case_dir = SENSITIVITY_OUTPUT_ROOT / _safe_sensitivity_slug(_monthly_case['name']) / 'ev_bess_pv_building_retail'
        _monthly_rows, _daily_rows, _daily_product_rows, _product_rows = [], [], [], []
        for _fs, _fn, _fa in _monthly_case['forecasts']:
            _forecast_label = 'Perfect' if _fs.startswith('Perfect') and _fn.startswith('Perfect') else 'Persistence'
            _forecast_tag = f'2025_{_fs}_{_fn}_{_fa}'
            _cost_dir = _monthly_case_dir / 'Plots/Cost' / _forecast_tag
            _monthly_df = pd.read_csv(_cost_dir / 'monthly_financial_summary.csv')
            _both = _monthly_df[_monthly_df['Case / US$'] == 'Both'].iloc[-1].to_dict()
            _both['Forecast'] = _forecast_label
            _monthly_rows.append(_both)
            _daily_df = pd.read_csv(_cost_dir / 'daily_financial_detail.csv')
            _daily_df = _daily_df[_daily_df['Case'] == 'Both'].copy()
            _daily_df.insert(1, 'Forecast', _forecast_label)
            _daily_rows.append(_daily_df)
            _audit_files = sorted((_monthly_case_dir / 'Plots/Solver_RT_Choices' / f'{_forecast_tag}_full').glob('*/RT_market_daily_summary_with_prices.csv'))
            if len(_audit_files) != len(_monthly_case['days']):
                raise ValueError(f'{_forecast_label}: expected {len(_monthly_case["days"])} daily market audits, found {len(_audit_files)}')
            _audit_df = pd.concat([pd.read_csv(p) for p in _audit_files], ignore_index=True)
            _audit_df.insert(0, 'Forecast', _forecast_label)
            _daily_product_rows.append(_audit_df.copy())
            _sum_cols = [c for c in _audit_df.columns if c.endswith('($)') or c.endswith('(kWh)') or c.endswith('(kWh or kW-h)')]
            _monthly_product = _audit_df.groupby('Product', as_index=False)[_sum_cols].sum()
            _monthly_product.insert(0, 'Forecast', _forecast_label)
            _product_rows.append(_monthly_product)
        pd.DataFrame(_monthly_rows).to_csv(_monthly_case_dir / 'july2025_monthly_forecast_comparison.csv', index=False, float_format='%.6f')
        pd.concat(_daily_rows, ignore_index=True).to_csv(_monthly_case_dir / 'july2025_daily_forecast_comparison.csv', index=False, float_format='%.6f')
        pd.concat(_daily_product_rows, ignore_index=True).to_csv(_monthly_case_dir / 'july2025_daily_product_revenue_with_prices.csv', index=False, float_format='%.9f')
        pd.concat(_product_rows, ignore_index=True).to_csv(_monthly_case_dir / 'july2025_monthly_product_revenue_with_prices.csv', index=False, float_format='%.6f')
        print('Monthly forecast comparison saved to:', _monthly_case_dir)

    # Formal July 2x2: forecast information x passive PV/building at the retail meter.
    _btm_2x2_cases = [c for c in _all_enabled_cases if c.get('run_kind') == 'july_btm_2x2']
    if _btm_2x2_cases:
        import matplotlib.pyplot as plt
        _cmp_dir = SENSITIVITY_OUTPUT_ROOT / 'july2025_forecast_btm_2x2_comparison'
        _cmp_dir.mkdir(parents=True, exist_ok=True)
        _monthly_rows, _daily_rows, _daily_product_rows, _monthly_product_rows = [], [], [], []
        _validation_rows, _contrast_rows = [], []
        _matched = {}
        for _case in _btm_2x2_cases:
            _resource_tag = ('ev_bess_pv_building_retail' if _case.get('passive_der_enabled', False) else 'ev_bess')
            _resource_label = ('EV+BESS+passive PV/building' if _case.get('passive_der_enabled', False) else 'EV+BESS')
            _case_dir = SENSITIVITY_OUTPUT_ROOT / _safe_sensitivity_slug(_case['name']) / _resource_tag
            _manifest = json.loads((_case_dir / 'sensitivity_manifest.json').read_text())
            _expected_dates = [f'2025-07-{int(d):02d}' for d in _case['days']]
            _expected_date_keys = [f'202507{int(d):02d}' for d in _case['days']]
            for _fs, _fn, _fa in _case['forecasts']:
                _forecast_label = ('Perfect' if _fs.startswith('Perfect') and _fn.startswith('Perfect') else 'Persistence')
                _forecast_tag = f'2025_{_fs}_{_fn}_{_fa}'
                _audit_key = f'{_fs}|{_fn}|{_fa}|full'
                _solver = _manifest['solver_audit'][_audit_key]
                _cost_dir = _case_dir / 'Plots/Cost' / _forecast_tag
                _monthly_df = pd.read_csv(_cost_dir / 'monthly_financial_summary.csv')
                _monthly_both = _monthly_df[_monthly_df['Case / US$'] == 'Both'].iloc[-1].to_dict()
                _monthly_both = {'Resource case': _resource_label, 'Forecast': _forecast_label, **_monthly_both}
                _monthly_rows.append(_monthly_both)
                _daily_df = pd.read_csv(_cost_dir / 'daily_financial_detail.csv')
                _daily_df = _daily_df[(_daily_df['Case'] == 'Both') & (_daily_df['Date'].astype(str).isin(_expected_dates))].copy()
                _daily_df.insert(0, 'Forecast', _forecast_label); _daily_df.insert(0, 'Resource case', _resource_label)
                _daily_rows.append(_daily_df)
                _trace_dir = _case_dir / 'Validation_Traces' / f'{_forecast_tag}_full'
                _plot_dir = _case_dir / 'Plots/Solver_RT_Choices' / f'{_forecast_tag}_full'
                _trace_files = [_trace_dir / f'rolling_validation_trace_{key}.csv' for key in _expected_date_keys]
                _product_files = [_plot_dir / key / 'RT_market_daily_summary_with_prices.csv' for key in _expected_date_keys]
                _missing = [str(p) for p in _trace_files + _product_files if not p.exists()]
                if _missing:
                    raise FileNotFoundError('Missing July 2x2 artifacts: ' + '; '.join(_missing[:5]))
                _trace = pd.concat([pd.read_csv(p) for p in _trace_files], ignore_index=True)
                _product = pd.concat([pd.read_csv(p) for p in _product_files], ignore_index=True)
                _product.insert(0, 'Forecast', _forecast_label); _product.insert(0, 'Resource case', _resource_label)
                _daily_product_rows.append(_product)
                _sum_cols = [c for c in _product.columns if c.endswith('($)') or c.endswith('(kWh)') or c.endswith('(kWh or kW-h)')]
                _monthly_product = _product.groupby('Product', as_index=False)[_sum_cols].sum()
                _monthly_product.insert(0, 'Forecast', _forecast_label); _monthly_product.insert(0, 'Resource case', _resource_label)
                _monthly_product_rows.append(_monthly_product)
                _identity = (_daily_df['Total Revenue'] - _daily_df[['WM Revenue','TOU Cost','PD Cost','NCD Cost','EV Revenue']].sum(axis=1)).abs()
                _expected_steps = 96 * len(_case['days'])
                _timestamp = pd.to_datetime(_trace['Interval start'], errors='coerce')
                _load_max = float(_trace['p_load_kW'].abs().max()); _pv_max = float(_trace['p_PV_kW'].abs().max())
                _passive_data_ok = ((_load_max > 1e-6 and _pv_max > 1e-6) if _case.get('passive_der_enabled', False)
                                    else (_load_max <= 1e-9 and _pv_max <= 1e-9))
                _validation = {
                    'Resource case': _resource_label, 'Forecast': _forecast_label,
                    'Expected days': len(_case['days']), 'Daily financial rows': len(_daily_df),
                    'Expected RT steps': _expected_steps, 'Optimal RT steps': int(_solver['status_counts'].get('optimal', 0)),
                    'Time-limit events': len(_solver['time_limit_events']), 'Trace rows': len(_trace),
                    'Unique timestamps': int(_timestamp.nunique()), 'Invalid timestamps': int(_timestamp.isna().sum()),
                    'Meter residual max (kW)': float(_trace['meter_balance_residual_kW'].abs().max()),
                    'Passive WM position max (kW)': float(_trace['passive_WM_position_kW'].abs().max()),
                    'Controlled identity max (kW)': float((_trace['p_actual_kW'] - _trace['p_controlled_residual_kW']).abs().max()),
                    'Up slack min (kW)': float(_trace['actual_up_capability_slack_kW'].min()),
                    'Down slack min (kW)': float(_trace['actual_down_capability_slack_kW'].min()),
                    'AS actual min (kW)': float(_trace[['c_RU_actual_kW','c_RD_actual_kW','c_SP_actual_kW','c_NSP_actual_kW']].min().min()),
                    'SOC min': float(_trace['SOC'].min()), 'SOC max': float(_trace['SOC'].max()),
                    'EV energy (kWh)': float(_trace['p_EV_kW'].sum() * dt_h),
                    'Building max abs (kW)': _load_max, 'PV max abs (kW)': _pv_max,
                    'Financial identity residual max ($)': float(_identity.max()), 'Passive data check': _passive_data_ok,
                }
                _validation['PASS'] = bool(
                    _validation['Daily financial rows'] == _validation['Expected days']
                    and _validation['Optimal RT steps'] == _validation['Expected RT steps']
                    and _validation['Time-limit events'] == 0 and _validation['Trace rows'] == _expected_steps
                    and _validation['Unique timestamps'] == _expected_steps and _validation['Invalid timestamps'] == 0
                    and _validation['Meter residual max (kW)'] <= 1e-6
                    and _validation['Passive WM position max (kW)'] <= 1e-9
                    and _validation['Controlled identity max (kW)'] <= 1e-6
                    and _validation['Up slack min (kW)'] >= -1e-6 and _validation['Down slack min (kW)'] >= -1e-6
                    and _validation['AS actual min (kW)'] >= -1e-6
                    and _validation['SOC min'] >= SOC_BESS_min - 1e-6 and _validation['SOC max'] <= SOC_BESS_max + 1e-6
                    and _validation['Financial identity residual max ($)'] <= 0.02 and _passive_data_ok)
                _validation_rows.append(_validation)
                _matched[(_resource_tag, _forecast_label)] = {
                    'initial_baseline': _trace['baseline_kW'].iloc[:96].to_numpy(float),
                    'full_baseline_path': _trace['baseline_kW'].to_numpy(float),
                    'ev_energy_kWh': _validation['EV energy (kWh)'],
                }
        _monthly_out = pd.DataFrame(_monthly_rows)
        _daily_out = pd.concat(_daily_rows, ignore_index=True)
        _validation_out = pd.DataFrame(_validation_rows)
        for _forecast_label in ['Persistence', 'Perfect']:
            _off = _matched[('ev_bess', _forecast_label)]; _on = _matched[('ev_bess_pv_building_retail', _forecast_label)]
            # The first-day baseline must match.  Later 10/4-day baselines may diverge
            # because the passive retail load changes optimized EV/BESS dispatch history.
            _initial_baseline_diff = float(np.max(np.abs(_off['initial_baseline'] - _on['initial_baseline'])))
            _full_baseline_path_diff = float(np.max(np.abs(_off['full_baseline_path'] - _on['full_baseline_path'])))
            _ev_energy_diff = float(abs(_off['ev_energy_kWh'] - _on['ev_energy_kWh']))
            _contrast_rows.append({'Check': f'{_forecast_label}: BTM on versus off',
                                   'Initial baseline difference max (kW)': _initial_baseline_diff,
                                   'Full-period baseline path difference max (kW)': _full_baseline_path_diff,
                                   'EV energy difference (kWh)': _ev_energy_diff,
                                   'PASS': bool(_initial_baseline_diff <= 1e-9 and _ev_energy_diff <= 1e-4)})
        _financial_cols = ['Total Revenue','WM Revenue','TOU Cost','PD Cost','NCD Cost','EV Revenue']
        for _resource_label in _monthly_out['Resource case'].unique():
            _g = _monthly_out[_monthly_out['Resource case'] == _resource_label].set_index('Forecast')
            for _metric in _financial_cols:
                _contrast_rows.append({'Check': f'{_resource_label}: Perfect - Persistence', 'Metric': _metric,
                                       'Difference (US$)': float(_g.loc['Perfect', _metric] - _g.loc['Persistence', _metric])})
        for _forecast_label in ['Persistence','Perfect']:
            _g = _monthly_out[_monthly_out['Forecast'] == _forecast_label].set_index('Resource case')
            for _metric in _financial_cols:
                _contrast_rows.append({'Check': f'{_forecast_label}: passive BTM - EV+BESS', 'Metric': _metric,
                                       'Difference (US$)': float(_g.loc['EV+BESS+passive PV/building', _metric] - _g.loc['EV+BESS', _metric])})
        _contrast_out = pd.DataFrame(_contrast_rows)
        _monthly_out.to_csv(_cmp_dir / 'july2025_btm_forecast_2x2_monthly.csv', index=False, float_format='%.6f')
        _daily_out.to_csv(_cmp_dir / 'july2025_btm_forecast_2x2_daily.csv', index=False, float_format='%.6f')
        pd.concat(_daily_product_rows, ignore_index=True).to_csv(_cmp_dir / 'july2025_btm_forecast_2x2_daily_product_revenue_with_prices.csv', index=False, float_format='%.9f')
        pd.concat(_monthly_product_rows, ignore_index=True).to_csv(_cmp_dir / 'july2025_btm_forecast_2x2_monthly_product_revenue_with_prices.csv', index=False, float_format='%.6f')
        _validation_out.to_csv(_cmp_dir / 'july2025_btm_forecast_2x2_validation.csv', index=False, float_format='%.9f')
        _contrast_out.to_csv(_cmp_dir / 'july2025_btm_forecast_2x2_contrasts.csv', index=False, float_format='%.6f')
        if not (_validation_out['PASS'].all() and _contrast_out.loc[_contrast_out['PASS'].notna(), 'PASS'].all()):
            raise AssertionError('July forecast x BTM 2x2 validation failed; inspect the comparison audit CSVs')
        _order = pd.MultiIndex.from_product([['EV+BESS','EV+BESS+passive PV/building'], ['Persistence','Perfect']],
                                            names=['Resource case','Forecast'])
        _plot = _monthly_out.set_index(['Resource case','Forecast']).loc[_order]
        _x = np.arange(len(_plot)); _labels = ['EV+BESS\nPersistence','EV+BESS\nPerfect','BTM\nPersistence','BTM\nPerfect']
        _fig, _axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
        _bars = _axes[0].bar(_x, _plot['Total Revenue'].to_numpy(float), color=['#4c78a8','#72b7b2','#f58518','#e45756'])
        _axes[0].axhline(0, color='black', lw=0.8); _axes[0].set_title('(a) Monthly net revenue'); _axes[0].set_ylabel('US$/month')
        _axes[0].bar_label(_bars, fmt='$%.0f', padding=3, fontsize=8)
        _wm = _plot['WM Revenue'].to_numpy(float); _tou = _plot['TOU Cost'].to_numpy(float)
        _axes[1].bar(_x - 0.18, _wm, width=0.36, label='WM revenue', color='#59a14f')
        _axes[1].bar(_x + 0.18, _tou, width=0.36, label='TOU cost', color='#e15759')
        _axes[1].axhline(0, color='black', lw=0.8); _axes[1].set_title('(b) Principal market components'); _axes[1].set_ylabel('US$/month')
        _axes[1].legend(frameon=False)
        for _ax in _axes:
            _ax.set_xticks(_x, _labels); _ax.grid(axis='y', alpha=0.25)
        _fig.suptitle('Rolling 24-h MPC | July 2025 forecast x passive-BTM comparison', fontsize=14, fontweight='bold')
        _fig.savefig(_cmp_dir / 'july2025_btm_forecast_2x2_financial_comparison.png', dpi=300, bbox_inches='tight'); plt.close(_fig)
        print('July forecast x BTM 2x2 comparison saved to:', _cmp_dir)
        print(_validation_out[['Resource case','Forecast','Optimal RT steps','Time-limit events','PASS']].to_string(index=False))


EV sensitivity input: 2025Data/EV_data/UCSD_AllSites_Merge_PostProcessedSession_QC.csv
Number of Intervals: 374890
Test year: 2025
Date range: 2025-01-01 03:15:00 to 2025-07-31 21:00:00
Unique sites: 23
Unique chargers: 440
Passive building/PV retail meter enabled: True; output: Sensitivity_Results/Rolling_24h/july2025_retail_btm_caiso_alpha/ev_bess_pv_building_retail
All dispatch files from last implementation have been deleted.

Test month: 7
Month 7: M_Th_NCD init=0.0 kW, M_Th_PD init=0.0 kW (initialized to 0)



Processing days:   0%|          | 0/31 [00:00<?, ?it/s]


Processing days:   3%|▎         | 1/31 [00:27<13:39, 27.31s/it]


Processing days:   6%|▋         | 2/31 [00:53<12:46, 26.45s/it]

Skipping 6-panel figure for 20250702 (PLOT_DAILY_6PANEL_DATES=['20250701'])


...


Processing days: 100%|██████████| 31/31 [15:40<00:00, 29.53s/it]


Processing days: 100%|██████████| 31/31 [15:40<00:00, 30.34s/it]

Skipping 6-panel figure for 20250731 (PLOT_DAILY_6PANEL_DATES=['20250701'])
Main loop runtime: 940.52 seconds
Saved 93 files across 31 days.
Financial summary restricted to run dates: ['20250701', '20250702', '20250703', '20250704', '20250705', '20250706', '20250707', '20250708', '20250709', '20250710', '20250711', '20250712', '20250713', '20250714', '20250715', '20250716', '20250717', '20250718', '20250719', '20250720', '20250721', '20250722', '20250723', '20250724', '20250725', '20250726', '20250727', '20250728', '20250729', '20250730', '20250731']



  Run-Period Financial Summary  (US$)
            Total Revenue  WM Revenue   TOU Cost  PD Cost  NCD Cost  EV Revenue
Case / US$                                                                     
Both           -252575.08     4127.63 -254081.05  -3949.4 -20620.06     21947.8
Daily detail CSV saved to: Sensitivity_Results/Rolling_24h/july2025_retail_btm_caiso_alpha/ev_bess_pv_building_retail/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv
Summary CSV saved to: Sensitivity_Results/Rolling_24h/july2025_retail_btm_caiso_alpha/ev_bess_pv_building_retail/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/monthly_financial_summary.csv
Paper figures are generated by paper_financial_comparison_plots.ipynb
Completed isolated sensitivity case: Sensitivity_Results/Rolling_24h/july2025_retail_btm_caiso_alpha/ev_bess_pv_building_retail


Sensitivity comparison saved to: Sensitivity_Results/Rolling_24h/july01_callrate_comparison
                           Case  Optimal steps  Time-limit events  PASS
july2025_retail_btm_caiso_alpha           2976                  0  True
Monthly forecast comparison saved to: Sensitivity_Results/Rolling_24h/july2025_retail_btm_caiso_alpha/ev_bess_pv_building_retail


## Chronological March–May site-specific study
Run this opt-in batch only after the one-day gate passes. February baseline history is reused; each later month consumes the preceding months' executed history. Existing June/July results remain untouched.

In [ ]:
# Reproducible chronological site-study batch; OFF unless explicitly enabled.
# Run this cell after reviewing the standard main-notebook settings above.
import os, json, contextlib, traceback, time
from pathlib import Path
if os.environ.get('RUN_MULTIMONTH_SITE_STUDY', '0') == '1':
    _batch_root = Path('Results_Rolling/Site_Cases_2025/monthly_runs')
    _batch_root.mkdir(parents=True, exist_ok=True)
    _batch_notebook = json.loads(Path('upscaledev_imp_rollingMPC.ipynb').read_text())
    _batch_months = [int(m) for m in os.environ.get('SITE_STUDY_MONTHS', '3,4,5').split(',')]
    if not _batch_months or min(_batch_months)<3 or _batch_months != list(range(min(_batch_months), max(_batch_months)+1)):
        raise ValueError('Selected months must be contiguous from March onward; prior executed months are required')
    _status_path = _batch_root/'march_onward_batch_status.json'
    _batch_status = json.loads(_status_path.read_text()) if _status_path.exists() else []
    _batch_status = [r for r in _batch_status if r['month'] not in _batch_months]
    for _month in _batch_months:
        os.environ.update({'RUN_CANONICAL_STUDY':'1', 'CANONICAL_SITE_CASE':'Both',
            'CANONICAL_RESOURCE_CASE':'FULL_BTM', 'CANONICAL_FORECAST_CASE':'Both',
            'CANONICAL_RUN_SCOPE':'month', 'CANONICAL_RUN_MONTH':str(_month),
            'CANONICAL_CHAIN_START_MONTH':'3', 'CANONICAL_SOLVER_THREADS':'2'})
        _period = _batch_root / f'2025-{_month:02d}'
        _period.mkdir(parents=True, exist_ok=True)
        _started = time.time()
        print(f'Starting site-specific month {_month}; log: {_period / "execution.log"}', flush=True)
        try:
            with (_period/'execution.log').open('w', buffering=1) as _log:
                with contextlib.redirect_stdout(_log), contextlib.redirect_stderr(_log):
                    _namespace = {'__name__':'__main__'}
                    from IPython.display import display
                    _namespace['display'] = display
                    for _index in [3,5,12]:
                        exec(compile(''.join(_batch_notebook['cells'][_index]['source']),
                             f'rolling_cell_{_index}', 'exec'), _namespace)
            _batch_status.append({'month':_month,'status':'PASS','elapsed_s':time.time()-_started})
            (_batch_root/'march_onward_batch_status.json').write_text(json.dumps(_batch_status,indent=2))
            print(f'Month {_month} validated in {(time.time()-_started)/60:.1f} minutes', flush=True)
        except BaseException as _error:
            with (_period/'execution.log').open('a') as _log: traceback.print_exc(file=_log)
            _batch_status.append({'month':_month,'status':'FAILED','error':repr(_error)})
            (_batch_root/'march_onward_batch_status.json').write_text(json.dumps(_batch_status,indent=2))
            raise
    # Generate only the cross-month publication cell, never legacy figure sections.
    _plot_book = json.loads(Path('paper_financial_comparison_plots.ipynb').read_text())
    _plot_cells = [c for c in _plot_book['cells'] if c['cell_type']=='code' and
                   ''.join(c['source']).startswith('# Site-specific cross-month publication comparison')]
    if len(_plot_cells) != 1: raise ValueError('Missing unique cross-month plotting cell')
    exec(compile(''.join(_plot_cells[0]['source']), 'cross_month_figures','exec'), {'__name__':'__main__'})
    _spec_cells = [c for c in _plot_book['cells'] if c['cell_type']=='code' and ''.join(c['source']).startswith('# Fig16-style monthly site power specifications')]
    if len(_spec_cells)!=1: raise ValueError('Missing Fig16 monthly source cell')
    exec(compile(''.join(_spec_cells[0]['source']), 'monthly_power_specs', 'exec'), {'__name__':'__main__'})
    print('MULTI-MONTH SITE STUDY AND PUBLICATION EXPORTS COMPLETE', flush=True)
else:
    print('Multi-month batch is OFF; set RUN_MULTIMONTH_SITE_STUDY=1 to run March-May.')
